In [1]:
# ARC-AGI-2 with Qwen3.8-Flash-Next on 4x NVIDIA L4, served by vLLM -- ALL cells in order, paste into ONE notebook cell.
# FULL MODEL + ARC-3 STYLE: the Intel W4A16 AutoRound checkpoint (512 experts), 32k history window with a 16k drain,
# and as many streams as its KV holds (3-8, from vLLM's memory profile at startup: 'ARC2 STREAMS').
# 8 STREAMS (31-oct-2026): 8 seats, graphs up to 32, 225 MB of KV per running stream (results_43, see below).
# Settings: accelerator "GPU L4 x4", internet off. Inputs:
#   - competition arc-prize-2026-arc-agi-2
#   - dataset  giovannyrodrguez/llmvbv                         (vLLM nightly 8a728663 wheels)
#   - dataset  dfranzen/pennyroyal-v253                       (only for its uv wheel)
#   - model    dfranzen/intel-qwen3.8-flash-next-w4a16-autoround
#   - model    dfranzen/albucino-qwen3-8-flash-next-drafter    (MTP draft)
# Server (cell 3): the vLLM recipe of tonyd2wild/Qwen38-Flash-Next-4x3090 on L4: TP 4 + expert parallel (Marlin MoE
# kernel), MTP 3, fp8 e4m3 KV, no vision tower, decode CUDA graphs, disk-backed PLE table, prefix caching on,
# the full model, 512 experts per layer (FULL MODEL, see below).
# Decode optimizations ON (runtime patches): int8 dense linears (W8), drafter scoring 8,735 token ids, PLE read
# without the thread pool -- bench v4 on Kaggle 4x L4 (24-oct-2026): 49.1 -> 35.4 ms per decode step.
# Batch-sharded sampling ON (--enable-batch-sharded-sampling, VLLM_USE_FLASHINFER_SAMPLER=0): bench v9 -2.0/-3.5 %
# per decode step; the same tokens (Colab L4 check, 25-oct-2026).
# Dense prefix cache ON (--prefix-cache-retention-interval None): the next turn restores the GDN state the decode saved
# inside the previous reply instead of recomputing the reply (results_13: re-prefill -50 %, +13 % tok/s per stream).
# Colab A100 cell 3m (26-oct-2026): it moves the model about half as much as W8 does (KL +0.0025 vs +0.0052).
# ARC2-L4 RecoverSSM (28-oct-2026): the GDN layers keep no per-draft states (lordhansolo's RecoverSSM, Triton GDN decode
# kernel), so the KV holds more streams: Colab A100 with the L4-sized pool, 8 streams, 20 tasks: 13.5 vs 13.0,
# 0 preemptions, KV peak 90.5 % (results_colab_a100_prompt_3).
# The solver runs 3-8 streams, set at startup from the KV vLLM measures (cell 3, 'ARC2 STREAMS').
# CPU telemetry (30-oct-2026, measurement only): /kaggle/working/arc2_cpu.jsonl every 30 s (CPU busy / steal / pressure,
# cgroup throttling, cores of vLLM / tool sandboxes / solver, GPU clocks), 'ARC2 CPU {json}' at the end of cell 4, and
# the per-step timeline 'ARC2STEP {json}' in serve.log (OPT STEPLOG on).
# Cell 2 precaches the vLLM wheels, not the Pennyroyal wheelhouse (31-oct-2026): read cold by uv they took 16m42s.
# FULL MODEL (31-oct-2026, final submissions): the production notebook qwen-4xl4_rsm8_w8_cpu_kv_pre without its
# pruning step (that notebook prunes the target to 448 experts per layer). EP 4 puts 128 experts per GPU instead of
# 112: 16 more x 48 layers x 2.53 MB = ~1.81 GiB more per GPU (the 19-oct vLLM run of the full model, before W8:
# 1.85). So the KV gets 1.7e9 bytes per GPU (--kv-cache-memory-bytes; ~204,000 tokens = 2.9 windows of 69,632)
# instead of 3.5e9, and the solver runs 3 streams instead of 8: with the KV use measured per stream (results_33,
# r29-r31) 4 streams would fill that pool ~15-20 % of the time, 3 streams ~2-4 %. vLLM gets 4 seats (MAXREQ):
# batch-sharded sampling needs max_num_seqs >= TP 4, and 4 seats capture the drafter's size-4 decode graph.
# Rank-0 margin: 21.71 free - 18.81 consumed - 0.65 activations - ~0.25 graphs (sizes up to 12) - 1.58 KV = ~0.42
# GiB (production: 0.41). Everything else is the production notebook: server flags, runtime patches, sampling,
# harness, time sharing.
# FULL MODEL + ARC-3 STYLE (31-oct-2026, last submission): history window 32k, drain 16k (ARC-3: 64k / 32k), and
# more KV for the full model without changing any number it computes: the input embedding tables in pinned host
# memory read through UVA (EMBED_HOST, 0.30 GiB per GPU each), the W8 int8 weights packed after the conversion
# (W8_COMPACT, up to ~0.65 GiB per GPU), and the KV from vLLM's own memory profile (GMU 0.95, rank-0 free at the
# peak ~0.78 GiB, as production) so it gets what those free. Streams = floor(0.72 x KV per GPU / 285 MB), 3 to 6
# (results_40: ~285 MB of KV per running stream with the 32k window). The previous paragraph's KV bytes and 3
# streams no longer apply. EMBED_HOST pins ~0.5 GiB of host RAM per table and GPU (~4 GiB in all; RAM had ~171 GiB
# free in results_33 / results_40).
# 8 STREAMS (31-oct-2026, results_43 = the Save & Run of the paragraph above): 6 streams used the KV (2.59 GiB per
# GPU) at p95 62 %, max 69 %, 0.3 % of samples waiting; the rule gave 7 and the 6 seats cut it. The decode step
# costs what the pruned model's does at the same number of requests (6: 47.6 vs 48.1 ms; 8: 55.0 ms), so 8 streams
# give ~+15 % tokens per second. Now 8 seats, graphs up to 8 x 4, and the rule with the median KV per running
# stream measured there (225 MB): 8 streams from ~2.33 GiB of KV per GPU. The bigger graphs come out of the KV
# budget (vLLM's estimate is inside it), so rank 0 keeps the same ~0.8 GiB free at the peak.
# W8 kernel tuned for 8 / 11 streams (28-oct-2026, Colab L4 w8_tune.zip): 21 entries in ARC2_W8_CONFIGS for
# M 17-64; W8 time per verify step with 8 streams 11.46 -> 8.84 ms (same kernel; <= 1 bf16 ulp).
# Equal time sharing (26-oct-2026): each time a task ends, the time left is shared again in equal parts among every
# unfinished task, running ones included (results_31: the first tasks were cut at ~12-15 min, the last got 36).
# ARC-AGI-2 solver (cell 4): unchanged from the SGLang notebook -- retained reasoning, growing history, drain with
# persistent memory, python tool with retained functions; transform(grid) checked on the example pairs, two
# attempts per test input. KAGGLE_IS_COMPETITION_RERUN -> the test challenges for 12 hours; otherwise (Save & Run)
# the first 20 evaluation tasks with the time per task the 12-hour rerun gives (its share of the 12 hours over
# 240 test tasks: ~58 min for the 20), scored against the evaluation solutions.

# %% ---------------------------------------------------------------- [cell 1: environment]
import json, os, re, shutil, subprocess, sys, threading, time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() not in ("", "0", "false")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
SERVED_MODEL_NAME = "flashnext"
SERVED_MODEL_PORT = 8001
SERVER_STARTUP_TIMEOUT = 40 * 60

WHEELHOUSE_DIR  = "/kaggle/input/datasets/dfranzen/pennyroyal-v253"
MODEL_DIR       = "/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1"
DRAFT_MODEL_DIR = "/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1"
ARC2_DIR        = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
VLLM_DIR        = "/kaggle/input/datasets/giovannyrodrguez/llmvbv/vllm-8a728663"
print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}", flush=True)

def _sh(cmd, timeout=30):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return (r.stdout or "") + (r.stderr or "")
    except Exception as e:
        return f"failed: {e!r}"


def env_report():
    info = {"python": sys.version.split()[0], "cpus": os.cpu_count()}
    gpus = []
    for line in _sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version,compute_cap",
                     "--format=csv,noheader"]).splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) == 5:
            gpus.append({"index": p[0], "name": p[1], "memory": p[2], "driver": p[3], "cc": p[4]})
    info["gpus"] = gpus
    m = re.search(r"CUDA Version:\s*([\d.]+)", _sh(["nvidia-smi"]))
    info["driver_cuda"] = m.group(1) if m else None
    mem = {}
    for line in open("/proc/meminfo"):
        k, _, rest = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "SwapTotal"):
            mem[k] = round(int(rest.split()[0]) / 1024 ** 2, 1)
    info["ram_gib"] = mem
    disks = {}
    for path in ("/", "/kaggle/working", "/tmp", "/dev/shm"):
        try:
            s = shutil.disk_usage(path)
            disks[path] = {"free_gib": round(s.free / 1024 ** 3, 1), "total_gib": round(s.total / 1024 ** 3, 1)}
        except OSError:
            pass
    info["disk"] = disks
    info["gpu_links"] = _sh(["nvidia-smi", "topo", "-m"]).strip().splitlines()[:8]
    info["inputs"] = {name: Path(p).is_dir() for name, p in (
        ("pennyroyal", WHEELHOUSE_DIR), ("intel_model", MODEL_DIR), ("draft", DRAFT_MODEL_DIR),
        ("arc2", ARC2_DIR))}
    return info


# a valid submission before anything can stop the run: every task of the challenges file with [[0]]
_ch_path = Path(ARC2_DIR) / ("arc-agi_test_challenges.json" if TRUE_SUBMISSION else "arc-agi_evaluation_challenges.json")
if _ch_path.is_file():
    _ch = json.loads(_ch_path.read_text())
    (WORKING_DIR / "submission.json").write_text(json.dumps(
        {t: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in _ch[t]["test"]] for t in sorted(_ch)}))

ENV_INFO = env_report()
print("L4 ENV", json.dumps(ENV_INFO, indent=1), flush=True)
ENV_INFO["inputs"]["vllm"] = Path(VLLM_DIR).is_dir()
_missing = [k for k, v in ENV_INFO["inputs"].items() if not v]
if _missing:
    raise RuntimeError(f"missing notebook inputs: {_missing} (see the list at the top of this cell)")
if (ENV_INFO["ram_gib"].get("MemTotal") or 0) < 110:
    print("L4 ENV WARNING: less than 110 GiB of RAM; the BF16 PLE table (~95 GiB) is kept in pinned host RAM "
          "and will probably not fit", flush=True)

# ---- CPU telemetry (measurement only: nothing the model computes changes; 30-oct-2026) ----
# The ARC-AGI-3 top-3 entry saw 5-10 % lower scores from CPU throttling. Every 30 s, into /kaggle/working/arc2_cpu.jsonl:
# CPU busy / steal / iowait (/proc/stat, per core too), CPU / memory / io pressure (/proc/pressure), cgroup CPU
# throttling (cpu.stat), load average, CPU MHz, available memory; the cores used by the vLLM processes (each by name),
# by the python tool sandboxes (reaped children of this kernel) and by this kernel (the solver); every 60 s the GPUs
# (utilization, SM clock, power, temperature, throttle reasons). A failed sample is written as an error and the run goes
# on. Phases: setup, server, solving, done; the solving phase is summarized at the end of cell 4 ('ARC2 CPU {json}').
import statistics as _cpu_stats
import threading as _cpu_threading

ARC2_CPU = dict(EVERY_S=30, GPU_EVERY_S=60, PATH=str(WORKING_DIR / "arc2_cpu.jsonl"))
ARC2_CPU_PHASE = {"phase": "setup"}
ARC2_CPU_STOP = _cpu_threading.Event()
_CPU_TCK = os.sysconf("SC_CLK_TCK")
_CPU_SANDBOX_MARK = "records_path"                 # a word of the tool sandbox's source (its -c argument)
_CPU_CGROUP_STATS = ("/sys/fs/cgroup/cpu.stat", "/sys/fs/cgroup/cpu/cpu.stat", "/sys/fs/cgroup/cpu,cpuacct/cpu.stat")
_CPU_CGROUP_LIMITS = ("/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/cpu/cpu.cfs_quota_us",
                      "/sys/fs/cgroup/cpu/cpu.cfs_period_us", "/sys/fs/cgroup/cpu,cpuacct/cpu.cfs_quota_us")


def _cpu_text(path):
    with open(path) as f:
        return f.read()


def _cpu_procs():
    """{pid: (ppid, own ticks, reaped children's ticks, command line)} of every process."""
    out = {}
    for d in os.listdir("/proc"):
        if not d.isdigit():
            continue
        try:
            s = _cpu_text(f"/proc/{d}/stat")
            r = s[s.rindex(")") + 2:].split()
            with open(f"/proc/{d}/cmdline", "rb") as f:
                cmd = f.read().replace(b"\0", b" ").decode("utf-8", "replace").strip()
            out[int(d)] = (int(r[1]), int(r[11]) + int(r[12]), int(r[13]) + int(r[14]), cmd)
        except (OSError, ValueError, IndexError):
            pass
    return out


def _cpu_subtree(procs, roots):
    kids = {}
    for p, v in procs.items():
        kids.setdefault(v[0], []).append(p)
    out, stack = [], list(roots)
    while stack:
        p = stack.pop()
        out.append(p)
        stack.extend(kids.get(p, []))
    return out


def _cpu_name(cmd):
    if cmd.startswith("VLLM::"):
        return cmd.split()[0][:40]
    parts = cmd.split()
    if "-m" in parts and parts.index("-m") + 1 < len(parts):
        return parts[parts.index("-m") + 1][:40]
    return (os.path.basename(parts[0]) if parts else "?")[:40]


def _cpu_stat():
    rows = {}
    for line in _cpu_text("/proc/stat").splitlines():
        if line.startswith("cpu"):
            p = line.split()
            rows[p[0]] = [int(x) for x in p[1:9]]   # user nice system idle iowait irq softirq steal
    return rows


def _cpu_kv(path):
    out = {}
    for line in _cpu_text(path).splitlines():
        p = line.split()
        if len(p) == 2:
            out[p[0]] = int(p[1])
    return out


def _cpu_psi(kind):
    out = {}
    for line in _cpu_text(f"/proc/pressure/{kind}").splitlines():
        p = line.split()
        out[p[0]] = int(dict(x.split("=") for x in p[1:])["total"])     # microseconds
    return out


def _cpu_snapshot():
    snap = {"time": time.time(), "procs": _cpu_procs(), "stat": _cpu_stat(), "psi": {}, "cgroup": None}
    for kind in ("cpu", "memory", "io"):
        try:
            snap["psi"][kind] = _cpu_psi(kind)
        except (OSError, KeyError, ValueError):
            pass
    for path in _CPU_CGROUP_STATS:
        if os.path.isfile(path):
            snap["cgroup"] = _cpu_kv(path)
            break
    return snap


def _cpu_gpus():
    q = "index,utilization.gpu,clocks.sm,power.draw,temperature.gpu,clocks_throttle_reasons.active"
    r = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"], capture_output=True,
                       text=True, timeout=30)
    return [[x.strip() for x in line.split(",")] for line in r.stdout.strip().splitlines()]


def _cpu_sample(prev, cur, me):
    dt = cur["time"] - prev["time"]
    rec = {"t": round(cur["time"] - NOTEBOOK_START_TIME, 1), "phase": ARC2_CPU_PHASE["phase"], "dt": round(dt, 1)}
    a, b = prev["stat"]["cpu"], cur["stat"]["cpu"]
    d = [y - x for x, y in zip(a, b)]
    tot = max(1, sum(d))
    busy = d[0] + d[1] + d[2] + d[5] + d[6]
    rec["cpu"] = {"busy_pct": round(100 * busy / tot, 1), "user_pct": round(100 * (d[0] + d[1]) / tot, 1),
                  "sys_pct": round(100 * (d[2] + d[5] + d[6]) / tot, 1), "iowait_pct": round(100 * d[4] / tot, 1),
                  "steal_pct": round(100 * d[7] / tot, 1), "idle_pct": round(100 * d[3] / tot, 1)}
    per = []
    for k, v in cur["stat"].items():
        if k != "cpu" and k in prev["stat"]:
            dd = [y - x for x, y in zip(prev["stat"][k], v)]
            per.append(100 * (dd[0] + dd[1] + dd[2] + dd[5] + dd[6]) / max(1, sum(dd)))
    rec["cpu"]["cores"] = len(per)
    rec["cpu"]["cores_over_90"] = sum(p > 90 for p in per)
    rec["cpu"]["max_core_pct"] = round(max(per), 1) if per else None
    rec["cpu"]["busy_cores"] = round(busy / max(1, tot) * len(per), 2)
    for kind, v in cur["psi"].items():
        if kind in prev["psi"]:
            for level, total in v.items():
                rec[f"psi_{kind}_{level}_pct"] = round(100 * (total - prev["psi"][kind].get(level, total)) / (dt * 1e6), 2)
    if cur["cgroup"] and prev["cgroup"]:
        c, p = cur["cgroup"], prev["cgroup"]
        per_d = c.get("nr_periods", 0) - p.get("nr_periods", 0)
        thr_d = c.get("nr_throttled", 0) - p.get("nr_throttled", 0)
        thr_s = ((c["throttled_usec"] - p["throttled_usec"]) / 1e6 if "throttled_usec" in c
                 else (c.get("throttled_time", 0) - p.get("throttled_time", 0)) / 1e9)
        rec["cgroup"] = {"periods": per_d, "throttled_periods": thr_d,
                         "throttled_frac": round(thr_d / per_d, 4) if per_d else None,
                         "throttled_s_per_s": round(thr_s / dt, 4)}
    pp, cp = prev["procs"], cur["procs"]
    kids = [p for p, v in cp.items() if v[0] == me]
    vllm_roots = [p for p in kids if "vllm" in cp[p][3] and _CPU_SANDBOX_MARK not in cp[p][3]]
    vllm = _cpu_subtree(cp, vllm_roots)
    per_proc = []
    for p in vllm:
        if p in cp:
            ticks = cp[p][1] - (pp[p][1] if p in pp else 0)
            per_proc.append((_cpu_name(cp[p][3]), ticks / _CPU_TCK / dt))
    top = {}
    for n, c in per_proc:
        top[n] = top.get(n, 0.0) + c
    rec["cores"] = {
        "vllm": round(sum(c for _, c in per_proc), 2),
        "sandboxes": round((cp[me][2] - pp[me][2]) / _CPU_TCK / dt, 2) if me in cp and me in pp else None,
        "solver": round((cp[me][1] - pp[me][1]) / _CPU_TCK / dt, 2) if me in cp and me in pp else None,
    }
    rec["vllm_top"] = sorted(([n, round(c, 2)] for n, c in top.items()), key=lambda x: -x[1])[:8]
    rec["vllm_max_process_cores"] = round(max((c for _, c in per_proc), default=0.0), 2)
    rec["sandboxes_alive"] = sum(1 for p in kids if _CPU_SANDBOX_MARK in cp[p][3])
    try:
        la = _cpu_text("/proc/loadavg").split()
        rec["load"] = [float(la[0]), float(la[1]), float(la[2]), la[3]]
        mhz = [float(l.split(":")[1]) for l in _cpu_text("/proc/cpuinfo").splitlines() if l.startswith("cpu MHz")]
        rec["mhz"] = [round(min(mhz)), round(_cpu_stats.median(mhz)), round(max(mhz))] if mhz else None
        mem = {l.split(":")[0]: int(l.split()[1]) for l in _cpu_text("/proc/meminfo").splitlines()
               if l.startswith(("MemAvailable", "Cached"))}
        rec["mem_avail_gib"] = round(mem.get("MemAvailable", 0) / 2**20, 1)
        rec["page_cache_gib"] = round(mem.get("Cached", 0) / 2**20, 1)
    except (OSError, ValueError, IndexError):
        pass
    return rec


def _cpu_write(rec):
    with open(ARC2_CPU["PATH"], "a") as f:
        f.write(json.dumps(rec) + "\n")


def _cpu_loop():
    me = os.getpid()
    limits = {}
    for path in _CPU_CGROUP_LIMITS:
        try:
            limits[path] = _cpu_text(path).strip()
        except OSError:
            pass
    _cpu_write({"t": round(time.time() - NOTEBOOK_START_TIME, 1), "start": True, "cpus": os.cpu_count(),
                "affinity": len(os.sched_getaffinity(0)), "cgroup_limits": limits, "tck": _CPU_TCK})
    prev, last_gpu = _cpu_snapshot(), 0.0
    while not ARC2_CPU_STOP.wait(ARC2_CPU["EVERY_S"]):
        try:
            cur = _cpu_snapshot()
            rec = _cpu_sample(prev, cur, me)
            prev = cur
            if time.time() - last_gpu >= ARC2_CPU["GPU_EVERY_S"]:
                last_gpu = time.time()
                rec["gpu"] = _cpu_gpus()
            _cpu_write(rec)
        except Exception as exc:                       # telemetry only: record it and keep the run going
            try:
                _cpu_write({"t": round(time.time() - NOTEBOOK_START_TIME, 1), "error": f"{type(exc).__name__}: {exc}"[:300]})
            except OSError:
                pass


def arc2_cpu_summary(phase="solving"):
    """Medians / p90 / max of the telemetry over one phase (read back from the jsonl)."""
    rows, errors = [], 0
    with open(ARC2_CPU["PATH"]) as f:
        for line in f:
            r = json.loads(line)
            errors += "error" in r
            if r.get("phase") == phase and "cpu" in r:
                rows.append(r)
    if not rows:
        return {"phase": phase, "samples": 0, "errors": errors}

    def st(vals):
        vals = sorted(v for v in vals if v is not None)
        if not vals:
            return None
        return {"med": round(_cpu_stats.median(vals), 2), "p90": round(vals[int(0.9 * (len(vals) - 1))], 2),
                "max": round(vals[-1], 2)}

    out = {"phase": phase, "samples": len(rows), "minutes": round(sum(r["dt"] for r in rows) / 60, 1)}
    for k in ("busy_pct", "steal_pct", "iowait_pct", "busy_cores", "cores_over_90", "max_core_pct"):
        out[k] = st([r["cpu"].get(k) for r in rows])
    for k in ("psi_cpu_some_pct", "psi_cpu_full_pct", "psi_memory_some_pct", "psi_io_some_pct"):
        out[k] = st([r.get(k) for r in rows])
    out["cgroup_throttled_frac"] = st([(r.get("cgroup") or {}).get("throttled_frac") for r in rows])
    out["cgroup_throttled_s_per_s"] = st([(r.get("cgroup") or {}).get("throttled_s_per_s") for r in rows])
    for k in ("vllm", "sandboxes", "solver"):
        out[f"{k}_cores"] = st([r["cores"].get(k) for r in rows])
    out["vllm_max_process_cores"] = st([r.get("vllm_max_process_cores") for r in rows])
    out["sandboxes_alive"] = st([r.get("sandboxes_alive") for r in rows])
    out["load1"] = st([(r.get("load") or [None])[0] for r in rows])
    out["mhz_median"] = st([(r.get("mhz") or [None, None])[1] for r in rows])
    g = [x for r in rows for x in (r.get("gpu") or [])]
    try:
        out["gpu_util_pct"] = st([float(x[1]) for x in g])
        out["gpu_sm_mhz"] = st([float(x[2]) for x in g])
        out["gpu_power_w"] = st([float(x[3]) for x in g])
        reasons = {}
        for x in g:
            reasons[x[5]] = reasons.get(x[5], 0) + 1
        out["gpu_throttle_reasons"] = reasons
    except (ValueError, IndexError):
        out["gpu_parse_error"] = True
    out["errors"] = errors
    return out


ARC2_CPU_THREAD = _cpu_threading.Thread(target=_cpu_loop, name="arc2-cpu-telemetry", daemon=True)
ARC2_CPU_THREAD.start()
print(f"ARC2 CPU telemetry every {ARC2_CPU['EVERY_S']} s -> {ARC2_CPU['PATH']}", flush=True)

# %% ---------------------------------------------------------------- [cell 2: precache]
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

# the vLLM wheels cell 3 installs with uv (31-oct-2026: read cold from /kaggle/input by uv they took 16m42s; the
# Pennyroyal wheelhouse precached here before is only used for its uv wheel, one small read in cell 3)
VLLM_WHEEL_FILES = sorted(str(w) for d in sorted({p.parent for p in Path(VLLM_DIR).rglob("vllm-*.whl")})
                          for w in d.glob("*.whl"))
print(f"Precaching the vLLM wheels: {len(VLLM_WHEEL_FILES)} files, "
      f"{sum(os.path.getsize(w) for w in VLLM_WHEEL_FILES) / 1e9:.2f} GB", flush=True)
precache_wheels_thread = threading.Thread(target=precache, args=tuple(VLLM_WHEEL_FILES))
precache_wheels_thread.start()
precache_wheels_thread.join()

precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})


# %% ---------------------------------------------------------------- [cell 3: server (vLLM)]
# vLLM on 4x L4: the recipe of tonyd2wild/Qwen38-Flash-Next-4x3090 (vLLM lane), adapted to Ada (sm_89).
#   - vLLM nightly 8a728663 from the dataset giovannyrodrguez/llmvbv, installed offline with uv;
#   - TP 4 + expert parallel, so each GPU holds whole experts and the Marlin W4A16 MoE kernel runs (the recipe:
#     103 -> 193 tok/s on 3090s, 1.3 GB less per card);
#   - MTP, 3 draft tokens, with albucino's INT4 draft (the Kaggle input dfranzen/albucino-qwen3-8-flash-next-drafter);
#   - fp8 e4m3 KV (the sparse-attention Triton path needs compute capability 8.9: it did not compile on the 3090s,
#     the L4 is 8.9);
#   - --language-model-only (no vision tower), CUDA graphs for decode without torch.compile;
#   - prefix caching on, with a patch so it works with MTP (run 6: off, 2.98M prompt tokens re-prefilled);
#   - the recipe's files over the installed vLLM (Apache-2.0, LICENSE and NOTICE kept next to them): the disk-backed
#     PLE table (ple_layer.py, ple_mmap.py, model_state.py, "staged" mode) and the upstream overlays it runs with
#     (ops/ple.py, ops/qsa.py, qsa.py, platforms/interface.py). The overlays are whole files of vLLM at that commit,
#     so the run stops if the installed vLLM is another commit.
# Model: the Intel W4A16 AutoRound checkpoint as published (FULL MODEL: 512 experts per layer, not pruned); BF16 PLE
# table read from disk by the staged gather. No fallback: if the server does not come up the run stops with the log tail.
import base64, hashlib, io, os, re, shlex, socket, subprocess, sys, tarfile, time, urllib.request, zipfile
import json
from pathlib import Path

VLLM_COMMIT = "8a728663"
PREFIX = "/tmp/vllm-l4"
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
LOG = str(Path(WORKING_DIR) / "serve.log")
# solver streams, fixed here (not computed from the KV cache): bench v4 with W8 + draft vocab + PLE_SEQ left 321,633
# KV tokens = 4 windows of 69,632; 4 streams give each stream x0.87 of the 3-stream speed, ~x1.64 tokens per task
# ARC2-L4 RecoverSSM: ~5.2 windows (estimate), 8 streams: r31's live contexts (token-weighted mean ~28k) overflow that pool
# ~0.5 % of the time (vLLM preempts and recomputes: slower, same answers)
# FULL MODEL + ARC-3 STYLE: the streams come from the KV vLLM measures at startup, after the memory levers (header):
# N = floor(KV_FRACTION x KV bytes per GPU / STREAM_KV_BYTES), between 3 and CFG MAXREQ ('ARC2 STREAMS')
ARC2_STREAMS_KV_FRACTION = 0.72      # mean KV use where results_33 had 1.7 % waiting (results_40 at ~79 %: 9.1 %)
ARC2_STREAM_KV_BYTES = 225e6         # 8 STREAMS: KV per running stream with the 32k window, full model (results_43,
                                     # median with 6 running; results_40 had ~285 MB with the pruned model)
ARC2_STREAMS_MIN = 3
CFG = dict(
    CTX=(32 + 12 + 8) * 1024,   # solver window: 32k prompt + 12k output, plus 8k margin (ARC-3 STYLE)
    # FULL MODEL + ARC-3 STYLE, 8 STREAMS: 8 seats (results_43: 6 streams left the KV at p95 62 %). vLLM 8a728663
    # refuses --enable-batch-sharded-sampling with max_num_seqs below TP 4, and the drafter's decode graphs need a
    # capture size <= max_num_seqs (sizes 4, 6, 8 here)
    MAXREQ=8,
    # the KV from vLLM's memory profile, after the memory levers (header): 0.95 x 22.03 GiB requested, the CUDA graph
    # estimate inside that budget; rank 0 keeps ~0.78 GiB free at the peak, as production does
    GMU=0.95,
    CHUNK=2048,
    MTP=3,
    # 8 STREAMS: multiples of MTP+1 up to 8 seats x 4, and 6: the drafter's decode graphs take 1 token per request
    # and only sizes <= max_num_seqs, so 4 / 6 / 8 cover 1-4 / 5-6 / 7-8 running requests (the target rounds 6 up
    # to 8). vLLM takes its graph estimate (~0.5 GiB with sizes up to 32; results_33 captured 0.43) out of the KV
    # budget, so the rank-0 margin stays
    CAPTURE_SIZES=[4, 6, 8, 12, 16, 20, 24, 28, 32],
    KVDTYPE="fp8_e4m3",
    # ARC2-L4 decode optimizations (runtime patches), ON in this notebook. Bench v4 on Kaggle 4x L4 (24-oct-2026), GPU
    # time per decode step with 3 streams: base 49.1 ms; W8 41.4; draft vocab 44.4 (MTP acceptance unchanged, 3.07);
    # PLE_SEQ 46.2; all three 35.4 ms (x1.39); with 4 streams 40.5 ms. Quality checked on the Colab A100 (W8, vocab).
    #   W8: dense bf16 linears -> int8 + fp32 scale per 128 weights (own Triton kernel); -1.44 GiB per GPU, +49% KV
    #   DRAFT_VOCAB: the MTP drafter scores only the 8,735 token ids seen in the solver's outputs (exact: the target
    #       still verifies over the whole vocabulary; only the acceptance can change)
    #   PLE_SEQ: PLE rows read in the calling thread (no pool): -2.9 ms per step on Kaggle (bench v4)
    #   STEPLOG: per-step timeline 'ARC2STEP {json}' in serve.log every 200 decode steps
    #   EMBED_HOST: the input embedding tables in pinned host memory, read through UVA (same values; 0.30 GiB per GPU)
    #   W8_COMPACT: the int8 weights wait in host memory during the W8 conversion and come back packed (same values)
    OPT=dict(W8=True, DRAFT_VOCAB=True, PLE_SEQ=True, STEPLOG=True, EMBED_HOST=True, W8_COMPACT=True),
    # drafts sampled from the drafter's distribution instead of its argmax; rejection sampling keeps the output
    # distribution exact. Bench run 23: decode 158.9 -> 171.7 tok/s (3 requests), acceptance 2.99 -> 3.14
    SPEC_EXTRA={"draft_sample_method": "probabilistic"},
    # batch-sharded sampling: each GPU samples only the requests it owns (an all-to-all of vocabulary slices instead of
    # all-gathering the full-vocabulary logits of every request). Bench v9 on Kaggle 4x L4: decode period -2.0 / -3.5 %.
    # Same tokens with VLLM_USE_FLASHINFER_SAMPLER=0 in the server environment (Colab L4 check of vLLM's own sampling
    # code, results_emu_7: 3,894 / 3,894 tokens identical); needs compute_logits_local (patched in after the runtime
    # patches)
    # FULL MODEL + ARC-3 STYLE: no --kv-cache-memory-bytes; the KV comes from vLLM's memory profile (GMU above)
    EXTRA_ARGS=["--enable-batch-sharded-sampling"],
)

L4_PATCH_TGZ_B64 = "..."  # tar.gz of the recipe files, LICENSE and NOTICE


def _run(cmd, env=None):
    print("$ " + shlex.join(map(str, cmd)), flush=True)
    p = subprocess.run(list(map(str, cmd)), env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:], flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {shlex.join(map(str, cmd))}")
    return p.stdout


def _unique(paths, what):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise RuntimeError(f"expected one {what}, found {len(paths)}: {[str(p) for p in paths[:6]]}")
    return paths[0]


with socket.socket() as _probe:
    if _probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"port {SERVED_MODEL_PORT} is already in use; stop the old server first")

# ---- offline install: vLLM wheels from the dataset, uv from the Pennyroyal wheelhouse ----
VLLM_WHEELS = _unique((p.parent for p in Path(VLLM_DIR).rglob("vllm-*.whl")), "folder with the vllm wheel")
VLLM_WHEEL = _unique(VLLM_WHEELS.glob("vllm-*.whl"), "vllm wheel")
print("vLLM wheel:", VLLM_WHEEL.name, flush=True)
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(PYTHON).exists() or not Path(PREFIX, "installed.txt").is_file():
    uv_whl = _unique(Path(WHEELHOUSE_DIR).rglob("uv-*.whl"), "uv wheel in the Pennyroyal dataset")
    uv_bin = Path(PREFIX, "bin/uv")
    uv_bin.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_whl) as z:
        member = _unique((n for n in z.namelist() if Path(n).name == "uv" and not n.endswith("/")), "uv binary")
        uv_bin.write_bytes(z.read(member))
    uv_bin.chmod(0o755)
    if not Path(PYTHON).exists():
        _run([uv_bin, "venv", "--python", sys.executable, VENV], env=install_env)
    _run([uv_bin, "pip", "install", "--python", PYTHON, "--no-index", "--find-links", VLLM_WHEELS, VLLM_WHEEL],
         env=install_env)
    Path(PREFIX, "installed.txt").write_text(VLLM_WHEEL.name)
_versions = _run([PYTHON, "-c", "import vllm, torch, triton; print('vllm', vllm.__version__); "
                  "print('torch', torch.__version__, 'cuda', torch.version.cuda); print('triton', triton.__version__); "
                  "print('gpus', torch.cuda.device_count(), torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))"],
                 env=install_env)
if VLLM_COMMIT[:7] not in VLLM_WHEEL.name and VLLM_COMMIT[:7] not in _versions:
    raise RuntimeError(f"the installed vLLM ({VLLM_WHEEL.name}) is not commit {VLLM_COMMIT}: the recipe's overlays are "
                       "whole files of that commit; run stopped")

# ---- the recipe's files over the installed vLLM ----
VP = Path(_unique(Path(VENV).glob("lib/python*/site-packages/vllm"), "installed vllm package"))
MP = VP / "models" / "qwen4_exp" / "nvidia"
PATCH_DIR = Path(PREFIX, "l4patch")
PATCH_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(L4_PATCH_TGZ_B64)), mode="r:gz") as tar:
    tar.extractall(PATCH_DIR)
_overlays = {
    "ple_layer.py": MP / "ple_layer.py", "model_state.py": MP / "model_state.py",
    "ple_mmap.py": MP / "ops" / "ple_mmap.py",            # new file
    "ops_ple.py": MP / "ops" / "ple.py", "ops_qsa.py": MP / "ops" / "qsa.py", "qsa.py": MP / "qsa.py",
    "platforms_interface.py": VP / "platforms" / "interface.py",
}
for src, dst in _overlays.items():
    if src != "ple_mmap.py" and not dst.is_file():
        raise RuntimeError(f"{dst} is not in the installed vLLM: not the tree the recipe's overlays replace; run stopped")
    backup = dst.with_name(dst.name + ".l4-orig")
    if dst.is_file() and not backup.exists():
        backup.write_bytes(dst.read_bytes())
    dst.write_bytes((PATCH_DIR / src).read_bytes())
for pyc in list(MP.rglob("__pycache__/*.pyc")) + list((VP / "platforms" / "__pycache__").glob("interface*.pyc")):
    pyc.unlink(missing_ok=True)
print(f"L4 vLLM: {len(_overlays)} recipe files placed under {VP}", flush=True)

# ---- ARC2-L4 RecoverSSM: lordhansolo's ARC3 GDN RecoverSSM, ported to vLLM 8a728663 ----
# The GDN layers verify the MTP window from the running state without writing per-draft states and fold the
# accepted prefix in after sampling: the GDN and PLE groups hold 2 state blocks per request instead of 5
# (Colab A100: max concurrency 9.11x -> 11.25x; the L4 estimate 4.61 -> ~5.2 windows). Kernels checked bit for
# bit against the stock spec kernel on the CPU (BF16 and FP32 state, 1-4 accepted, split over 4 TP ranks).
# The 7 files (tar.gz, base64) and their pins: sha256 of the patched file and of the stock 8a728663 one.
# Needs VLLM_GDN_DECODE_KERNEL=triton (set with the environment below).
import gzip, shutil
RSM_OVERLAY_B64 = "..."
RSM_MANIFEST = json.loads(r"""[
 {
  "target": "vllm/config/vllm.py",
  "sha256": "df071e6811a08d098aa20d1eb736c3f22c390f7f645e37dc27ea5120cd0cff8a",
  "stock_sha256": "4239a36c25f641a3f944b6cca2c1c01a5d3dcb48a0a89d2c131c56fd3d1edd8e"
 },
 {
  "target": "vllm/model_executor/layers/mamba/abstract.py",
  "sha256": "de75c1566cdc859ee5419fc2a76698f8d25bb95955128d73723cc5bfee9fa28b",
  "stock_sha256": "dfdc8c836894f574e41d93d76ab989cacd3d7ecf59c97a929abfa48d9364e8ad"
 },
 {
  "target": "vllm/model_executor/layers/mamba/gdn/qwen_gdn_linear_attn.py",
  "sha256": "272919123ed232aaee03102556c62b72aa65fcfac7f6cb6bbbb698ffd4f8735e",
  "stock_sha256": "cb819264c1e791177d866f484ca0579a7d57141926705a1f75dd4a79de4d0125"
 },
 {
  "target": "vllm/v1/attention/backends/gdn_attn.py",
  "sha256": "1b9f4bfdffdefa776d2402f1ee5dfaed52943a387146a4db43c63a9d55e15b1e",
  "stock_sha256": "f78ae94fa99bfc13a705c3fd319347d55163420bc5dd6aceaf60f95171c54e4c"
 },
 {
  "target": "vllm/v1/worker/mamba_utils.py",
  "sha256": "3c05a9fa35ab87814cc242dbd7f850b19ef6acda3350481e4a7f42e0982433d1",
  "stock_sha256": "7382bcc8a7b070eaec44339379ebce485777fe23b97e05ee108088faa707c7e6"
 },
 {
  "target": "vllm/v1/worker/gpu/model_states/mamba_hybrid.py",
  "sha256": "9ea2dff4a2286ea35408a2cb30258f17754175cae4dd2bcc41343f6fcc343b34",
  "stock_sha256": "44d0a20cee12535ac7f29fc3d14ffca9994af3af84fd17ad93e71acf5bc79066"
 },
 {
  "target": "vllm/model_executor/layers/mamba/ops/arc3_gdn_recoverssm.py",
  "sha256": "31a305677c8bca5b0380de608848d13b5c2313c8133506555afa368f385b72b9",
  "stock_sha256": null
 }
]""")


def rsm_install(site):
    """Write the 7 files; each existing target must be our stock 8a728663 file (or already the patched one)."""
    with tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(RSM_OVERLAY_B64))), mode="r:") as tar:
        payload = {m.name: tar.extractfile(m).read() for m in tar if m.isfile()}
    done = []
    for e in RSM_MANIFEST:
        data = payload[e["target"]]
        if hashlib.sha256(data).hexdigest() != e["sha256"]:
            raise RuntimeError(f"RecoverSSM: {e['target']} in the bundle does not match its pin; cell stopped")
        tgt = site / e["target"]
        if tgt.exists():
            cur = hashlib.sha256(tgt.read_bytes()).hexdigest()
            if cur == e["sha256"]:
                continue
            if cur != e["stock_sha256"]:
                raise RuntimeError(f"RecoverSSM: {tgt} is not vLLM 8a728663's file (sha {cur[:16]}); cell stopped")
            bak = tgt.with_name(tgt.name + ".arc2rsm-orig")
            if not bak.exists():
                shutil.copyfile(tgt, bak)
        elif e["stock_sha256"] is not None:
            raise RuntimeError(f"RecoverSSM: {tgt} is missing; cell stopped")
        elif not tgt.parent.is_dir():
            raise RuntimeError(f"RecoverSSM: {tgt.parent} is not in the installed vLLM; cell stopped")
        tgt.write_bytes(data)
        for pyc in tgt.parent.glob(f"__pycache__/{tgt.stem}*.pyc"):
            pyc.unlink(missing_ok=True)
        done.append(e["target"])
    return done


print(f"RecoverSSM overlay over {VP.parent}: {rsm_install(VP.parent)}", flush=True)

# ---- ARC2 runtime patches (env-gated; see emu_patches.py) ----
import re
def _arc2_edit(path, edits, marker):
    """Apply (old, new) replacements to the file's pristine text, kept once as <name>.arc2orig, so a newer notebook in
    the same runtime re-patches cleanly; stop with an error if an anchor is missing or not unique."""
    path = Path(path)
    orig = path.with_name(path.name + ".arc2orig")
    current = path.read_text()
    if re.search(r"(?m)^# ARC2-L4 [\w-]+ patch$", current):     # a marker line appended by this function
        if not orig.is_file():
            raise RuntimeError(f"ARC2 patch {marker}: {path} was patched by an older notebook version (no {orig.name}); "
                               "Runtime > Disconnect and delete runtime, then run all cells again")
        text = orig.read_text()
    else:
        orig.write_text(current)
        text = current
    for old, new in edits:
        n = text.count(old)
        if n != 1:
            raise RuntimeError(f"ARC2 patch {marker}: anchor found {n} times in {path}: {old[:80]!r}")
        text = text.replace(old, new)
    text += f"\n# {marker}\n"
    if text == current:
        return "already"
    path.write_text(text)
    for pyc in path.parent.glob(f"__pycache__/{path.stem}*.pyc"):
        pyc.unlink(missing_ok=True)
    return "patched"


ARC2_STEPLOG_MODULE = """# ARC2-L4 step timeline (env ARC2_STEP_PROFILE=1). CUDA events and host clocks around the phases of each engine
# step of the V2 model runner; every ARC2_STEP_PROFILE_EVERY steps the dominant batch shape is summarized in the server
# log as 'ARC2STEP {json}' (tensor-parallel rank 0 only). GPU phases are read on the device timeline:
#   gap = previous step's end -> this forward's start (input prep kernels, PLE copy, and idle GPU),
#   fwd = target forward, sample = sampler + rejection, mid = postprocess, draft = MTP proposal.
# Host phases: prep = execute_model entry -> before the PLE staging, ple = the staged PLE gather (includes its wait
# for the GPU), launch = PLE done -> graph launch, period = entry to entry.
import json
import os
import statistics
import time

import torch

ENABLED = os.environ.get("ARC2_STEP_PROFILE", "0") == "1"
EVERY = max(20, int(os.environ.get("ARC2_STEP_PROFILE_EVERY", "200")))
_cur = None
_steps = []
_prev_end = None
_prev_enter = None
_rank0 = None
_logger = None


def _is_rank0():
    global _rank0
    if _rank0 is None:
        try:
            import torch.distributed as dist
            _rank0 = (not dist.is_initialized()) or dist.get_rank() == 0
        except Exception:
            _rank0 = True
    return _rank0


def begin(num_tokens, num_reqs):
    global _cur, _prev_enter
    if not ENABLED:
        return
    now = time.perf_counter()
    _cur = {"t": {"enter": now}, "e": {}, "shape": (int(num_tokens), int(num_reqs)), "prev_enter": _prev_enter}
    _prev_enter = now


def cpu(name):
    if _cur is not None:
        _cur["t"][name] = time.perf_counter()


def gpu(name):
    if _cur is not None:
        ev = torch.cuda.Event(enable_timing=True)
        ev.record()
        _cur["e"][name] = ev


def end():
    global _cur, _prev_end
    if _cur is None:
        return
    s, _cur = _cur, None
    s["t"]["exit"] = time.perf_counter()
    s["prev_end"] = _prev_end
    _prev_end = s["e"].get("draft1") or s["e"].get("samp1")
    _steps.append(s)
    if len(_steps) >= EVERY:
        _flush()


def _ms(a, b):
    return a.elapsed_time(b) if (a is not None and b is not None) else None


def _stats(vals):
    vals = sorted(v for v in vals if v is not None)
    if not vals:
        return None
    return {"med": round(statistics.median(vals), 3), "p90": round(vals[int(0.9 * (len(vals) - 1))], 3),
            "mean": round(sum(vals) / len(vals), 3)}


def _flush():
    global _steps, _logger
    steps, _steps = _steps, []
    last = steps[-1]["e"].get("draft1") or steps[-1]["e"].get("samp1")
    if last is not None:
        last.synchronize()
    if not _is_rank0():
        return
    shapes = {}
    for s in steps:
        shapes[s["shape"]] = shapes.get(s["shape"], 0) + 1
    shape = max(shapes, key=shapes.get)
    rows = []
    for s in steps:
        if s["shape"] != shape:
            continue
        e, t = s["e"], s["t"]
        end_ev = e.get("draft1") or e.get("samp1")
        rows.append({
            "gap": _ms(s["prev_end"], e.get("fwd0")), "fwd": _ms(e.get("fwd0"), e.get("samp0")),
            "sample": _ms(e.get("samp0"), e.get("samp1")), "mid": _ms(e.get("samp1"), e.get("draft0")),
            "draft": _ms(e.get("draft0"), e.get("draft1")), "step_gpu": _ms(s["prev_end"], end_ev),
            "prep": 1e3 * (t["pre_ple"] - t["enter"]) if "pre_ple" in t else None,
            "ple": 1e3 * (t["post_ple"] - t["pre_ple"]) if "post_ple" in t and "pre_ple" in t else None,
            "launch": 1e3 * (t["launch"] - t["post_ple"]) if "launch" in t and "post_ple" in t else None,
            "sample_tokens": 1e3 * (t["exit"] - t["samp0c"]) if "samp0c" in t else None,
            "period": 1e3 * (t["enter"] - s["prev_enter"]) if s["prev_enter"] else None,
        })
    out = {"steps": len(rows), "of": len(steps), "num_tokens": shape[0], "num_reqs": shape[1]}
    for k in ("step_gpu", "gap", "fwd", "sample", "mid", "draft", "prep", "ple", "launch", "sample_tokens", "period"):
        out[k] = _stats([r[k] for r in rows])
    if _logger is None:
        from vllm.logger import init_logger
        _logger = init_logger("vllm.arc2_steplog")
    _logger.info("ARC2STEP %s", json.dumps(out))
"""


def apply_arc2_runtime_patches(VP):
    VP = Path(VP)
    MPD = VP / "models" / "qwen4_exp" / "nvidia"
    res = {}
    # 1. step timeline: a module next to the V2 runner and hooks in model_runner.py
    (VP / "v1" / "worker" / "gpu" / "arc2_steplog.py").write_text(ARC2_STEPLOG_MODULE)
    res["steplog"] = _arc2_edit(VP / "v1" / "worker" / "gpu" / "model_runner.py", [
        ("logger = init_logger(__name__)\n",
         "logger = init_logger(__name__)\nfrom vllm.v1.worker.gpu import arc2_steplog as _arc2_steplog  # ARC2-L4\n"),
        ("        if not dummy_run:\n            # Update the request states.\n",
         "        if not dummy_run:\n            _arc2_steplog.begin(scheduler_output.total_num_scheduled_tokens,\n"
         "                                len(scheduler_output.num_scheduled_tokens))  # ARC2-L4\n"
         "            # Update the request states.\n"),
        ("        model_inputs = {\n            \"input_ids\": input_ids,\n",
         "        _arc2_steplog.cpu(\"pre_ple\")  # ARC2-L4\n        model_inputs = {\n            \"input_ids\": input_ids,\n"),
        ("        if not self.is_first_pp_rank:\n            # Update for non-first PP ranks.\n",
         "        _arc2_steplog.cpu(\"post_ple\")  # ARC2-L4\n"
         "        if not self.is_first_pp_rank:\n            # Update for non-first PP ranks.\n"),
        ("        self.step_timing.forward_start()\n",
         "        self.step_timing.forward_start()\n        _arc2_steplog.gpu(\"fwd0\")  # ARC2-L4\n"
         "        _arc2_steplog.cpu(\"launch\")\n"),
        ("        sampler_output, num_sampled, num_rejected = self.sample(\n"
         "            hidden_states, input_batch, grammar_output\n        )\n",
         "        _arc2_steplog.gpu(\"samp0\")  # ARC2-L4\n        _arc2_steplog.cpu(\"samp0c\")\n"
         "        sampler_output, num_sampled, num_rejected = self.sample(\n"
         "            hidden_states, input_batch, grammar_output\n        )\n        _arc2_steplog.gpu(\"samp1\")  # ARC2-L4\n"),
        ("            with use_workspace_lane(self._draft_workspace_lane):\n"
         "                draft_tokens = self.speculator.propose(\n",
         "            _arc2_steplog.gpu(\"draft0\")  # ARC2-L4\n"
         "            with use_workspace_lane(self._draft_workspace_lane):\n"
         "                draft_tokens = self.speculator.propose(\n"),
        ("            self.req_states.draft_tokens[input_batch.idx_mapping] = draft_tokens\n",
         "            self.req_states.draft_tokens[input_batch.idx_mapping] = draft_tokens\n"
         "            _arc2_steplog.gpu(\"draft1\")  # ARC2-L4\n"),
        ("        return async_output\n\n    def take_draft_token_ids",
         "        _arc2_steplog.end()  # ARC2-L4\n        return async_output\n\n    def take_draft_token_ids"),
    ], "ARC2-L4 steplog patch")
    # 2. PLE stub (timing attribution only)
    res["ple_stub"] = _arc2_edit(MPD / "ple_layer.py", [
        ("        if not real or num_tokens <= 0:\n            self._ple_stage_zero(padded_tokens)\n            return\n",
         "        if not real or num_tokens <= 0 or _os.environ.get(\"ARC2_PLE_STUB\") == \"1\":  # ARC2-L4 (timing only)\n"
         "            self._ple_stage_zero(padded_tokens)\n            return\n"),
    ], "ARC2-L4 ple-stub patch")
    # 3. reduced-vocabulary drafting in the MTP head
    res["draft_vocab"] = _arc2_edit(MPD / "mtp.py", [
        ("        return self.logits_processor(self.lm_head, hidden_states)\n\n    def load_weights(",
         "        if _arc2_draft_vocab_k() > 0 and hasattr(self.lm_head, \"weight\"):  # ARC2-L4\n"
         "            return _arc2_reduced_vocab_logits(self, hidden_states)\n"
         "        return self.logits_processor(self.lm_head, hidden_states)\n\n    def load_weights("),
        ("__all__ = [\"Qwen4ExpMTP\", \"Qwen4ExpMultiTokenPredictor\"]\n",
         "__all__ = [\"Qwen4ExpMTP\", \"Qwen4ExpMultiTokenPredictor\"]\n" + ARC2_DRAFT_VOCAB_FUNCS),
    ], "ARC2-L4 draft-vocab patch")
    # 4. fused multi-step draft decode with the QSA state caches
    res["qsa_fused_draft"] = _arc2_edit(VP / "models" / "qwen4_exp" / "common" / "qsa_cache.py", [
        ("    _cudagraph_support: ClassVar[AttentionCGSupport] = AttentionCGSupport.UNIFORM_BATCH\n\n    def __init__(",
         "    _cudagraph_support: ClassVar[AttentionCGSupport] = AttentionCGSupport.UNIFORM_BATCH\n"
         "    # ARC2-L4: in-place draft-decode update (below) lets vLLM fuse the draft decode steps\n"
         "    supports_draft_decode_metadata_update = __import__(\"os\").environ.get(\"ARC2_QSA_FUSED_DRAFT\", \"0\") == \"1\"\n\n"
         "    def __init__("),
        ("        return QSAForwardMetadata(\n            block_table=common_attn_metadata.block_table_tensor,\n",
         "        _arc2_md = QSAForwardMetadata(\n            block_table=common_attn_metadata.block_table_tensor,\n"),
        ("            storage_block_size=self.storage_block_size,\n            compress_ratio=self.compress_ratio,\n"
         "        )\n\n\nclass QSAStateBackend(AttentionBackend):",
         "            storage_block_size=self.storage_block_size,\n            compress_ratio=self.compress_ratio,\n"
         "        )\n        _arc2_md._arc2_common = common_attn_metadata  # ARC2-L4: kept for the draft update\n"
         "        return _arc2_md\n\n" + ARC2_QSA_UPDATE_METHOD + "\n\nclass QSAStateBackend(AttentionBackend):"),
    ], "ARC2-L4 qsa-fused-draft patch")
    # 5. PLE rows gathered from mmaps with numpy (no thread pool) for decode-size requests
    res["ple_mmap_gather"] = _arc2_edit(MPD / "ops" / "ple_mmap.py", [
        ("        id_list = ids.view(-1).tolist()\n",
         "        if _arc2_ple_fast(n):  # ARC2-L4: numpy gather over mmaps of the shard files (warm page cache)\n"
         "            _arc2_mmap_gather(self, ids.view(-1).numpy(), out.numpy())\n"
         "            return out\n"
         "        id_list = ids.view(-1).tolist()\n"),
        ("__all__ = [\"PLEMmapTable\"]\n", "__all__ = [\"PLEMmapTable\"]\n" + ARC2_PLE_FAST_FUNCS),
    ], "ARC2-L4 ple-mmap-gather patch")
    # 6. int8 weight-only dense linears: the kernel module, converted at the end of process_weights_after_loading
    (VP / "model_executor" / "layers" / "arc2_w8a16.py").write_text(ARC2_W8A16_MODULE)
    res["w8a16"] = _arc2_edit(VP / "model_executor" / "model_loader" / "utils.py", [
        ("    # Model-level post-load hook, after the per-layer quant finalize.\n"
         "    if hasattr(model, \"process_weights_after_loading\"):\n"
         "        model.process_weights_after_loading()\n",
         "    # Model-level post-load hook, after the per-layer quant finalize.\n"
         "    if hasattr(model, \"process_weights_after_loading\"):\n"
         "        model.process_weights_after_loading()\n\n"
         "    # ARC2-L4: int8 weight-only dense linears (env ARC2_W8_MODULES), after every other post-load step\n"
         "    if __import__(\"os\").environ.get(\"ARC2_W8_MODULES\"):\n"
         "        from vllm.model_executor.layers.arc2_w8a16 import arc2_w8_convert\n\n"
         "        arc2_w8_convert(model)\n"
         "    # ARC2-L4: input embedding tables in pinned host memory, read through UVA (env ARC2_EMBED_HOST)\n"
         "    if __import__(\"os\").environ.get(\"ARC2_EMBED_HOST\") == \"1\":\n"
         "        from vllm.model_executor.layers.arc2_w8a16 import arc2_embed_to_host\n\n"
         "        arc2_embed_to_host(model)\n"),
    ], "ARC2-L4 w8a16 patch")
    # 6b. the torch.compile caches (AOT and Inductor) are keyed by envs.compile_factors(): add ARC2_W8_MODULES, or a
    # graph compiled for bf16 weights would be loaded from disk for int8 ones (same config, same traced sources)
    res["w8a16_cache_key"] = _arc2_edit(VP / "envs.py", [
        ("    from vllm.config.utils import normalize_value\n\n    factors: dict[str, object] = {}\n",
         "    from vllm.config.utils import normalize_value\n\n    factors: dict[str, object] = {}\n"
         "    factors[\"ARC2_W8_MODULES\"] = os.environ.get(\"ARC2_W8_MODULES\", \"\")  # ARC2-L4: int8 layers change the graph\n"
         "    factors[\"ARC2_W8_GROUP\"] = os.environ.get(\"ARC2_W8_GROUP\", \"128\")  # ARC2-L4: the group is a graph constant\n"),
    ], "ARC2-L4 w8a16-cache-key patch")
    return res


ARC2_PLE_FAST_FUNCS = """

# ---- ARC2-L4: decode-size PLE gathers without the thread pool. The staged path blocks the host until the GPU has
# finished the previous step, then gathers 16 rows per token; with the pool (chunks of 8 rows, one future each) that
# took ~3.4 ms of GPU idle per decode step on the 1x L4 rank emulator (22-oct). One numpy gather over read-only
# mmaps of the shard files costs page-cache lookups only. Env ARC2_PLE_GATHER=mmap, up to ARC2_PLE_MMAP_MAX_ROWS rows.
def _arc2_ple_fast(n):
    import os as _o
    if _o.environ.get("ARC2_PLE_GATHER", "") != "mmap":
        return False
    return 0 < n <= int(_o.environ.get("ARC2_PLE_MMAP_MAX_ROWS", "8192"))


def _arc2_mmap_gather(table, ids, out):
    import numpy as np
    st = getattr(table, "_arc2_mm", None)
    if st is None:
        fd2path = {fd: p for p, fd in table._fds.items()}
        paths = sorted({fd2path[fd] for fd, _ in table.shard_base})
        index = {p: i for i, p in enumerate(paths)}
        maps = [np.memmap(p, dtype=np.uint8, mode="r") for p in paths]
        shard_file = np.array([index[fd2path[fd]] for fd, _ in table.shard_base], dtype=np.int64)
        shard_off = np.array([off for _, off in table.shard_base], dtype=np.int64)
        st = (maps, shard_file, shard_off, np.arange(table.row_bytes, dtype=np.int64))
        table._arc2_mm = st
    maps, shard_file, shard_off, cols = st
    shard, local = np.divmod(ids.astype(np.int64, copy=False), table.shard_rows)
    offs = shard_off[shard] + local * table.row_bytes
    if len(maps) == 1:
        out[:] = maps[0][offs[:, None] + cols]
        return
    files = shard_file[shard]
    for fi in np.unique(files):
        sel = files == fi
        out[sel] = maps[int(fi)][offs[sel][:, None] + cols]
"""


ARC2_DRAFT_VOCAB_FUNCS = """

# ---- ARC2-L4: reduced-vocabulary drafting (FR-Spec idea; the tonyd2wild recipe's mtp_draft_vocab.py generalized to
# a frequency-ranked id list). Each TP rank scores the listed ids that fall in its lm_head shard, the ranks sum into
# one [n, k] tensor (every column is owned by one rank), and the result goes into a full-width -inf tensor, so draft
# sampling and the rejection sampler see an ordinary distribution that is zero outside the list.
def _arc2_draft_vocab_k():
    import os as _os
    try:
        return int(_os.environ.get("QWEN4EXP_DRAFT_VOCAB", "0") or 0)
    except ValueError:
        return 0


def _arc2_draft_vocab_tables(model, vocab):
    cache = getattr(model, "_arc2_draft_vocab", None)
    if cache is not None:
        return cache
    import json as _json
    import os as _os
    if torch.cuda.is_current_stream_capturing():
        raise RuntimeError("ARC2 draft vocab: tables must be built before CUDA graph capture (eager warm-up)")
    k = _arc2_draft_vocab_k()
    path = _os.environ.get("QWEN4EXP_DRAFT_VOCAB_FILE", "")
    if path:
        with open(path) as fh:
            ranked = [int(i) for i in _json.load(fh)]
        ids, seen = [], set()
        for i in ranked:
            if 0 <= i < vocab and i not in seen:
                seen.add(i)
                ids.append(i)
            if len(ids) >= k:
                break
    else:
        ids = list(range(min(k, vocab)))
    ids = sorted(ids)
    head = model.lm_head
    start = int(head.shard_indices.org_vocab_start_index)
    end = int(head.shard_indices.org_vocab_end_index)
    dev = head.weight.device
    mine = [(p, i) for p, i in enumerate(ids) if start <= i < end]
    all_ids = torch.tensor(ids, dtype=torch.long, device=dev)
    pos = torch.tensor([p for p, _ in mine], dtype=torch.long, device=dev)
    rows = torch.tensor([i - start for _, i in mine], dtype=torch.long, device=dev)
    weight = None
    if mine:
        w8_scale = getattr(head, "arc2_w8_scale", None)
        if w8_scale is not None:   # ARC2 W8A16 lm_head (int8 + scales): a dequantized bf16 table of the listed rows
            from vllm.model_executor.layers.arc2_w8a16 import arc2_w8_rows
            weight = arc2_w8_rows(head.weight, w8_scale, int(head.arc2_w8_group), rows)
        else:
            weight = head.weight.index_select(0, rows)
        weight = weight.contiguous()
    cache = (all_ids, pos, weight, len(ids))
    model._arc2_draft_vocab = cache
    from vllm.logger import init_logger as _il
    _il("vllm.arc2_draft_vocab").info(
        "ARC2 draft vocab: %d ids (%s), %d in this rank's lm_head shard [%d, %d)",
        len(ids), path or "ids 0..k-1", len(mine), start, end)
    return cache


def _arc2_reduced_vocab_logits(model, hidden_states):
    import torch.nn.functional as F
    from vllm.distributed import get_tensor_model_parallel_world_size, tensor_model_parallel_all_reduce
    vocab = int(model.config.vocab_size)
    all_ids, pos, weight, k = _arc2_draft_vocab_tables(model, vocab)
    n = hidden_states.shape[0]
    small = torch.zeros((n, k), dtype=hidden_states.dtype, device=hidden_states.device)
    if weight is not None:
        small.index_copy_(1, pos, F.linear(hidden_states.to(weight.dtype), weight).to(small.dtype))
    if get_tensor_model_parallel_world_size() > 1:
        small = tensor_model_parallel_all_reduce(small)
    full = torch.full((n, vocab), float("-inf"), dtype=small.dtype, device=small.device)
    full.index_copy_(1, all_ids, small)
    scale = getattr(model.logits_processor, "scale", 1.0)
    if scale != 1.0:
        full.mul_(scale)
    return full
"""

ARC2_QSA_UPDATE_METHOD = """    def update_draft_decode_metadata(self, metadata: QSAForwardMetadata) -> None:
        # ARC2-L4: rerun the metadata kernel on the same persistent buffers. The draft loop advances seq_lens and the
        # slot mapping in place, the token count and request layout stay fixed, so the outputs land in the tensors
        # this metadata already points to (capture-safe: one Triton launch, no host sync).
        common = metadata._arc2_common
        build_k_work = not self.is_circular_buffer and self.compress_ratio != 1
        build_qsa_metadata(
            common,
            self.token_to_req_buffer,
            self.logical_positions_buffer,
            self.visible_blocks_buffer,
            self.slot_mapping_buffer,
            storage_block_size=self.storage_block_size,
            compress_ratio=self.compress_ratio,
            circular_buffer_size=(self.kv_cache_spec.block_size if self.is_circular_buffer else 0),
            k_work_metadata_buffer=metadata.k_work_metadata if build_k_work else None,
            request_capacity=self.request_capacity if build_k_work else None,
        )
"""

# the W8A16 kernel module (w8a16_src.py), written to vllm/model_executor/layers/arc2_w8a16.py
ARC2_W8A16_MODULE = '\n# ---- ARC2-L4 INT8 weight-only GEMM (W8A16): y = x @ dequant(q, scale).T, x bf16 [M, K], q int8 [N, K] ----\n#   GROUP = 0:  scale fp32 [N]               (one per output row)\n#   GROUP = G:  scale fp32 [ceil(K / G), N]  (one per G consecutive weights of a row, group-major)\nimport json as _json\nimport os as _os\n\nimport torch\nimport triton\nimport triton.language as tl\n\n\n@triton.jit(do_not_specialize=["M"])\ndef _arc2_w8a16_kernel(\n    x_ptr, w_ptr, s_ptr, y_ptr,\n    M, N, K, K_PER,\n    stride_xm, stride_xk, stride_wn, stride_wk, stride_sg, stride_ys, stride_ym, stride_yn,\n    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, SPLIT: tl.constexpr, GROUP: tl.constexpr,\n):\n    # grid (M blocks, N blocks, K splits); each program reads its BLOCK_N int8 weight rows over its K range once\n    pid_m = tl.program_id(0)\n    pid_n = tl.program_id(1)\n    pid_k = tl.program_id(2)\n    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)\n    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)\n    mask_m = offs_m < M\n    mask_n = offs_n < N\n    k_lo = pid_k * K_PER\n    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)\n    if GROUP == 0:\n        rk = tl.arange(0, BLOCK_K)\n        for k0 in range(0, K_PER, BLOCK_K):\n            offs_k = k_lo + k0 + rk\n            mask_k = offs_k < K\n            x = tl.load(x_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk,\n                        mask=mask_m[:, None] & mask_k[None, :], other=0.0)\n            w = tl.load(w_ptr + offs_k[:, None] * stride_wk + offs_n[None, :] * stride_wn,\n                        mask=mask_k[:, None] & mask_n[None, :], other=0)\n            acc += tl.dot(x, w.to(x_ptr.dtype.element_ty))     # int8 -> bf16/fp16 is exact\n        s = tl.load(s_ptr + offs_n, mask=mask_n, other=0.0)\n        acc = acc * s[None, :]\n    else:\n        rg = tl.arange(0, GROUP)\n        for k0 in range(0, K_PER, BLOCK_K):                 # BLOCK_K and K_PER are multiples of GROUP\n            for u in tl.static_range(BLOCK_K // GROUP):\n                kg = k_lo + k0 + u * GROUP\n                offs_k = kg + rg\n                mask_k = offs_k < K\n                x = tl.load(x_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk,\n                            mask=mask_m[:, None] & mask_k[None, :], other=0.0)\n                w = tl.load(w_ptr + offs_k[:, None] * stride_wk + offs_n[None, :] * stride_wn,\n                            mask=mask_k[:, None] & mask_n[None, :], other=0)\n                s = tl.load(s_ptr + (kg // GROUP) * stride_sg + offs_n, mask=mask_n & (kg < K), other=0.0)\n                acc += tl.dot(x, w.to(x_ptr.dtype.element_ty)) * s[None, :]\n    y = y_ptr + pid_k * stride_ys + offs_m[:, None] * stride_ym + offs_n[None, :] * stride_yn\n    if SPLIT == 1:\n        tl.store(y, acc.to(y_ptr.dtype.element_ty), mask=mask_m[:, None] & mask_n[None, :])\n    else:\n        tl.store(y, acc, mask=mask_m[:, None] & mask_n[None, :])\n\n\n@triton.jit(do_not_specialize=["MN"])\ndef _arc2_splitk_reduce(p_ptr, y_ptr, MN, SPLIT: tl.constexpr, BLOCK: tl.constexpr):\n    pid = tl.program_id(0)\n    offs = pid * BLOCK + tl.arange(0, BLOCK)\n    mask = offs < MN\n    acc = tl.zeros((BLOCK,), dtype=tl.float32)\n    for s in tl.static_range(SPLIT):\n        acc += tl.load(p_ptr + s * MN + offs, mask=mask, other=0.0)\n    tl.store(y_ptr + offs, acc.to(y_ptr.dtype.element_ty), mask=mask)\n\n\n@triton.jit\ndef _arc2_w8_dequant_kernel(w_ptr, s_ptr, o_ptr, N, K, stride_sg,\n                            BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, GROUP: tl.constexpr):\n    offs_n = tl.program_id(0) * BLOCK_N + tl.arange(0, BLOCK_N)\n    offs_k = tl.program_id(1) * BLOCK_K + tl.arange(0, BLOCK_K)\n    mask = (offs_n < N)[:, None] & (offs_k < K)[None, :]\n    w = tl.load(w_ptr + offs_n[:, None] * K + offs_k[None, :], mask=mask, other=0)\n    if GROUP == 0:\n        s = tl.load(s_ptr + offs_n, mask=offs_n < N, other=0.0)[:, None]\n    else:\n        s = tl.load(s_ptr + (offs_k // GROUP)[None, :] * stride_sg + offs_n[:, None], mask=mask, other=0.0)\n    tl.store(o_ptr + offs_n[:, None] * K + offs_k[None, :], (w.to(tl.float32) * s).to(o_ptr.dtype.element_ty),\n             mask=mask)\n\n\n# (M bucket, N, K, GROUP) -> (BLOCK_M, BLOCK_N, BLOCK_K, SPLIT, num_warps, num_stages); BLOCK_M = 0 means\n# "dequantize + cuBLAS". Bucket 16/32/64 = decode-size M, 0 = larger M (prefill chunks). Measured on the Colab L4 by the\n# microbench (torch 2.11 + Triton 3.6); env ARC2_W8_CONFIGS_FILE adds/overrides ({"bucket,N,K[,GROUP]": [...]}).\nARC2_W8_CONFIGS = {\n    # per-row scales (results_w8_1, 22-oct; M=12, bf16 and int8 timed back to back)\n    (16, 62080, 2560, 0): (16, 32, 512, 1, 4, 4), (16, 336, 10240, 0): (16, 16, 256, 2, 4, 4),\n    (16, 10240, 320, 0): (16, 64, 128, 1, 8, 2), (16, 2560, 1536, 0): (16, 64, 256, 1, 8, 3),\n    (16, 320, 2560, 0): (16, 32, 256, 3, 4, 3), (16, 640, 2560, 0): (16, 16, 256, 1, 4, 4),\n    (16, 2560, 160, 0): (16, 64, 64, 1, 8, 3), (16, 320, 10240, 0): (16, 32, 256, 4, 4, 3),\n    (0, 4096, 2560, 0): (0, 0, 0, 1, 0, 0), (0, 336, 10240, 0): (64, 128, 64, 1, 4, 3),\n    (0, 10240, 320, 0): (128, 128, 32, 1, 4, 4), (0, 2560, 1536, 0): (0, 0, 0, 1, 0, 0),\n    (0, 3584, 2560, 0): (0, 0, 0, 1, 0, 0), (0, 320, 2560, 0): (0, 0, 0, 1, 0, 0),\n    (0, 640, 2560, 0): (64, 128, 64, 1, 4, 3), (0, 2560, 160, 0): (128, 128, 32, 1, 4, 4),\n    (0, 320, 10240, 0): (128, 64, 64, 1, 4, 3), (0, 24, 2560, 0): (0, 0, 0, 1, 0, 0),\n    # groups of 128 (results_w8_2, 22-oct; M=12 tuned, M=2048 against bf16 in the same pass)\n    (16, 10240, 320, 128): (16, 16, 128, 1, 2, 2), (16, 24, 2560, 128): (16, 16, 128, 4, 4, 4),\n    (16, 2560, 1536, 128): (16, 32, 256, 1, 8, 2), (16, 2560, 160, 128): (16, 32, 256, 1, 4, 4),\n    (16, 320, 10240, 128): (16, 32, 256, 4, 4, 3), (16, 320, 2560, 128): (16, 32, 128, 4, 4, 3),\n    (16, 336, 10240, 128): (16, 32, 256, 4, 8, 3), (16, 3584, 2560, 128): (16, 64, 256, 1, 8, 2),\n    (16, 4096, 2560, 128): (16, 32, 256, 1, 4, 2), (16, 62080, 2560, 128): (16, 32, 512, 1, 4, 3),\n    (16, 640, 2560, 128): (16, 32, 128, 4, 4, 3), (0, 10240, 320, 128): (0, 0, 0, 1, 0, 0),\n    (0, 24, 2560, 128): (0, 0, 0, 1, 0, 0), (0, 2560, 1536, 128): (0, 0, 0, 1, 0, 0),\n    (0, 2560, 160, 128): (0, 0, 0, 1, 0, 0), (0, 320, 10240, 128): (128, 128, 128, 1, 8, 2),\n    (0, 320, 2560, 128): (0, 0, 0, 1, 0, 0), (0, 336, 10240, 128): (128, 128, 128, 1, 8, 2),\n    (0, 3584, 2560, 128): (0, 0, 0, 1, 0, 0), (0, 4096, 2560, 128): (0, 0, 0, 1, 0, 0),\n    (0, 640, 2560, 128): (0, 0, 0, 1, 0, 0),\n    # groups of 128, decode sizes of 8 and 11 streams (w8_tune.zip, 28-oct; Colab L4, torch 2.13 + Triton 3.7.1,\n    # the L2 cold: weight copies > 2x the L2 per timed call); bucket 32: M=32 tuned, M=20/24 checked; bucket 64:\n    # M=44 tuned, M=36/48 checked; adopted when >= 3 % faster and not > 2 % slower at a check M\n    (32, 24, 2560, 128): (16, 16, 128, 4, 4, 3), (32, 320, 2560, 128): (16, 32, 128, 4, 4, 3),\n    (32, 320, 10240, 128): (16, 32, 256, 8, 4, 2), (32, 336, 10240, 128): (16, 32, 256, 4, 4, 3),\n    (32, 640, 2560, 128): (16, 32, 128, 4, 4, 3), (32, 2560, 160, 128): (16, 32, 256, 1, 4, 3),\n    (32, 2560, 1536, 128): (16, 32, 256, 1, 4, 2), (32, 3584, 2560, 128): (32, 64, 256, 1, 8, 2),\n    (32, 4096, 2560, 128): (32, 128, 128, 1, 8, 3), (32, 10240, 320, 128): (32, 64, 128, 1, 8, 2),\n    (64, 24, 2560, 128): (16, 16, 128, 4, 4, 3), (64, 320, 2560, 128): (16, 32, 128, 4, 4, 3),\n    (64, 320, 10240, 128): (16, 32, 128, 8, 4, 3), (64, 336, 10240, 128): (16, 64, 128, 6, 8, 3),\n    (64, 640, 2560, 128): (16, 32, 128, 4, 4, 2), (64, 2560, 160, 128): (16, 32, 256, 1, 4, 2),\n    (64, 2560, 1536, 128): (64, 64, 128, 1, 8, 3), (64, 3584, 2560, 128): (64, 64, 128, 1, 4, 3),\n    (64, 4096, 2560, 128): (64, 64, 128, 1, 4, 3), (64, 10240, 320, 128): (16, 64, 128, 1, 4, 2),\n    (64, 62080, 2560, 128): (64, 128, 128, 1, 4, 4),\n}\n_ARC2_SMS = {}\n\n\ndef _arc2_load_configs():\n    path = _os.environ.get("ARC2_W8_CONFIGS_FILE", "")\n    if path:\n        with open(path) as fh:\n            for key, cfg in _json.load(fh).items():\n                parts = [int(v) for v in key.split(",")]\n                if len(parts) == 3:\n                    parts.append(0)\n                ARC2_W8_CONFIGS[tuple(parts)] = tuple(int(v) for v in cfg)\n\n\n_arc2_load_configs()\n\n\ndef _arc2_num_sms(device):\n    idx = device.index if device.index is not None else torch.cuda.current_device()\n    if idx not in _ARC2_SMS:\n        _ARC2_SMS[idx] = torch.cuda.get_device_properties(idx).multi_processor_count\n    return _ARC2_SMS[idx]\n\n\ndef arc2_smem_bytes(bm, bn, bk, ns):\n    """Shared memory of the pipelined loop (Triton 3.5-3.7 on sm_89): (stages - 1) x (x tile + int8 w tile)."""\n    return max(1, ns - 1) * (bm * bk * 2 + bk * bn)\n\n\ndef arc2_m_bucket(M):\n    return 16 if M <= 16 else (32 if M <= 32 else (64 if M <= 64 else 0))\n\n\ndef arc2_w8a16_config(M, N, K, sms, group=0):\n    """(BLOCK_M, BLOCK_N, BLOCK_K, SPLIT, num_warps, num_stages). Decode-size M: enough programs to cover the SMs\n    twice, splitting K when the weight has few rows; larger M (prefill chunks): a plain tiled GEMM. With groups,\n    BLOCK_K is a multiple of the group."""\n    tuned = ARC2_W8_CONFIGS.get((arc2_m_bucket(M), N, K, group))\n    if tuned is not None:\n        return tuned\n    if M > 64:\n        return (64, 64, 64, 1, 4, 3) if group == 0 else (64, 64, group, 1, 4, 3)\n    bm = arc2_m_bucket(M)\n    target = 2 * sms\n    bk = 256 if K >= 2048 else (128 if K >= 256 else 64)\n    if group:\n        bk = max(bk, group)\n    bn = None\n    for cand in (64, 32):\n        if triton.cdiv(N, cand) >= target:\n            bn, split = cand, 1\n            break\n    if bn is None:\n        bn = 16 if N <= 4096 else 32\n        split = max(1, min(16, triton.cdiv(target, triton.cdiv(N, bn)), K // (2 * bk)))\n    while bk > max(64, group) and arc2_smem_bytes(bm, bn, bk, 3) > 64 * 1024:   # well under the L4\'s 99 KB\n        bk //= 2\n    return bm, bn, bk, split, 4, 3\n\n\ndef arc2_w8a16_linear(x, wq, scale, group=0, cfg=None):\n    """x [..., K] bf16/fp16, wq int8 [N, K], scale fp32 ([N] if group == 0 else [ceil(K/group), N]) -> [..., N]."""\n    shape = x.shape\n    x2 = x.reshape(-1, shape[-1])\n    M, K = x2.shape\n    N = wq.shape[0]\n    if M == 0:\n        return torch.empty((*shape[:-1], N), dtype=x.dtype, device=x.device)\n    if cfg is None:\n        cfg = arc2_w8a16_config(M, N, K, _arc2_num_sms(x.device), group)\n        if M > 64 and _os.environ.get("ARC2_W8_PREFILL", "triton") == "dequant":\n            cfg = (0,) + tuple(cfg[1:])\n    bm, bn, bk, split, nw, ns = cfg\n    if bm == 0:     # \'dequant + cuBLAS\' (env, or a tuned large-M entry with BLOCK_M = 0)\n        return torch.nn.functional.linear(x2, arc2_w8_dequant(wq, scale, x.dtype, group)).reshape(*shape[:-1], N)\n    if group and bk % group:\n        raise ValueError(f"ARC2 W8A16: BLOCK_K {bk} is not a multiple of the scale group {group}")\n    stride_sg = scale.stride(0) if group else 0\n    y = torch.empty((M, N), dtype=x.dtype, device=x.device)\n    k_per = triton.cdiv(triton.cdiv(K, split), bk) * bk\n    split = triton.cdiv(K, k_per)\n    grid = (triton.cdiv(M, bm), triton.cdiv(N, bn), split)\n    if split == 1:\n        _arc2_w8a16_kernel[grid](x2, wq, scale, y, M, N, K, k_per,\n                                 x2.stride(0), x2.stride(1), wq.stride(0), wq.stride(1), stride_sg,\n                                 0, y.stride(0), y.stride(1),\n                                 BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, SPLIT=1, GROUP=group, num_warps=nw, num_stages=ns)\n    else:\n        part = torch.empty((split, M, N), dtype=torch.float32, device=x.device)\n        _arc2_w8a16_kernel[grid](x2, wq, scale, part, M, N, K, k_per,\n                                 x2.stride(0), x2.stride(1), wq.stride(0), wq.stride(1), stride_sg,\n                                 part.stride(0), part.stride(1), part.stride(2),\n                                 BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, SPLIT=split, GROUP=group,\n                                 num_warps=nw, num_stages=ns)\n        mn = M * N\n        _arc2_splitk_reduce[(triton.cdiv(mn, 1024),)](part, y, mn, SPLIT=split, BLOCK=1024, num_warps=4)\n    return y.reshape(*shape[:-1], N)\n\n\ndef arc2_w8_dequant(wq, scale, dtype=torch.bfloat16, group=0):\n    """int8 [N, K] + scales -> dense [N, K] (the \'dequant + cuBLAS\' path for large M)."""\n    N, K = wq.shape\n    out = torch.empty((N, K), dtype=dtype, device=wq.device)\n    _arc2_w8_dequant_kernel[(triton.cdiv(N, 32), triton.cdiv(K, 256))](\n        wq.contiguous(), scale, out, N, K, scale.stride(0) if group else 0,\n        BLOCK_N=32, BLOCK_K=256, GROUP=group, num_warps=4)\n    return out\n\n\ndef arc2_w8_dequant_torch(wq, scale, group=0):\n    """fp32 dense weight from int8 + scales (plain PyTorch)."""\n    if group == 0:\n        return wq.float() * scale[:, None].float()\n    return wq.float() * scale.t().float().repeat_interleave(group, dim=1)[:, :wq.shape[1]]\n\n\ndef arc2_w8_rows(wq, scale, group, rows, dtype=torch.bfloat16):\n    """Selected rows of the dequantized weight (e.g. the reduced-vocabulary draft table of an int8 lm_head)."""\n    q = wq.index_select(0, rows).float()\n    if group == 0:\n        s = scale.index_select(0, rows)[:, None].float()\n    else:\n        s = scale.index_select(1, rows).t().float().repeat_interleave(group, dim=1)[:, :wq.shape[1]]\n    return (q * s).to(dtype)\n\n\ndef arc2_w8a16_reference(x, wq, scale, group=0):\n    """Plain PyTorch reference of the kernel (fp32 math on the dequantized weight)."""\n    return (x.float() @ arc2_w8_dequant_torch(wq, scale, group).t()).to(x.dtype)\n\n\n@torch.no_grad()\ndef arc2_quantize(w, group=128, chunk_rows=4096):\n    """bf16/fp16/fp32 [N, K] -> (int8 [N, K], fp32 scales), symmetric (max |w| / 127): group == 0 -> scale [N] per\n    row; group > 0 -> scale [ceil(K/group), N], one per group of consecutive weights in a row (last group partial)."""\n    n, k = w.shape\n    q = torch.empty((n, k), dtype=torch.int8, device=w.device)\n    if group == 0:\n        s = torch.empty((n,), dtype=torch.float32, device=w.device)\n        for a in range(0, n, chunk_rows):\n            wf = w[a:a + chunk_rows].float()\n            sc = wf.abs().amax(dim=1).clamp_min(1e-12) / 127.0\n            q[a:a + chunk_rows] = torch.round(wf / sc[:, None]).clamp_(-127, 127).to(torch.int8)\n            s[a:a + chunk_rows] = sc\n        return q, s\n    ng = triton.cdiv(k, group)\n    s = torch.empty((ng, n), dtype=torch.float32, device=w.device)\n    for a in range(0, n, chunk_rows):\n        wf = w[a:a + chunk_rows].float()\n        if ng * group != k:\n            wf = torch.nn.functional.pad(wf, (0, ng * group - k))\n        wg = wf.view(wf.shape[0], ng, group)\n        sc = wg.abs().amax(dim=2).clamp_min(1e-12) / 127.0\n        q[a:a + chunk_rows] = torch.round(wg / sc[:, :, None]).clamp_(-127, 127).to(torch.int8).view(\n            wf.shape[0], ng * group)[:, :k]\n        s[:, a:a + chunk_rows] = sc.t()\n    return q, s\n\n\ndef arc2_quantize_rows(w, chunk_rows=4096):\n    return arc2_quantize(w, 0, chunk_rows)\n\n\n# ---- vLLM glue: custom op (opaque to torch.compile), int8 linear / lm_head methods, the post-load converter ----\nimport re as _re\n\nfrom vllm.logger import init_logger as _init_logger\nfrom vllm.model_executor.layers.linear import LinearBase, LinearMethodBase, UnquantizedLinearMethod\nfrom vllm.model_executor.layers.vocab_parallel_embedding import (\n    ParallelLMHead,\n    UnquantizedEmbeddingMethod,\n    VocabParallelEmbedding,\n)\nfrom vllm.utils.torch_utils import direct_register_custom_op\n\n_logger = _init_logger("vllm.arc2_w8a16")\n\n# never converted: GDN conv1d keeps its conv weights in a linear and reads them directly; the MoE router decides\n# which experts run (exact top-k matters more than its 1-2 MB)\n_ARC2_W8_NEVER = _re.compile(r"(?:^|\\.)(?:conv1d|gate)$")\n\n\ndef _arc2_w8a16_gemm(x: torch.Tensor, wq: torch.Tensor, scale: torch.Tensor, group: int) -> torch.Tensor:\n    return arc2_w8a16_linear(x, wq, scale, group)\n\n\ndef _arc2_w8a16_gemm_fake(x: torch.Tensor, wq: torch.Tensor, scale: torch.Tensor, group: int) -> torch.Tensor:\n    return x.new_empty((*x.shape[:-1], wq.shape[0]))\n\n\ndirect_register_custom_op(op_name="arc2_w8a16_gemm", op_func=_arc2_w8a16_gemm, fake_impl=_arc2_w8a16_gemm_fake)\n\n\ndef _arc2_apply(layer, x, bias):\n    y = torch.ops.vllm.arc2_w8a16_gemm(x, layer.weight, layer.arc2_w8_scale, layer.arc2_w8_group)\n    return y if bias is None else y + bias\n\n\nclass Arc2W8A16LinearMethod(LinearMethodBase):\n    """Installed by arc2_w8_convert after loading: layer.weight int8 [N, K] (the bf16 copy is freed), buffer\n    layer.arc2_w8_scale fp32 ([N] or [ceil(K/group), N]), int layer.arc2_w8_group."""\n\n    def create_weights(self, *args, **kwargs):\n        raise NotImplementedError("Arc2W8A16LinearMethod only replaces loaded unquantized weights")\n\n    def process_weights_after_loading(self, layer):\n        pass\n\n    def apply(self, layer, x, bias=None):\n        return _arc2_apply(layer, x, bias)\n\n\nclass Arc2W8A16EmbeddingMethod(UnquantizedEmbeddingMethod):\n    """int8 lm_head: logits through the same kernel. Not usable as an input embedding."""\n\n    def process_weights_after_loading(self, layer):\n        pass\n\n    def apply(self, layer, x, bias=None):\n        return _arc2_apply(layer, x, bias)\n\n    def embedding(self, layer, input_):\n        raise RuntimeError("ARC2 W8A16: this lm_head holds int8 rows and cannot serve as an input embedding")\n\n\ndef _arc2_swap_to_int8(mod, group, park=False):\n    w = mod.weight\n    q, s = arc2_quantize(w.data, group)\n    freed = w.numel() * w.element_size() - q.numel() - 4 * s.numel()\n    if park:   # ARC2-L4 W8_COMPACT: the int8 copy waits in host memory, so the bf16 freed below leaves no hole\n        q, s = q.to("cpu"), s.to("cpu")\n    mod.weight = torch.nn.Parameter(q, requires_grad=False)\n    mod.register_buffer("arc2_w8_scale", s, persistent=False)\n    mod.arc2_w8_group = group\n    return freed\n\n\ndef arc2_w8_convert(model):\n    """Env ARC2_W8_MODULES (regex searched in module names): those unquantized bf16/fp16 linears (and a matching\n    ParallelLMHead) become int8 + fp32 scales (env ARC2_W8_GROUP: 128 = per group of 128 weights, the default;\n    0 = per row). Logs what was converted and the largest unconverted ones."""\n    pattern = _os.environ.get("ARC2_W8_MODULES", "")\n    if not pattern:\n        return 0\n    group = int(_os.environ.get("ARC2_W8_GROUP", "128"))\n    if group not in (0, 64, 128, 256):\n        raise ValueError(f"ARC2_W8_GROUP={group}: use 0 (per row), 64, 128 or 256")\n    rx = _re.compile(pattern)\n    park = _os.environ.get("ARC2_W8_COMPACT") == "1"\n    parked = []\n    embed_ptrs = {m.weight.data_ptr() for _, m in model.named_modules()\n                  if type(m) is VocabParallelEmbedding and getattr(m, "weight", None) is not None}\n    done, freed, kinds, skipped = 0, 0, {}, {}\n    for name, mod in model.named_modules():\n        w = getattr(mod, "weight", None)\n        if not isinstance(w, torch.Tensor) or w.dim() != 2 or w.dtype not in (torch.bfloat16, torch.float16):\n            continue\n        is_linear = isinstance(mod, LinearBase) and type(getattr(mod, "quant_method", None)) is UnquantizedLinearMethod\n        is_head = isinstance(mod, ParallelLMHead) and type(getattr(mod, "quant_method", None)) is UnquantizedEmbeddingMethod\n        if not (is_linear or is_head):\n            continue\n        key = _re.sub(r"\\.\\d+\\.", ".N.", name)\n        if _ARC2_W8_NEVER.search(name) or not rx.search(name):\n            skipped[key] = skipped.get(key, 0) + w.numel() * w.element_size()\n            continue\n        if is_head:\n            if w.data_ptr() in embed_ptrs:\n                raise RuntimeError(f"ARC2 W8A16: {name} shares its weight with the input embedding (tied); "\n                                   "leave lm_head out of ARC2_W8_MODULES")\n            head_dtype = getattr(getattr(model, "logits_processor", None), "head_dtype", None)\n            if head_dtype is not None and head_dtype != w.dtype:\n                raise RuntimeError(f"ARC2 W8A16: logits head_dtype {head_dtype} reads {name}.weight directly; "\n                                   "leave lm_head out of ARC2_W8_MODULES")\n        device = mod.weight.device\n        freed += _arc2_swap_to_int8(mod, group, park=park)\n        if park:\n            parked.append((mod, device))\n        mod.quant_method = Arc2W8A16EmbeddingMethod() if is_head else Arc2W8A16LinearMethod()\n        kinds[key] = kinds.get(key, 0) + 1\n        done += 1\n    torch.cuda.empty_cache()\n    if parked:\n        arc2_w8_unpark(parked)\n    top = sorted(skipped.items(), key=lambda kv: -kv[1])[:12]\n    _logger.info("ARC2 W8A16: %d modules -> int8, scale group %s (%.2f GiB freed), pattern %r: %s | largest "\n                 "unconverted bf16 (MiB): %s", done, group or "row", freed / 2**30, pattern, dict(sorted(kinds.items())),\n                 {k: round(v / 2**20, 1) for k, v in top})\n    return done\n\n\ndef arc2_w8_unpark(parked):\n    """ARC2-L4 W8_COMPACT: every bf16 is freed and the cache emptied; the parked int8 weights and scales go back to\n    their GPU, packed into the space the bf16 weights left (same values; logged as \'ARC2 W8 COMPACT\')."""\n    dev = parked[0][1]\n    free0, _ = torch.cuda.mem_get_info(dev)\n    res0 = torch.cuda.memory_reserved(dev)\n    moved = 0\n    for mod, device in parked:\n        mod.weight = torch.nn.Parameter(mod.weight.data.to(device), requires_grad=False)\n        mod.arc2_w8_scale = mod.arc2_w8_scale.to(device)\n        moved += mod.weight.numel() * mod.weight.element_size() + 4 * mod.arc2_w8_scale.numel()\n    torch.cuda.empty_cache()\n    free1, _ = torch.cuda.mem_get_info(dev)\n    res1 = torch.cuda.memory_reserved(dev)\n    bad = [m for m, d in parked if m.weight.device != d or m.arc2_w8_scale.device != d]\n    if bad:\n        raise RuntimeError(f"ARC2 W8 COMPACT: {len(bad)} modules did not return to their GPU")\n    _logger.info("ARC2 W8 COMPACT: %d modules, %.2f GiB of int8 weights and scales back on the GPU; reserved "\n                 "%.2f GiB, free %.2f GiB (before the int8 came back: reserved %.2f, free %.2f)", len(parked),\n                 moved / 2**30, res1 / 2**30, free1 / 2**30, res0 / 2**30, free0 / 2**30)\n    return moved\n\n\ndef arc2_embed_to_host(model):\n    """ARC2-L4 EMBED_HOST (env ARC2_EMBED_HOST=1): every input embedding table (VocabParallelEmbedding, not the\n    lm_head) moves to pinned host memory and the GPU reads it through a UVA view, as vLLM\'s own CPU weight offload\n    does (get_accelerator_view_from_cpu_tensor). Same values; logged as \'ARC2 EMBED HOST\'."""\n    from vllm.utils.platform_utils import is_pin_memory_available\n    from vllm.utils.torch_utils import get_accelerator_view_from_cpu_tensor\n\n    if not is_pin_memory_available():\n        raise RuntimeError("ARC2 EMBED HOST: pinned host memory is not available")\n    moved, nbytes, seen = 0, 0, set()\n    for name, mod in model.named_modules():\n        if type(mod) is not VocabParallelEmbedding:\n            continue\n        w = getattr(mod, "weight", None)\n        if (not isinstance(w, torch.Tensor) or w.device.type != "cuda" or w.numel() == 0\n                or getattr(w, "_vllm_is_uva_offloaded", False) or w.data_ptr() in seen):\n            continue\n        seen.add(w.data_ptr())\n        host = torch.empty_like(w.data, device="cpu", pin_memory=True).copy_(w.data)\n        w.data = get_accelerator_view_from_cpu_tensor(host)\n        w._vllm_is_uva_offloaded = True\n        mod._arc2_host_weight = host\n        moved += 1\n        nbytes += host.numel() * host.element_size()\n    torch.cuda.empty_cache()\n    _logger.info("ARC2 EMBED HOST: %d input embedding table(s), %.2f GiB in pinned host memory (UVA view)", moved,\n                 nbytes / 2**30)\n    return moved\n'
print("ARC2 runtime patches:", apply_arc2_runtime_patches(VP), flush=True)

# ---- batch-sharded sampling (CFG EXTRA_ARGS): compute_logits_local for Qwen4ExpForCausalLM, as qwen3_5.py has it ----
_BSS_OLD = ("    def compute_logits(self, hidden_states: torch.Tensor) -> torch.Tensor | None:\n"
            "        return self.logits_processor(self.lm_head, hidden_states)\n\n"
            "    def get_mtp_target_hidden_states(self) -> torch.Tensor | None:\n")
_BSS_NEW = ("    def compute_logits(self, hidden_states: torch.Tensor) -> torch.Tensor | None:\n"
            "        return self.logits_processor(self.lm_head, hidden_states)\n\n"
            "    def compute_logits_local(self, hidden_states: torch.Tensor) -> torch.Tensor:\n"
            "        # ARC2-L4: this rank's vocabulary slice, for --enable-batch-sharded-sampling (as in qwen3_5.py)\n"
            "        return self.logits_processor(self.lm_head, hidden_states, skip_gather=True)\n\n"
            "    def get_mtp_target_hidden_states(self) -> torch.Tensor | None:\n")
print("ARC2 bss patch:", _arc2_edit(MP / "model.py", [(_BSS_OLD, _BSS_NEW)], "ARC2-L4 bss patch"), flush=True)

# FULL MODEL: no pruning step here; MODEL_DIR stays the Intel W4A16 AutoRound checkpoint of cell 1 (512 experts per layer).
# read-ahead over the files the server will read
precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})

# ---- the MTP draft: albucino's INT4 group-32 checkpoint inside the drafter input ----
def _draft_dir(root):
    found = []
    for cfg_path in Path(root).rglob("config.json"):
        q = json.loads(cfg_path.read_text()).get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            found.append(cfg_path.parent)
    return _unique(found, "INT4 g32 MTP draft checkpoint")


DRAFT_DIR = _draft_dir(DRAFT_MODEL_DIR)
print("MTP draft:", DRAFT_DIR, flush=True)

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = Path(PREFIX, "cache")
cache.mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_HOME": str(cache / "hf"),
    "XDG_CACHE_HOME": str(cache), "TRITON_CACHE_DIR": str(cache / "triton"), "VLLM_CACHE_ROOT": str(cache / "vllm"),
    "VLLM_ENGINE_READY_TIMEOUT_S": "3600", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_CUDA_ARCH_LIST": "8.9", "NCCL_P2P_LEVEL": "NVL", "NCCL_CUMEM_ENABLE": "0", "VLLM_USE_DEEP_GEMM": "0",
    "VLLM_USE_V2_MODEL_RUNNER": "1", "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    # the regular sampler (batches without drafts) with vLLM's own top-k / top-p instead of flashinfer's, which drew
    # 1.13 % of its tokens outside that support (Colab L4, results_emu_6); the rejection sampler never used flashinfer
    "VLLM_USE_FLASHINFER_SAMPLER": "0",
    # the recipe's disk-backed PLE table, "staged": 16 rows per token gathered into a fixed GPU buffer
    "QWEN4EXP_PLE_MMAP": "1", "QWEN4EXP_PLE_STAGED": "1", "QWEN4EXP_PLE_MMAP_THREADS": "32",
    "QWEN4EXP_PLE_MODEL_DIR": str(MODEL_DIR),
    "OMP_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
})
# ARC2-L4 RecoverSSM: on, with the Triton GDN decode kernel it needs (until 28-oct: vLLM's fused CUDA one)
env.update({"VLLM_GDN_DECODE_KERNEL": "triton", "VLLM_ARC3_GDN_RECOVERSSM": "1"})

# ---- prefix caching with MTP: mark the drafter's KV groups (patch of the installed kv_cache_utils.py) ----
# vLLM flags the KV groups that hold drafter layers for the EAGLE/MTP last-block drop; it finds them only by a spec
# marker (DSpark) or by model type (DeepseekV4). For this MTP drafter it finds none, flags every group, the Mamba
# groups cannot satisfy that, and prefix reuse drops to zero (run 6: "no KV cache group could be identified as the
# draft model's", 2.98M prompt tokens re-prefilled). The drafter's layers are registered after the target's
# num_hidden_layers ("<prefix>.layers.<N+i>") under the "mtp" prefix: flag the groups that hold one of those.
KVU = Path(_unique(Path(VENV).glob("lib/python*/site-packages/vllm/v1/core/kv_cache_utils.py"), "kv_cache_utils.py"))
_KVU_ANCHOR = '''    if not use_deepseek_v4_fallback:
        return
    last_layer = next(reversed(kv_cache_spec))
'''
_KVU_PATCH = '''    # ARC2-L4: MTP drafter layers come after the target's num_hidden_layers ("<prefix>.layers.<N+i>")
    if spec_config.method == "mtp" and not any(g.is_eagle_group for g in kv_cache_groups):
        _arc2_n = getattr(vllm_config.model_config.hf_text_config, "num_hidden_layers", None)
        if _arc2_n:
            import re as _arc2_re
            for _arc2_i, group in enumerate(kv_cache_groups):
                for _arc2_name in group.layer_names:
                    _arc2_m = _arc2_re.search(r"(?:^|\\.)layers\\.(\\d+)\\.", _arc2_name)
                    if (_arc2_m and int(_arc2_m.group(1)) >= _arc2_n) or _arc2_re.search(r"(?:^|\\.)mtp\\.", _arc2_name):
                        group.is_eagle_group = True
                        logger.info("ARC2-L4 MTP draft KV group %d (layer %s)", _arc2_i, _arc2_name)
                        break
'''
_kvu_backup = KVU.with_name(KVU.name + ".l4-orig")
if not _kvu_backup.exists():
    _kvu_backup.write_bytes(KVU.read_bytes())
_kvu_src = _kvu_backup.read_text()
if _kvu_src.count(_KVU_ANCHOR) != 1:
    raise RuntimeError(f"{KVU}: the anchor of the MTP prefix-cache patch is not there once; run stopped")
KVU.write_text(_kvu_src.replace(_KVU_ANCHOR, _KVU_PATCH + _KVU_ANCHOR))
for _pyc in KVU.parent.glob("__pycache__/kv_cache_utils*.pyc"):
    _pyc.unlink(missing_ok=True)
print("vLLM patch: MTP draft KV groups flagged for prefix caching", KVU, flush=True)

# ---- optional: vLLM's custom all-reduce on the 4 PCIe GPUs (patch of the installed custom_all_reduce.py) ----
# vLLM turns its custom all-reduce off for more than 2 PCIe-only GPUs (performance, not correctness: the kernels use
# P2P loads, which the L4s support -- run 23 reported P2P on all pairs). With ARC2_CUSTOM_AR_PCIE=1 it is treated as
# fully connected, used only for messages up to ARC2_CUSTOM_AR_MAX_BYTES (decode: 12x2560 bf16 = 60 KB; prefill
# chunks stay on NCCL), and its all-gather / reduce-scatter stay off. Without the variable nothing changes.
CAR = Path(_unique(Path(VENV).glob("lib/python*/site-packages/vllm/distributed/device_communicators/custom_all_reduce.py"),
                   "custom_all_reduce.py"))
_CAR_EDITS = [
    ("            fully_connected = current_platform.is_fully_connected(physical_device_ids)\n",
     "            fully_connected = current_platform.is_fully_connected(physical_device_ids) or (\n"
     "                __import__(\"os\").environ.get(\"ARC2_CUSTOM_AR_PCIE\") == \"1\")  # ARC2-L4\n"),
    ("        if self.world_size == 2 or self.fully_connected:\n            return inp_size < self.max_size\n",
     "        if self.world_size == 2 or self.fully_connected:\n"
     "            return inp_size < min(self.max_size, int(__import__(\"os\").environ.get(\"ARC2_CUSTOM_AR_MAX_BYTES\", self.max_size)))  # ARC2-L4\n"),
]
_CAR_GATHER = "            and (self.fully_connected or bool(self.mnnvl_multicast_ptr))\n"
_CAR_GATHER_NEW = ("            and ((self.fully_connected and __import__(\"os\").environ.get(\"ARC2_CUSTOM_AR_PCIE\") != \"1\")"
                   " or bool(self.mnnvl_multicast_ptr))  # ARC2-L4\n")
_car_backup = CAR.with_name(CAR.name + ".l4-orig")
if not _car_backup.exists():
    _car_backup.write_bytes(CAR.read_bytes())
_car_src = _car_backup.read_text()
for _old, _new in _CAR_EDITS:
    if _car_src.count(_old) != 1:
        raise RuntimeError(f"{CAR}: an anchor of the custom all-reduce patch is not there once; run stopped")
    _car_src = _car_src.replace(_old, _new)
if _car_src.count(_CAR_GATHER) != 2:
    raise RuntimeError(f"{CAR}: the all-gather / reduce-scatter anchors are not there twice; run stopped")
CAR.write_text(_car_src.replace(_CAR_GATHER, _CAR_GATHER_NEW))
for _pyc in CAR.parent.glob("__pycache__/custom_all_reduce*.pyc"):
    _pyc.unlink(missing_ok=True)
print("vLLM patch: custom all-reduce on PCIe available with ARC2_CUSTOM_AR_PCIE=1", CAR, flush=True)

# flashinfer JIT-links its kernels with -lcuda; the Kaggle image has only the driver's libcuda.so.1 (no libcuda.so,
# no CUDA stubs), so the link failed on the first sampling call (run of 19-oct). A libcuda.so symlink on LIBRARY_PATH.
_libcuda = None
try:
    for _ln in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
        if "libcuda.so.1 " in _ln and "x86-64" in _ln and "=>" in _ln:
            _libcuda = _ln.split("=>")[1].strip()
            break
except Exception:
    pass
if _libcuda is None:
    for _d in ("/usr/lib/x86_64-linux-gnu", "/usr/local/nvidia/lib64", "/usr/lib64", "/usr/local/cuda/compat"):
        if Path(_d, "libcuda.so.1").exists():
            _libcuda = str(Path(_d, "libcuda.so.1"))
            break
if _libcuda is None:
    raise RuntimeError("libcuda.so.1 not found: flashinfer cannot link its JIT kernels; run stopped")
_linkdir = Path(PREFIX, "linklib")
_linkdir.mkdir(parents=True, exist_ok=True)
if _linkdir.joinpath("libcuda.so").is_symlink() or _linkdir.joinpath("libcuda.so").exists():
    _linkdir.joinpath("libcuda.so").unlink()
_linkdir.joinpath("libcuda.so").symlink_to(os.path.realpath(_libcuda))
env["LIBRARY_PATH"] = str(_linkdir) + (":" + env["LIBRARY_PATH"] if env.get("LIBRARY_PATH") else "")
print("libcuda for JIT linking:", os.path.realpath(_libcuda), "->", _linkdir / "libcuda.so", flush=True)


# ---- ARC2-L4 optimizations -> server environment (CFG["OPT"]; every switch maps to env vars of the runtime patches) ----
# dense bf16 linears converted to int8 (module-name regex): hyper-connections (incl. the final mixer), GDN projections,
# QSA qkv/o and indexer, shared expert, lm_head. Never: GDN conv1d, the MoE router (enforced by the patch too).
ARC2_W8_DENSE = (r"hyper_connection(_mixer)?\.input_mix_weight|linear_attn\.(in_proj_qkvz|in_proj_ba|out_proj)$"
                 r"|self_attn\.(qkv_proj|o_proj)$|index_qk_proj$|shared_expert\.(gate_up_proj|down_proj)$|(^|\.)lm_head$")
# the drafter's reduced vocabulary: 77 special ids, then the ids of the solver's outputs (runs 6/21/22, 60 tasks) by
# frequency; 99.34% of the tokens of 20 held-out tasks. TP4 rows per rank: 7684 / 917 / 15 / 119.
ARC2_DRAFT_VOCAB_IDS = [248000,248001,248002,248003,248004,248005,248006,248007,248008,248009,248010,248011,248012,248013,248014,248015,248016,248017,248018,248019,248020,248021,248022,248023,248024,248025,248026,248027,248028,248029,248030,248031,248032,248033,248034,248035,248036,248037,248038,248039,248040,248041,248042,248043,248044,248045,248046,248047,248048,248049,248050,248051,248052,248053,248054,248055,248056,248057,248058,248059,248060,248061,248062,248063,248064,248065,248066,248067,248068,248069,248070,248071,248072,248073,248074,248075,248076,198,27,1628,28,12305,29,15704,1787,87,510,271,220,16,11,15,17,18,19,23,25,279,13,21,318,22,20,24,12,8,2713,506,369,283,11069,579,321,303,6761,264,1327,16502,328,314,30,553,82,6505,22812,2468,681,364,26,1,50615,310,440,1879,1590,6558,1829,262,1987,728,427,85152,2047,7515,513,421,413,1076,539,1245,633,81,5215,7,11393,694,6970,1892,494,49300,504,486,10,6784,60,9764,272,660,58,77264,424,2691,9809,3145,357,1844,1754,3498,682,561,1179,1404,5364,66,733,381,874,417,430,524,1228,0,285,14,10811,383,2424,5728,748,1716,4272,5893,1906,1095,4031,2020,10484,15656,9761,1133,1576,11795,353,1921,313,8984,585,478,466,498,2919,309,6,4944,1132,1332,1000,1541,11504,1788,6835,359,685,14362,914,1748,1510,1174,15725,2144,5445,436,7736,763,1017,5646,1441,680,799,578,7890,864,1683,12086,883,659,3002,2487,2297,3990,471,1042,4727,288,1092,13139,2006,3618,830,5972,3655,4844,6607,580,1353,1118,1970,9,245,2107,9146,24708,1040,71093,1719,491,599,388,92,3870,2234,3559,3008,22690,1752,9714,1690,10380,5978,12204,1330,567,84485,1500,2956,12579,3163,6820,259,2844,2515,3158,3899,1845,9599,63,1857,1048,20724,293,6961,3294,345,2695,17312,745,1141,4990,54,7481,7699,4678,1089,338,487,280,23214,2086,1307,14336,1066,6669,25164,975,348,2387,31,781,1410,5346,86,1368,3074,1250,1328,5637,1891,1411,13772,2911,8826,23194,727,1010,5759,1423,64,4838,311,3166,1220,72,460,28778,557,2,5821,6206,2984,12931,1673,3010,88842,21128,628,809,18522,20182,6491,1331,296,10521,1436,10480,281,15415,1246,49761,32645,1698,8240,8858,50821,777,10016,7278,2414,458,7179,1144,1257,3055,2400,36004,4282,482,791,12867,2282,1262,437,4147,3444,1780,1902,423,48264,1302,5000,2521,653,410,5396,1959,6681,14430,13876,4220,4613,20734,3313,449,2233,13784,1727,2785,65,2923,7772,1989,7025,9727,8529,79,20555,1881,18716,2404,19690,1083,4558,615,37781,12594,7403,351,3850,27736,32,629,4935,8798,1148,1103,888,42418,2423,26370,7946,2450,33,378,2100,1442,5571,462,94,3667,1723,6851,493,4965,5924,9378,1542,2397,6486,1866,20254,19868,1303,1543,35680,12331,948,43,52448,2814,3755,2019,2493,906,3369,6365,495,1151,2235,13007,37732,18009,411,8173,4206,19163,496,22048,15436,292,13211,4916,41895,1290,821,16602,13142,7010,15360,3165,20456,533,15949,642,7658,5601,7385,2717,919,1472,2880,2166,1445,37073,635,2982,9845,4391,25465,1305,29691,343,16550,3254,958,12558,4008,286,39,9896,1238,2962,2510,290,998,528,3753,716,1420,23985,7563,8213,2441,8410,44102,7701,17743,23769,8681,9989,4290,6599,3491,10560,49,13962,5122,4774,4161,1483,1324,2255,18169,256,35,326,15740,1521,2361,32375,8211,2829,397,10804,1362,15289,74,67,3437,641,6729,10029,8142,1345,1380,8574,6235,2353,4706,3070,577,446,3404,6762,44292,3566,550,3172,5799,15060,866,457,1809,9155,4732,3481,3710,25710,464,9542,3793,10663,5732,12784,12366,56896,5491,3706,4880,1680,35475,29203,5436,7624,11639,28028,6836,74994,3315,78,363,760,387,3944,26548,4087,43614,7054,13428,5435,1170,2119,4,2069,33887,5098,9808,1786,77,7904,1304,31091,9897,5956,957,2343,23598,10672,47875,15796,1558,17385,24935,72757,2784,3889,13640,1030,1301,350,22283,23235,12306,2697,71,11396,2798,4205,2129,804,69905,21172,623,1495,1056,5324,7167,59317,4778,6103,4139,1070,45690,2721,7435,83,16642,62236,12391,1960,8772,57538,5774,3953,3387,1889,274,22903,5605,14522,974,66702,12435,836,34410,1821,10300,995,2663,2944,1503,24515,258,269,1363,3621,3501,5748,4060,20648,53,49211,1105,40,16234,8582,735,16904,12219,620,73174,20618,7110,2526,10962,1381,23697,795,1765,2597,5505,85,695,9590,7007,9828,4362,593,73,3808,4374,8097,709,9409,16996,6747,9933,4534,5951,15484,17291,2381,646,1782,4430,500,20450,1532,6483,169361,34,2425,3512,9987,49629,8328,4417,722,2800,4808,5686,9868,1117,25721,7014,28945,361,4370,2224,2929,6992,68278,1669,7461,19388,2073,59624,492,304,8236,3015,886,5226,7995,81404,29094,56484,470,4255,4833,7441,23964,76,4035,2998,4045,13427,4072,1923,4650,5055,49222,1172,16097,414,1277,14255,7739,4924,8639,38704,12261,4058,8442,2185,63368,2980,10458,9087,14505,34124,1143,51,1049,6575,18090,606,1560,11283,2873,17294,46726,871,18950,1518,3580,14103,2445,2083,16424,43143,42824,2336,3181,4364,12611,1602,2007,7697,14595,27653,4831,5469,18146,13854,380,18922,5081,1379,70,3515,20791,9111,1856,2018,307,1288,81122,1684,1204,4138,5313,88,17592,708,39248,3164,6833,4010,52519,90739,8387,21517,61880,4816,20970,22057,1818,13031,2198,1705,837,14387,12583,2939,5251,1439,295,2702,9785,31712,8910,794,68,2394,2330,1149,869,2590,11882,982,523,1877,7916,2574,1994,2195,12090,2167,10014,15346,172800,3525,3620,1396,67619,5181,6699,4799,2985,11861,9639,6703,30854,34346,17962,10973,48142,73165,1387,2842,469,22746,7640,6560,22632,907,5184,3065,1944,6160,10263,7099,719,9637,35497,52,2645,942,15309,64498,2099,2168,47488,4062,11378,652,8307,16327,2640,3555,835,17397,2715,6530,17754,8866,3272,7920,177036,16563,267,1550,16874,73585,15623,291,29169,7620,38854,2332,8626,4115,17044,22774,2624,7949,28997,2834,4055,33153,6220,26321,2759,3136,2250,2348,17425,27700,23424,3102,13862,12813,6411,46819,67521,26808,616,66473,44404,8359,2959,639,11872,10738,7127,14159,1527,2922,2683,2308,4581,1165,21531,10092,33702,3521,45193,3222,3050,8882,8421,1608,1718,14053,12450,12587,2212,82078,91569,1459,8874,26401,4252,429,10219,39950,13402,1189,10110,6643,4203,1440,4598,18101,11650,15003,47944,12929,19013,51390,4177,29629,5643,4489,4786,5981,45065,9884,19382,40984,9257,14848,11173,9170,1599,30863,1429,15804,6686,3069,57172,6489,70575,26351,65975,3274,35684,2582,426,4307,12454,10582,14973,70735,9055,5533,85232,13559,20374,70617,64191,4145,3497,18721,38818,8516,65059,68142,1804,1942,23323,8141,534,18814,242,1104,65787,22258,25098,3446,9150,2610,6632,36059,10578,42100,6942,32107,1817,73406,18483,36,41042,74483,6000,6821,3840,4891,20564,34217,11597,3023,22271,3736,7561,17454,23310,10287,4210,62,3681,27379,27916,8871,3449,21244,29130,2272,2843,2094,26621,23783,1926,65704,16218,4109,4437,40181,420,90,1067,11223,4425,1534,2639,14378,1724,48819,6234,67981,73401,9628,1801,6157,261,17640,8938,7551,13333,61898,1767,4623,11683,1308,1865,59432,1248,83947,13628,7386,1925,51344,609,696,41662,55979,2434,5979,4961,8439,38495,23958,20006,362,2173,11096,59056,22134,25153,87826,854,26587,851,1402,3366,26612,25181,7840,1120,23556,52920,7526,15734,5426,3608,2892,6550,4156,12103,33208,4064,3672,11767,16168,19754,3568,7728,30748,15625,5722,9496,22718,6136,4992,11543,2688,55,17759,7869,8906,1097,50863,11053,22773,7587,64173,2532,12215,10744,4202,13601,11394,6999,16528,13077,488,2382,51159,42058,18941,11654,54444,955,4471,1264,3377,632,3296,2061,5257,403,7692,22645,43138,7334,13634,64045,23295,84818,8222,37725,1142,4543,16866,94698,3337,4101,20495,5675,25170,3092,12832,1406,9236,631,3482,8311,23144,14902,9766,4565,61310,22221,12187,2945,5433,2751,4570,34131,2136,67964,20385,6435,11800,1428,6977,1990,8205,7030,26386,12259,12665,43938,4752,13849,7063,931,1126,5090,8366,11930,1212,4128,967,2543,40731,396,10694,1831,17800,60052,23856,11712,3284,23833,4523,2583,28115,18761,1586,55294,912,20735,299,25284,9712,35931,15395,7918,5265,33060,26843,13398,3383,352,9469,17384,3335,48282,2905,29430,61934,91,391,15958,2957,6082,5117,26602,3993,7123,15451,20865,7346,9281,13102,2407,15087,3728,1309,47069,2420,7875,19018,5553,54455,1908,22525,38453,11496,43001,5033,69,5369,58634,27155,80,23850,2479,54102,15707,14856,10355,42103,3314,3579,1393,6513,39931,14549,66926,2426,8014,448,1467,22676,2313,38553,73971,47095,53915,3349,5144,2243,1898,34765,24242,5629,8385,10229,1146,4047,984,3043,316,23756,6316,16045,219689,3058,75959,8618,18630,1395,374,9001,12167,30835,588,1562,767,306,3105,12807,20861,4563,12277,2746,19710,1764,22030,9952,9813,4088,11220,6060,25493,10957,3905,9337,2222,10342,4406,65409,10424,1061,71766,5303,71552,16547,8317,38787,31136,8443,1870,2592,56999,1551,10021,11040,14107,2098,3835,840,46723,7145,6576,277,13534,1973,3480,29619,4507,5984,1446,33105,8341,988,36188,10304,20758,8977,54946,45881,13881,1012,1284,17077,10242,5617,3750,21094,211178,9238,12482,916,69777,331,2586,57296,4796,20032,47,70927,9042,3614,9284,14273,14081,12354,24443,19671,7618,8845,4568,266,10512,9264,2302,6348,8166,1060,20621,73979,1347,9376,354,4592,29566,3088,15672,33251,4221,787,25278,11641,6033,61869,10917,1528,103,61551,23044,81367,68924,20740,13927,2091,23098,86201,1091,4082,18830,1325,270,1316,1630,13239,24009,13264,12945,8268,12663,23331,7065,4776,5038,43364,11244,17544,65393,2322,12615,23677,30003,11823,4125,1454,39794,4283,10562,6507,5691,3902,2921,18465,59678,3359,17589,60637,3913,10598,49231,4829,2754,592,71222,7249,925,50,3799,28924,1318,13600,7610,7426,22730,4089,10646,49693,78025,2830,7940,55787,31521,5372,3175,5384,13554,37848,33472,9328,4222,27224,7936,4645,1691,15387,287,6588,15464,29588,94090,46429,12846,5615,945,23359,2379,65086,12748,5480,6521,2324,8099,30101,9740,17515,1027,26817,902,87244,13765,22898,8701,12948,11937,61,9167,814,12041,62096,12237,39147,44543,20631,5357,51234,4886,2542,15218,1375,31030,51442,67802,275,323,69330,36546,38,74206,32254,11784,4535,17076,23064,17437,52850,10124,35313,12092,6195,22985,18200,1236,8020,91050,4549,34852,4918,5443,1496,1783,12317,24919,3107,45021,71563,9261,14086,4178,6463,41139,4766,2888,11863,1110,11461,9956,8615,14180,4097,1868,1520,7060,28284,327,1547,21605,29015,8198,8169,8176,21500,1848,3804,53440,6149,44,1021,58317,3162,382,2519,3788,21642,20460,2072,20321,5176,9687,3749,55775,20908,8597,3833,8030,3075,24170,48781,723,28966,7459,22005,5020,34276,16878,263,70307,13581,47675,12351,2477,48743,24681,19777,16682,3759,7896,21262,14272,27545,6817,8839,4884,17831,477,5136,7225,522,2311,6150,75,7320,1159,1634,39874,3384,1310,27692,2085,951,18091,34618,31153,39674,1112,13611,9536,10279,5572,31092,23150,33267,19809,10426,6356,15648,9063,7367,548,7415,46055,944,2388,390,15888,23478,1575,80570,2342,62586,5568,8919,9202,349,17311,3544,6745,163401,4194,18188,257,39965,265,8187,6248,20952,1650,118,33358,93021,20653,11009,6857,8645,15133,10309,87245,30603,19565,27235,74492,3665,10071,8106,19458,18569,3140,14235,5792,3138,19902,6006,71084,1919,3184,64258,31768,3018,17554,16606,1506,3456,19201,23283,83317,16648,33932,2496,11319,41984,88884,89,11658,10381,45465,5511,7795,13157,74270,7637,25786,4945,3433,12719,10853,22294,4277,1249,4928,1412,10213,83662,1494,14892,15644,35742,3602,2399,3798,40039,6694,11128,41445,45771,1364,14943,23716,37431,7268,3810,9431,9057,37456,87834,18431,28408,62597,25659,714,87197,20834,3884,638,687,2943,33268,53396,3743,18402,5946,16924,14545,721,31338,5961,23443,31966,2518,5252,4777,10451,21402,32566,2839,20391,20125,8047,69892,1018,35944,17345,3429,463,2806,10702,10264,1453,3923,3590,28106,22687,3668,9646,27864,2577,15840,8610,2458,32471,72149,6991,18498,4473,9907,19384,4920,14135,6922,25133,7213,1640,14137,68894,33696,11359,34439,2977,5559,10299,39539,11116,2876,5183,4845,22800,5216,66486,47464,25708,542,56507,43413,29617,9071,60018,45632,33742,4006,4081,6379,42377,3124,5976,57739,47526,806,6391,19213,6807,45047,68393,6134,3863,1946,37844,19470,973,4983,60922,541,73643,82518,1861,45687,18805,18539,656,4016,1293,57589,31352,3376,2971,4334,69180,2627,82248,51245,10542,2676,4692,8003,70966,6187,14058,13146,810,76289,535,86539,3921,19646,4639,33094,325,19307,23681,2735,8252,30982,4821,33107,13004,4336,5576,17933,594,89480,1478,42360,27173,19860,14142,15029,18255,19011,51824,71633,44646,27469,17661,24725,46,77267,9692,26463,58115,13224,38627,6570,26565,38625,56374,190953,63653,46633,7033,26147,5291,94487,26148,2262,34045,13695,8752,3571,793,14577,83880,5493,8280,4603,19186,8806,1200,78297,21252,12771,25791,47532,3269,12709,2680,5072,9060,16360,16296,47563,6125,13205,59771,2741,52065,71821,44025,12174,29812,5016,21685,19133,19329,47771,18664,2940,28238,4483,10695,9607,8554,467,75711,3583,9775,4864,5247,79657,27396,7062,29270,77303,29044,21365,40550,32410,12975,26227,69736,936,54035,85349,78919,1366,23851,4702,47198,79141,50465,6504,404,13468,1529,228,1676,21173,12456,23185,94569,30116,9664,16073,28892,9422,50776,876,51380,6804,4284,6066,29458,1603,14411,4688,25334,4947,848,371,74614,2483,5514,5138,26899,320,1319,36213,4564,3795,3202,21192,13744,23314,8404,18167,1052,87687,61107,3977,10379,23410,7664,15924,25704,11051,5587,5935,549,34795,6696,18657,1283,12495,3562,39811,21343,9197,2707,16439,19055,34672,56830,13538,42108,33528,59276,53977,86499,19352,20003,56470,5272,10386,11140,8448,4191,2912,2714,5312,5284,8937,42151,1014,7450,1100,17572,22741,9245,23173,2665,5474,14377,55045,8761,9239,23024,3095,13687,35265,56611,35923,8295,53041,2995,48444,15836,29243,11844,6566,17582,28170,14623,337,24358,3311,13983,7044,51161,18264,6750,6405,344,38394,56140,77340,13095,38696,807,6778,60503,52696,13996,6298,28224,43967,53454,26475,26487,2871,48880,4657,4524,401,2965,92510,19033,59810,1956,14195,35175,16938,24530,586,2942,5499,8732,10926,35617,2758,4492,3101,3765,51775,5791,7830,913,11878,9584,45993,24217,25103,4004,35732,490,20770,20275,3600,35629,13571,334,5640,21902,10184,74366,27891,2790,46124,90010,1760,10002,31276,76360,14330,17164,31049,2508,12523,50717,79562,14459,6960,2276,17448,1373,87502,18081,1505,54938,45569,24921,2564,8103,1559,3288,89997,35893,6021,33935,640,3007,6927,53624,11811,11149,40486,2536,61501,79046,8388,30520,1733,9596,29890,53163,73681,37201,21513,3160,21220,3946,2874,7465,12518,9002,26277,10539,574,14844,13587,24219,2904,82848,12244,3322,12996,84591,3986,5471,24600,1822,6892,33582,7072,70684,9037,66073,102,16042,12680,24501,15771,11134,18290,40028,643,26824,1988,43487,3922,4240,53159,29689,5408,4590,5876,12564,4165,25467,31996,21826,9514,45531,34717,91074,13236,3439,20796,48601,847,3374,34468,15263,6829,2503,8796,33363,33010,5106,26547,44157,94439,17758,9405,8677,12323,41020,35542,28652,12904,31686,24917,18253,17038,21985,17068,11121,3084,28685,764,668,16201,308,5875,3063,2267,6849,801,18334,26035,4289,31291,10337,3529,6907,50799,11726,4265,4246,7683,65511,648,92115,33448,24572,3309,31639,60517,88558,3842,52659,27433,37,3533,7019,2763,13976,12037,17239,1209,33767,63319,6369,21649,13457,5242,44154,803,6427,7324,1546,3238,28377,3213,12609,47022,3019,4653,24635,11566,27193,2502,16722,80612,14561,3234,15693,26806,33939,77291,2908,6549,5274,21901,9971,13717,5627,51703,14460,4319,51520,3420,5947,37405,39019,2835,14708,27222,1958,8308,3232,3619,10033,34222,8012,2600,39867,9861,14031,12502,5074,23251,2907,4860,7156,51842,2191,46379,34290,72812,21652,4682,18236,11320,26025,51192,35539,13942,1167,24330,7247,33159,53057,15486,88515,17622,1899,35793,86806,10729,25075,84,13684,82188,54587,669,5693,58463,4515,40323,6995,13215,27588,58920,52569,26326,64968,4782,9252,17463,1414,55671,37797,53893,32735,27870,2413,15027,7472,10869,28450,7159,1777,537,78062,24979,74153,60296,16159,6063,53187,72337,16673,7462,1647,43070,1426,67759,29973,80403,3658,3267,10103,3258,16915,5402,16298,6306,2894,22090,56785,16609,4119,84776,70915,4382,339,8379,55950,4876,1593,20775,60735,2867,57892,5392,11421,28498,12726,406,15049,868,87565,11961,50390,67463,6915,36536,10087,12891,28365,39964,5769,3467,25662,17085,9516,12218,70543,20973,3679,12025,32283,9077,6923,2029,8298,16417,786,7077,83876,65134,51623,10921,7762,4499,11822,997,5653,20561,4937,2993,54133,85154,49388,560,12863,32524,884,49407,42911,12540,2902,1188,12441,46522,2961,23982,10390,77632,8846,40023,46564,21516,9973,35718,773,4258,6094,7059,14995,68118,2126,12639,2777,14540,12511,45106,16075,4546,14122,2070,219792,10659,157526,943,27676,347,5461,33782,158,36802,13458,9834,6503,17366,33601,21144,11925,5839,13975,17664,62910,14520,15064,42326,10277,35614,2429,19715,77142,28216,1701,13439,6830,60428,13240,30535,5263,8355,73332,41009,519,14477,9744,4462,60180,26010,5651,3336,3354,66856,44716,12486,1469,8065,2433,1452,12571,56738,26023,2022,6093,38449,10548,43254,1802,7559,10605,14890,23853,81029,4181,31128,15688,1825,4356,13094,2617,40890,27593,15767,38127,27658,58793,27929,34237,55090,71300,21076,39129,9447,10583,11569,20624,21219,3460,91311,89321,3282,18961,13081,3520,40373,14661,74604,39054,75458,52444,5308,1940,51742,38098,68841,6004,13362,6132,5064,52565,39752,45899,29440,56004,73546,16856,978,64726,1757,11177,30308,1242,5536,1918,19407,5306,3493,74669,13632,473,37857,15736,16429,24781,558,10625,43470,21415,59141,3455,5335,665,4285,5775,18794,52736,35727,7867,11048,28254,12339,40525,65575,9434,13501,9826,22702,16484,3418,18051,42962,5330,8996,34970,9936,7095,9446,3445,4681,1218,21974,77615,7776,45203,15546,53470,5776,5758,87117,15262,5488,1488,16018,16546,6668,60594,38585,19175,10643,69687,31126,85465,8771,51586,18868,4159,18092,24160,7968,10250,17541,15893,16162,26554,1121,9019,63858,1323,4286,68960,21998,62916,1254,31502,22824,595,2677,5356,1713,71194,15896,241326,3071,9395,24344,57893,284,10341,7172,69631,9889,12152,50921,79762,302,8156,50233,8219,22958,37367,79894,1616,232,11948,7354,2706,34282,47720,38402,16469,36642,67549,509,91543,576,64381,22287,8390,12083,9069,6978,20307,2179,182186,26876,408,90570,10639,2301,5527,35983,41339,20073,39914,20679,21351,2225,31560,40388,20961,14875,13706,9302,9737,22832,10847,231442,9711,90072,70613,16661,33662,10286,32426,8524,10273,7294,49310,2278,59609,18655,12102,12405,701,25162,3082,5823,5331,55687,4095,11258,7463,29014,16022,64358,1943,3531,39619,3875,2140,7662,45847,83481,11947,23065,23294,42328,35166,17955,6737,10269,5904,45347,41439,27848,5684,3636,692,40565,55774,71339,9263,39094,6217,11326,56163,1910,5141,20981,6716,45326,18176,981,49540,3661,3113,69458,3522,48230,7368,3012,8406,21814,6707,19934,6958,8877,1597,7478,3447,22152,3402,947,20306,24755,21301,10666,25730,17265,4332,6249,1536,6625,2299,11990,5096,25709,3837,94411,43181,11347,8174,2294,5855,1405,9121,27900,23061,2652,52907,35704,30009,22151,79920,86725,47848,22238,29051,8804,25286,46729,25152,34338,13838,2589,2211,12806,80088,28240,28078,1761,9976,19504,4952,26849,12076,7075,32016,33811,26543,46590,35234,1131,22255,56341,6813,5781,1287,46525,6791,431,7911,20447,2708,177884,19072,30945,20184,447,18280,41356,2658,47535,71068,12682,39751,16240,20040,35141,7401,40857,19194,34016,20665,16384,34163,4415,22501,83127,45776,11737,42655,5802,1704,485,6517,19306,7598,9576,86403,15808,3639,2315,6948,21019,23342,27484,10866,2306,32052,4830,1422,1176,8872,1020,35374,20260,55020,7284,3927,26334,6295,41523,16167,11717,27230,15327,93742,20539,4190,17198,60436,1201,1755,1954,22703,11702,340,2972,22419,14374,1326,85993,58118,1897,63456,839,24851,27336,81728,14445,1219,18839,14733,18971,1062,5132,58251,69149,24734,27129,91599,41664,7933,18597,26123,7629,15124,15250,39302,7428,12717,3896,17024,1114,38409,45163,57398,35894,58183,4361,4826,5606,4057,6743,19542,3278,28895,6310,18285,16695,18154,50113,273,6679,30090,46991,9363,31179,87075,46025,20482,1579,14408,23371,2050,5334,12408,93485,2978,39295,93,15702,9997,45277,5787,34032,25806,211614,32228,24235,18082,51301,5129,36450,56992,5943,7252,14519,5438,21716,11960,33129,13690,26756,52434,3651,17380,17145,9970,18677,1742,29648,94170,400,33095,4418,40019,940,16905,21342,58423,9293,7606,6744,65611,9191,13325,23287,39013,4699,31005,2364,4162,13653,3479,7900,12877,13992,21612,4230,8427,20331,3928,87987,3591,36414,40699,17108,46728,5878,12910,90765,15069,49591,27362,63879,88749,8334,39358,14789,23540,62266,2973,6343,27478,4888,14979,3816,3879,39850,7854,57081,1175,30296,419,66524,2723,22819,20500,14170,3167,11870,613,8508,9878,21103,3122,9370,13741,3051,8719,56240,16610,8755,2500,5,7834,25113,2402,27823,23571,29246,278,10018,5150,62906,1235,9244,29618,14184,8056,11088,4009,9240,7308,11147,24722,40581,12189,9301,40507,1231,55333,57804,7718,63098,1992,44182,16842,27101,13066,78589,8210,86518,456,29457,25393,6130,23107,83603,3187,14683,33395,3009,21965,15463,49264,30372,19878,14553,55492,3000,15969,12593,51748,14012,3291,4217,6469,2571,79174,14205,8791,11988,4490,2200,18375,8668,591,56606,24660,2599,867,3874,13578,26079,3079,5721,13555,332,18586,37193,35790,27365,2406,2257,12755,9945,11883,8217,48088,57710,3204,12669,54738,18788,39017,9209,27068,13235,1791,4299,2113,684,40523,5394,91689,12692,13818,853,34291,2456,1515,22027,35937,5198,5224,40195,9703,47983,4434,57355,13873,19114,1652,76068,40143,14934,1685,79517,26821,14268,2220,22365,25082,16025,31150,5365,45719,42258,16944,4486,16565,3595,24582,11365,16468,20843,36853,19598,1565,927,35076,3428,12830,12705,1762,38710,76828,41275,64649,21518,5532,8032,9579,4496,3123,1064,2803,479,60797,5902,28488,2958,499,63908,6983,18533,10709,7081,2805,25552,28068,5752,1581,12239,842,10517,15359,1785,32480,9272,673,9495,26259,58725,12908,20543,23057,43327,17201,26933,8306,9831,59833,75505,3325,9182,15842,924,8985,73972,29830,29101,1569,1267,45035,713,85779,6934,29616,4804,16676,27245,16005,64990,8635,54369,10298,24749,46065,85934,8102,64806,12858,14625,3110,28979,2074,23535,55670,5456,12964,30722,56391,920,57735,24782,12804,20877,33831,15797,6481,74700,83312,23277,11966,74708,13875,7230,6198,58990,90931,6171,870,15371,24463,5530,45462,11278,83524,6250,4747,26192,85236,13353,17251,15838,82505,18464,3720,62991,48975,17538,20896,34580,59,80103,9192,72632,13660,15050,234511,3873,44148,228701,2699,68780,9459,79969,1447,79805,12792,10306,91546,39356,40922,5167,1122,15305,21559,88129,4111,47326,4929,16807,11722,10708,16520,1826,8060,27794,220921,14429,11013,5281,11132,18053,5255,59666,66186,1653,35299,8855,10817,2609,324,10635,33777,15297,10551,305,30787,9205,54821,38400,294,23475,3367,23814,7609,5339,18755,1145,2491,6973,9381,5964,1425,40142,11313,7621,53953,1487,9290,8083,3431,15818,11605,26184,34192,7021,9476,29408,11079,547,24016,62366,25128,21835,322,4980,3054,12193,1901,10020,3727,86635,11227,33756,12752,82911,4021,37435,6848,4472,226001,85285,23365,72094,90336,55516,16749,41616,2719,750,5305,30251,35821,3803,14344,88336,2218,2773,6087,14452,35525,33303,17763,1229,46767,14311,34535,45349,6197,53238,21033,1340,30753,8714,7148,14290,44350,39740,83274,882,4670,82808,29527,3523,7832,13426,11651,29933,37356,27446,18125,78479,25700,7174,21765,11576,4574,15440,9611,57371,15715,1029,9168,23879,1435,5816,29184,25314,3152,19492,32191,19287,10457,5917,2538,5515,1223,51099,778,37990,4346,14663,738,20837,8255,45127,581,28830,13677,20139,39983,21254,30035,4407,17299,11211,4228,27654,15394,28183,23592,75305,1633,77385,46617,34030,25064,86272,16033,4536,53459,32713,24148,61486,25636,898,44232,22672,78582,29694,4389,18473,932,24080,7538,89768,25663,3648,36618,22627,80487,48047,8884,26260,48575,42898,10691,79062,11746,11164,20364,8689,5309,17839,24773,69060,3956,5822,18002,60059,4487,4661,43650,68336,4409,15577,14147,12269,6695,15866,8397,2149,49314,10959,16253,20797,11229,23493,30108,68600,58333,63157,30747,365,13733,47103,13273,3052,8717,1830,20412,54789,5529,44318,2781,33633,37717,44193,33061,33093,6866,18377,32346,16574,9940,68763,36586,5158,28461,32940,43508,13054,18781,33660,23470,4005,28662,63765,10226,5048,13027,60756,5125,14787,51541,6018,35398,50231,76475,6327,39075,14258,7876,70508,5199,4386,1932,2101,6554,60656,8471,15885,62219,14835,15752,11946,15673,79111,88204,565,91037,18939,81645,49198,44548,2716,87110,32505,8472,7440,26641,8474,4660,30634,28649,6975,4478,3436,7101,14746,14372,66232,5642,82115,7589,69402,17561,39741,12417,19279,6487,35426,10378,24447,21236,37488,14796,5834,33714,61794,12525,7556,4278,8017,11901,18172,10172,2560,6728,62862,11747,21725,27335,26569,3255,42144,35477,80019,5486,11290,21164,1335,6001,4621,9797,7359,22846,19613,61274,54004,10400,3032,21068,23515,15511,16757,52294,1139,47283,2909,31498,15435,26068,75032,24974,9770,48086,31855,60777,2614,91122,10746,95,34094,20958,747,1732,34867,85146,47558,31859,32127,1589,1438,3996,78030,56481,63896,1720,45079,39565,10128,21106,65419,6057,89759,16740,6946,42398,26211,12650,14868,47632,34973,55158,8340,37514,44159,19139,3707,12463,16406,18326,21774,33993,3041,20114,11222,43766,514,39441,31625,12713,50640,6951,81499,11656,10630,36823,66064,19383,70842,25766,86297,29482,8344,65861,22977,36071,16060,29208,16829,12333,54087,49622,27180,30047,43716,68761,47781,76563,47286,44738,6311,36667,43020,70007,25602,49024,1298,93984,442,1671,43038,4800,38339,42182,6236,2626,44900,5983,6642,72870,1086,67765,80273,73123,20004,7757,25432,56757,11342,706,83562,14653,31558,66969,49724,77470,44777,17067,40423,1849,2916,13181,5173,13011,2818,356,1065,38645,13749,192398,25504,8367,59315,14657,1239,11081,9566,15059,3490,2556,732,11918,67313,16921,13761,1746,7183,84502,40289,16674,24604,163719,4026,31896,57471,8887,15196,76168,11559,32388,59749,78301,77333,4484,53297,29261,10177,1196,2352,243004,2341,25511,18389,38074,35088,9816,3507,60210,86377,4325,2231,36704,44079,195406,11837,34344,35000,69416,1544,505,7304,8722,26164,16456,51883,14913,7427,5087,12836,58479,34773,25689,84554,169553,14265,22103,9885,10813,12961,19766,6420,22756,39410,29977,32290,13902,710,54551,15491,15679,23238,30260,6355,3761,9007,2153,56931,34322,3216,1009,5347,8797,698,38094,53140,315,9043,9461,4052,10895,10845,15445,434,24614,23532,34532,31519,4237,41553,11964,9501,7860,24663,27355,27526,51102,20537,56378,5555,11514,15689,11154,1769,30738,8046,10851,398,21078,9718,2147,60826,25471,5516,42029,25984,5208,46398,80625,25705,27573,3143,35481,525,48518,22791,62088,65297,48611,27917,12597,2194,82537,66762,82729,39736,24546,30245,11256,9084,47932,12461,34337,6508,4441,91938,6237,1665,75275,5933,6966,35257,33403,28504,49332,85000,79744,79442,61899,36146,6552,2664,21257,26104,5743,45197,2774,62966,32335,12234,40568,1525,28845,169629,38197,18479,70573,622,22637,62634,8005,19888,5351,20980,8623,79593,55317,31063,24188,24101,72792,3072,94002,8412,7390,20678,24401,26634,5954,23721,5373,48594,11264,12089,11693,22845,4758,15106,11440,61228,9959,15615,9888,13542,385,82206,1962,44098,17458,62475,32002,12283,36783,34082,7080,51948,9184,81705,87454,214679,23423,47506,27958,46449,7289,13673,7850,3950,41863,8944,12291,9855,35778,5304,20663,2438,2565,45607,48821,5425,741,15433,8900,783,94252,78161,5091,84575,11721,238,37093,22825,7638,63281,31103,880,4847,10545,39997,25353,41804,443,18772,57385,13787,8058,27107,8481,21083,11336,10734,18198,2133,25751,21820,12760,8225,3772,22096,10951,24295,2177,42896,82170,13823,3785,63600,59111,9101,13870,90463,31650,36381,13128,435,5918,6867,55288,7520,39691,65316,8978,27662,74430,45251,79733,41866,5586,9704,5231,37638,238816,18486,4713,2545,48755,12054,52152,59454,4497,14347,5148,32626,32892,41244,6818,11135,10771,30442,10130,14244,231807,87533,21693,44697,22284,13769,28673,66180,3221,66783,27210,2823,8794,9502,199662,9407,72346,12854,30766,47962,12962,2338,433,1282,80745,6891,57208,16369,12659,3154,5922,71592,41223,12777,28715,9283,34150,6182,6381,18207,4625,26802,11429,45262,21104,44926,1382,22916,9722,3704,19221,79572,27376,91015,13185,7234,38646,52483,21533,19228,26162,1005,16583,67190,1004,49991,86851,26943,2725,25456,2373,25216,88142,93609,15722,21426,67157,33041,38788,11346,32683,1448,42256,19752,31886,44917,33773,26952,21354,63851,4331,2155,15239,11352,80313,38425,29313,10596,5077,23823,20629,88086,72493,21627,12131,3066,7157,5307,80602,23521,3150,19873,66755,54084,73119,43274,1409,472,30379,1693,23950,21200,23709,38649,23346,64448,39928,43365,30917,28244,82520,51021,3397,508,3598,850,53596,86015,24903,2789,30697,11137,21793,92898,49465,14082,2537,60568,79307,11591,22980,10751,25888,3438,55657,33779,24460,69505,80620,24304,32224,61603,22602,24904,86034,16739,41211,13451,16496,13064,40988,37024,24102,5337,9780,44896,4234,31626,22166,30218,20590,9758,56364,57901,11324,1957,63933,8363,17673,21024,66230,24057,9768,58133,41260,930,7236,36980,7438,82525,10503,95561,2858,19181,21699,3733,16777,9175,29607,6859,3978,36519,16325,85062,3564,36828,9674,15421,34035,6177,3815,15233,2112,36633,10027,20170,3572,74945,4068,20523,818,17991,31733,4561,13057,10104,70313,28926,4618,7783,40541,29582,14029,36182,9153,28946,82353,14703,48402,10268,18526,56255,5687,25711,79207,3106,8452,51343,10490,20918,34769,41221,42018,1928,39792,8309,68173,1374,7381,46004,42939,33822,94506,28700,71433,2259,69472,95516,3198,4275,15611,17321,15999,4384,9064,1059,1457,3854,60162,3787,46137,19025,7789,71219,5620,7276,15919,1867,65319,11018,52689,3318,2359,57815,29133,41955,45209,4480,4970,24557,14833,11107,64804,32503,12985,52140,48621,2540,27340,67373,26007,1993,72622,3714,27841,58831,67654,46336,33333,28016,33534,9751,32235,2926,37556,18223,12581,670,66426,15146,8396,6568,14585,84141,73560,76458,62987,6361,46992,1799,57,4054,44207,50878,2002,33776,33108,11304,38849,27776,29625,1596,65048,18148,368,12728,7731,3845,39284,26137,15207,1124,33713,47681,7596,32827,11985,36751,52883,49992,14773,29201,8588,7584,40846,74322,11555,7881,15783,32958,27664,72878,28837,14749,7506,58173,3040,3191,24407,14760,2512,1947,48184,26709,376,5666,11544,4105,73802,33667,21261,46705,2428,4044,14981,9602,2329,38468,2992,91147,15119,35845,13358,4721,16088,16275,11855,74923,3766,46292,26288,10152,95001,46678,2052,34059,37685,14254,12914,22866,5073,37372,61809,68803,11248,53230,28177,31654,59543,33669,52666,12927,3858,53703,22440,29981,19111,36965,34356,39826,2495,4725,21831,40693,93856,55689,95707,2241,21863,6638,27092,22236,1511,1609,30810,19401,8218,46361,20174,73129,27962,31248,168977,27209,5676,61674,18282,27443,11832,237766,1171,68874,91483,12896,3935,10365,54006,3683,3604,13724,1860,22382,9563,30346,31637,35091,80113,5604,7744,28486,63871,3287,3464,18710,29942,22732,28215,13305,25387,752,3094,37316,54409,37794,14908,38030,6649,30946,45,16743,7418,6202,20741,50240,2043,7039,1880,461,897,8293,24869,18779,62127,24316,21312,10519,18913,36933,60333,89391,2213,6725,59036,31127,24297,70206,47997,30152,13143,2734,12098,25322,2528,94868,4823,30596,20266,2410,12497,64861,1803,14193,28821,82923,64915,16995,25396,4126,16917,42992,70504,5418,5323,51505,28671,20335,4575,8160,77474,84290,6518,9477,45552,45328,31114,56328,36885,15495,12230,9927,42559,14245,11610,1605,13643,438,26241,3689,20218,10377,2686,10155,607,17399,7262,56427,22442,13948,27032,18315,11545,40748,17030,91291,40355,36733,9325,62700,8177,1313,4637,25891,67528,24186,17723,45522,42051,94247,2791,11910,4245,159029,49283,624,3611,1654,54266,30374,44183,53897,62076,11969,8920,30456,1806,52293,6526,60770,68687,627,18643,805,1996,17249,10165,5134,26949,26619,28714,4233,16876,3820,2109,14248,2427,74574,13609,20852,34112,62243,50915,64294,21701,21328,65262,3641,40638,16421,21482,4866,28049,4979,1128,4862,22373,53188,59273,82490,1116,29578,15184,71623,50877,31220,17171,11125,24184,10814,51699,445,13214,44712,17750,10454,17675,512,79149,23379,11000,48382,2376,21729,48786,2899,4176,3131,84171,1876,11741,86379,12851,2054,35306,3049,45374,76308,53007,52735,23191,13522,35873,26760,14629,61161,80265,8807,55404,24765,5680,21478,6037,9781,93531,12217,52275,11205,8155,12116,935,87882,18797,55786,19867,15113,3938,62261,66172,9468,27735,8398,4695,53058,11848,1202,7578,501,13920,677,41403,71344,11995,9738,10116,26588,1344,30027,83784,16338,74988,71655,10431,24254,23008,35538,5170,83082,2932,4243,8231,961,64151,7290,1187,17320,32188,17083,26661,27642,48482,11974,94639,94834,25794,2885,9347,3417,9357,54572,4573,71312,12943,1449,7880,3826,32160,4034,11233,22254,39407,928,62086,79602,27837,1493,3975,72399,37796,17977,15803,73815,17748,33807,15773,3599,14314,37759,10931,544,92535,30871,829,2010,53416,8686,46120,79336,10970,4530,15006,50453,9340,61160,23989,12794,24147,6759,34574,83924,57248,72963,224024,25773,56330,3631,333,26507,42000,46142,7964,568,40802,20427,1320,47822,13351,9450,24268,55792,6434,1549,6805,4803,67584,7023,95023,9485,27389,9930,48030,2762,18912,41963,6941,1660,12296,85527,20827,21228,26604,14590,27041,50110,10616,44237,6213,22273,36793,254,276,62131,14542,5818,7965,27572,6749,31439,2612,63272,31540,98,9149,9013,3400,67115,53678,3024,41499,22006,16086,84398,28712,41763,22651,40444,19815,234,13864,46925,16249,63809,6525,47079,15733,10442,12837,22235,79603,16128,69542,84243,55502,39287,31982,30165,18709,18055,86045,14603,33241,5911,7321,877,71126,12566,15710,13505,11532,19794,25212,7784,8851,3135,49516,5871,6058,44480,23986,12490,66102,22869,57060,12630,48,8779,16681,15388,776,15156,89293,87162,47957,15062,26152,12125,8833,8666,43400,11218,8034,3738,40664,57496,32979,27372,34310,1674,14035,16382,62984,60584,37140,5344,30390,41582,11411,10036,6799,20028,17396,44142,35288,26880,33075,563,19001,3192,41050,14513,48716,19481,77884,49364,41514,20382,9297,37792,30600,56928,10126,47930,32895,6454,2391,1968,63471,8190,23016,7700,17932,49017,56571,32689,10505,39664,20536,21621,34674,7374,33265,1401,3702,94718,59993,33017,81944,85160,80818,4267,246,26343,6734,8139,4306,611,61569,11179,15288,15339,28151,32328,14543,2228,28822,26957,65114,24200,21849,3627,46625,67295,8594,16760,985,5912,28253,52783,57020,19654,4694,16141,4435,50415,246696,25364,8952,4188,911,52393,27718,1519,74927,474,54237,57501,20477,41973,29663,68290,28075,19635,42780,80832,78296,536,44782,48470,30344,10633,35604,69849,17973,5143,59626,45398,44095,82698,68792,38097,52486,600,20600,35222,38406,6957,12965,3955,1964,31082,19933,13280,47622,89562,66532,58435,78845,75504,66903,58855,20471,18516,35880,27879,75724,6450,30692,31192,29976,63520,9862,5447,22629,86678,17860,8553,88408,16337,8869,6766,36650,5745,268,2117,49563,71709,77269,20440,6698,3494,43380,4350,1658,5737,16800,13209,46485,8868,6611,34105,30885,29322,51874,13968,46348,68071,24028,76079,18890,1386,26289,16714,562,61837,52915,65737,43311,298,79118,29784,2732,21207,5028,895,60662,35761,15175,8613,64561,20703,5690,10312,20444,18466,1433,63987,4562,21146,7873,14457,8028,13736,29783,4674,2967,52761,878,45978,6387,30647,13860,13596,27494,50940,89346,13746,11467,8563,13140,6596,12953,35116,17591,55047,23857,3038,15275,6826,16368,13301,16261,73916,12718,12909,49918,6302,66990,2096,19248,79596,4450,77518,908,88144,9215,3364,27709,15749,21947,19119,13471,58432,7474,3758,82990,21035,7573,18211,40415,8993,18131,7285,25141,8928,61672,625,7710,74872,3584,6630,1531,25177,42424,23304,83822,12857,42166,678,31954,43900,67895,55024,1044,15867,757,3915,4193,29554,11239,1875,15036,47696,72795,29378,24244,10591,8466,2131,235673,11762,73096,83999,32365,41012,31157,17098,1855,2475,8825,58248,2060,45373,1533,25304,18880,6708,3929,47953,4519,14717,32878,37020,4394,52298,36590,14572,65584,4791,57106,10392,6064,18235,10324,60947,8055,17290,45759,8915,8698,49882,72562,10700,16988,407,82391,6199,14166,87528,14105,21780,68540,61695,21681,85978,5350,17928,1311,58001,63285,31233,36524,2604,11047,13910,65235,73728,664,48397,25277,87470,42803,16813,22230,13665,92002,3931,1180,7246,3919,18150,56400,56801,3381,29026,6803,21711,21854,16347,559,32487,8733,59365,11309,4495,5310,6702,9660,3495,22275,71771,58925,92209,59501,22208,50054,41513,54792,44094,3693,5149,12262,41982,49282,16494,31251,950,10952,19988,25067,3756,26674,5444,87773,1514,65903,49825,57417,13909,39846,60489,66480,89031,35608,89848,33992,77019,5292,55864,17957,45309,43618,5843,44921,7945,70141,66526,47241,55348,92840,1951,52264,11432,9610,13039,53829,802,51515,962,42672,28267,9036,53736,31130,4104,5075,81100,7909,19315,35672,37611,44877,45991,2270,10800,2573,66352,8941,14084,29112,42613,38133,89864,9200,49869,5192,76086,74468,91068,57265,29998,47065,38429,11036,66769,94939,29592,64452,13230,47993,14830,1763,8857,39110,27043,66925,1577,77154,7193,55336,5027,70527,16000,16407,92309,18916,24507,7652,12613,9789,25828,43703,991,2674,2621,20286,8019,63398,81987,51070,73516,1214,85596,39745,83960,22116,7994,53253,17498,1789,39655,58807,10641,47737,14548,31972,3443,10137,7260,14610,71537,41058,9343,41,1646,7142,66538,6061,27738,86771,18547,47491,9190,12533,1611,58694,41554,47914,38323,23426,452,93395,17930,11962,64839,24959,14294,52615,19744,686,10615,37224,53983,71755,859,4213,90651,34995,2471,70320,47587,20268,4341,82191,19208,21010,10936,10416,3462,5832,4442,25971,8287,25522,1980,10782,40889,71813,4963,9671,1046,966,25503,11330,14213,2554,30764,89288,22147,76980,2765,233,43022,31133,36883,1234,47364,6587,6604,84919,56377,2193,5110,46676,9321,6758,7243,6806,8523,64285,4238,27522,10159,49382,15699,22316,6886,8184,8024,59876,2579,13590,26482,1317,59514,45727,47455,18391,56300,6600,1416,82000,1490,82364,57036,12922,23293,9156,42903,13190,5836,29350,61774,14371,61199,3360,27614,24469,6925,22081,14155,12368,29797,31315,11263,8622,10083,5101,5886,14594,31789,64969,32607,50706,52886,14186,14777,1883,46499,3949,15324,80555,89621,89012,46261,22979,64240,11084,36981,21475,1367,15724,20406,4075,7383,1255,8747,545,9960,30285,82431,51692,18275,5259,13624,8602,20826,27403,61030,59042,15627,19747,3093,289,1016,19447,4941,7554,12488,50029,2924,481,21650,22597,35052,20671,771,18307,2058,209218,10281,43774,29768,8499,375,11782,42370,70546,16619,22482,3307,44722,48846,45975,22041,16100,28446,5761,50885,6441,3388,65837,2064,62943,5756,1008,57610,20878,52993,42768,35649,35750,85553,58518,26859,497,12112,79041,91338,3474,8737,45492,6105,4751,91841,1137,81080,20270,9640,2316,7956,88217,636,59965,53577,45891,7553,24716,23544,34098,4195,25654,39167,5012,26644,3792,9361,3684,47140,252,46668,75688,12431,78858,16531,86987,68573,1853,9592,50042,14482,31342,7139,65827,21551,8332,27661,43615,40950,23495,7862,3514,704,37403,13699,43188,8592,38154,7926,51274,14095,4199,11533,7676,14816,90860,11909,50083,894,11315,10856,14431,66590,65025,88050,61732,77684,89057,6349,37847,12870,13372,1350,15031,518,78486,2334,25034,69104,23819,78249,31777,11690,18897,77302,56933,10727,62373,13087,40624,78244,66299,14208,41697,14569,76871,14050,575,13756,6470,59521,4984,93435,75234,20043,21359,14080,11541,37198,4001,32897,37202,617,43510,73117,39089,32343,37887,5478,63113,5794,11181,32073,78785,24053,3708,4762,35070,9739,33218,39141,21778,14293,94646,28825,51899,49695,92494,45351,14260,92984,861,27743,11206,31917,42476,38522,10717,52943,27560,23718,81715,16140,32327,43374,34078,3112,19639,12917,9072,5882,81022,2009,11195,66390,20392,25587,47160,5830,15571,5531,1292,17246,9790,10631,5938,6769,80996,16158,78365,56111,47493,3868,65977,7745,4868,30324,47573,18743,12477,26209,81194,48073,64696,27477,7874,70507,8276,33431,39390,34414,68769,37169,9890,5197,65022,35516,72359,80018,8595,41590,15361,18608,583,48055,1916,20098,6196,3692,23377,43736,71750,22487,28100,34771,64951,12196,1610,394,37908,44898,50133,73858,26126,5927,56373,5982,4801,4911,76582,69567,41224,1625,7872,841,2549,520,32345,48219,47246,17765,54703,33586,16675,82759,60101,44827,25333,9380,17130,4611,28133,22188,51695,16873,20945,16021,37276,41054,16466,16067,58185,22957,82024,66585,23046,36141,8926,14862,20632,507,87923,219055,14226,57573,6658,17440,2955,71571,15812,13293,20957,38365,67608,21704,92707,56168,15123,22445,11279,89291,29964,5378,59049,87686,12071,17583,15632,42535,79133,25057,23218,80883,12034,35368,675,703,21946,22868,26901,2623,8001,16352,1938,15298,41641,27840,79445,405,40427,12436,12535,19562,91170,5484,48403,29329,70605,18054,10135,2935,43418,6297,6816,9274,59608,73139,1140,20814,56127,38016,79239,64635,5031,43156,15105,75335,3412,30021,40799,5623,2199,12307,72374,11540,52971,246280,8502,969,5084,697,8115,1619,5567,9241,6605,6614,21860,54718,25880,319,70563,48700,65932,9066,70807,57267,4793,95350,35594,4919,1896,90574,29552,6108,3320,22195,608,50611,1068,71588,171973,25435,25111,36034,32225,60372,3983,14965,67630,8027,5010,64510,11219,14984,59879,21474,605,7888,60955,35195,42441,76588,87413,1015,47029,12847,64914,25532,38460,32147,5639,34691,38005,8079,20480,47808,5206,32364,72325,15497,44268,21535,8053,8848,45750,10507,8259,91557,3721,18356,47651,23246,7464,2172,18944,88838,7955,49124,39571,91907,15507,23975,16954,20762,21640,72567,205774,33367,37333,18011,3855,10220,1259,4874,49703,90069,27384,57878,21278,18162,8914,29017,84494,64904,16672,88461,66477,8972,56291,6078,10028,56623,6615,2275,922,28682,5780,671,11246,1203,7322,7141,17657,73299,73596,48421,63982,12355,2757,18618,82382,1134,23835,10223,84438,63268,12146,11158,74803,37296,81487,44639,30211,4225,10469,61995,83374,46150,87990,56003,92007,75509,27392,58847,84055,1489,41633,81179,45691,10724,10138,36608,9696,39788,29001,52500,204189,7521,18770,8059,77420,17266,11652,17921,41227,17158,42092,1539,29731,9116,1774,14738,44969,7188,92649,2261,80681,50952,24813,71866,11095,20034,11653,12258,63841,5528,5353,2948,35815,46242,21539,73885,26681,27869,10205,73638,86261,13987,7199,24624,13564,7552,17573,25798,8464,2857,12563,48927,2815,34136,9742,35661,13945,8465,11965,91967,16036,94877,9853,30852,31764,77358,28962,2796,4936,41327,17114,23472,57236,15616,29866,3715,14782,22885,56166,34135,4241,5348,201787,57290,94453,34257,21747,70505,3838,46041,59933,813,21798,17872,47965,18824,71534,51201,11269,55352,20078,48304,25950,89899,11957,8232,2742,29396,8581,730,64575,71196,10537,60116,37043,15774,36158,2868,23961,60794,36893,37172,1666,23512,29954,1744,5717,20483,6354,70818,3301,53164,32641,74970,9306,13535,89413,85116,47461,56129,5999,30973,72553,1039,221111,7448,24123,19561,27817,15214,27051,53847,73360,60048,31464,57131,57014,68310,36480,9331,69163,4528,50333,28230,5565,9307,12320,3828,10256,35678,62807,3478,6144,12572,27047,1513,20183,38617,17213,24315,7222,50407,70435,23122,15209,47223,59469,91703,26590,50693,50147,6015,21397,30184,15572,89561,233522,8346,55718,815,52287,6624,41080,3319,39852,9553,11914,19178,23197,42684,65794,29593,8015,1227,55512,13780,51902,20359,71267,36953,51612,45559,4239,45417,70448,27419,30564,23027,9699,5203,14807,21370,662,69984,53390,22788,89644,65792,19463,56871,60684,857,12509,2712,89077,10115,8195,42728,33955,5059,17697,16401,10133,49463,42664,82833,73452,28475,57844,6645,1971,58510,2110,45543,37427,590,5434,28300,31667,4458,67537,46171,6546,29379,67378,3442,90410,18612,1079,15899,53580,29318,43095,36649,18267,2832,2017,26254,7845,2084,2472,7143,3193,4313,75515,15222,7910,3195,40905,81210,37753,31184,42705,14274,92734,7194,25224,12195,54488,52549,2286,14450,19603,44538,46345,17185,25774,9218,6232,70799,7382,46755,24752,4890,49131,81091,18525,48952,654,20994,5854,15795,2374,16160,66540,61409,4879,84810,210344,9947,20312,73860,4750,218210,91952,428,26193,41150,32690,55008,18730,1600,24532,2180,8441,40973,52870,9679,31032,74224,2205,12654,94860,14870,29650,4151,22467,4268,19831,32770,32483,2287,3739,57120,44564,25344,1473,39647,3426,79781,3047,15038,56705,41457,65752,1509,28969,873,9540,44030,15651,49682,29850,35713,80795,55471,36354,21218,6146,4602,355,5236,27044,12655,7190,35874,13271,56419,3807,4859,91109,16169,12874,25250,5849,36676,23688,1584,28395,29749,74355,12802,21333,3988,2659,5838,2150,10883,5570,6426,8409,64057,85339,227920,17650,19703,66392,29514,66550,64382,12484,9227,21054,82634,45441,33435,57233,4723,1162,41961,73695,14392,81295,24944,4518,32412,28266,222957,47325,51866,379,63860,8495,4542,93478,44770,24874,11799,64483,1847,89356,12315,95085,24956,25460,64268,2631,16940,24011,11550,28921,27025,38050,36670,10963,6117,62557,24785,87238,5517,38540,10459,10088,62830,41643,29960,21853,50531,20229,58477,8294,6281,734,26124,33911,3277,18936,48706,39226,551,5414,10765,25638,25943,16293,11039,38729,32639,2012,75307,67710,66870,19930,5762,71970,3329,25646,65478,21851,24772,94848,45447,67797,29091,61799,3732,10290,17086,76389,60496,5579,8348,93633,83441,63020,34826,16638,8856,65695,12633,890,76578,23146,42883,35459,6673,7698,16591,25434,94932,7723,223569,84397,24814,66522,40397,9113,39525,12888,13809,39618,22424,93133,22460,34832,47433,5940,1082,824,92027,34944,11400,48660,23485,17773,8712,5380,8418,64090,1734,30220,44798,29512,4729,29505,40337,336,42517,2104,37400,8724,852,4989,41813,9319,62683,22446,46491,86549,51730,22902,32579,53499,23644,13262,13065,91965,48308,60528,44984,59200,57925,43075,11585,48992,1431,20713,35209,831,7161,1737,7929,232936,26205,2325,22318,19729,27551,11101,24892,52426,57913,30040,53800,45949,6926,47543,58469,55346,57409,11237,1745,14376,31946,2893,32624,32176,26492,28301,34415,28705,41784,235190,38965,34653,46333,56,10739,31918,18461,82987,6331,65854,40362,8823,6345,34379,5314,543,56617,169143,80900,16282,43782,1088,90966,222436,21578,86451,50151,6553,6014,67016,91758,11344,94497,32096,2120,23818,51569,22739,41447,18933,49273,10234,11636,6385,41423,30007,66373,69321,31516,6609,24580,1798,2822,82135,19039,92180,29263,13614,27351,44932,8435,20496,54700,28382,56791,7615,117,1827,93260,4108,6594,7907,2184,3742,15057,34885,14264,30076,16720,21004,61826,60418,46730,9557,44795,6651,56540,22641,24459,54964,6882,88399,34533,3199,13048,19153,13285,69464,51819,18126,16561,8354,49371,18509,91368,16048,65245,939,34663,7233,19123,242044,75329,2898,1333,78224,45680,83108,47077,85213,43937,35342,17011,65847,82704,57874,3769,36234,9392,9304,82930,46457,83208,62980,28326,12476,12264,68620,8072,39834,18336,51529,57488,40576,6421,43741,7460,58730,64956,9366,5750,44778,4154,3502,386,14406,9141,7343,35960,35254,44800,48468,51042,11192,58236,11293,23312,11354,8506,49058,21041,5413,3237,47410,62420,19969,24503,80490,892,13533,8121,24231,9487,626,11486,74785,18902,54507,36988,39446,16507,49931,1085,12586,10047,27521,68957,3022,48926,94542,7715,37631,65992,15166,13685,29210,92584,63099,59167,57539,24695,38823,85238,47929,16465,224,1808,38921,31036,50098,46399,40775,34816,56819,2517,65289,44717,39582,7656,45299,74554,9669,33283,26931,46303,80445,11267,43004,4447,11201,74768,21692,17778,11761,6276,58176,17571,71257,31838,92876,24578,47746,3141,85514,3650,3535,25862,21299,28902,60322,40138,25349,2801,4259,6122,22197,24151,2768,46455,22947,34860,45403,20220,20169,66125,20112,66561,92386,25524,11684,93064,658,29143,4416,49259,232888,18724,34864,12313,6154,933,1035,34429,35282,62810,19336,21487,14910,54211,39692,34243,46966,66713,16037,24427,53843,22054,10732,4025,6364,32655,75530,30195,175267,6979,26606,35571,22469,38676,12757,30719,19174,33744,11838,66184,22101,91278,16184,15135,12376,4850,22967,27059,90694,14664,15448,13187,76717,38593,3528,13585,47272,85789,30996,6475,14349,86827,43876,33567,12080,2673,21194,9143,1365,4431,61185,9825,88104,51453,53264,1566,7617,78773,26642,23938,13001,10141,64422,61235,95064,6635,5404,18918,23326,4551,16165,59187,92743,2747,51241,69702,10371,88395,19222,41647,65219,22207,13995,16192,18873,24776,32817,83376,12878,77370,52464,24298,42095,18043,85811,90438,57576,10621,37672,24939,68394,4046,6763,77039,65885,14280,86592,86914,1814,3197,7790,33224,41170,63262,1840,11490,14199,16526,30118,65332,14971,22433,50882,59243,55807,73343,20194,8413,72405,3342,41846,32323,23417,3511,1084,5467,25115,84109,39072,56930,19772,10196,18206,188917,4065,8535,70371,15016,24836,12626,23734,89160,7090,55680,17413,8509,37922,3924,66976,21419,93408,14120,13852,47796,84419,38056,28761,8570,5618,1129,32768,63259,1587,57871,15174,17981,58334,26170,63885,50746,12194,88881,36323,28530,67976,23455,4781,15589,41078,21905,60856,432,6902,23611,1371,78802,64697,9194,44996,11362,13613,1526,36382,30875,20226,6042,52151,78349,37536,1552,6184,40066,12154,7793,7135,11483,87782,20489,15537,28466,36879,55566,15417,59199,74641,24385,30057,26963,53579,13553,7458,8214,18729,4050,12263,81123,31047,8670,20828,32728,35479,30979,19581,12819,27408,2124,70588,41685,42417,50123,32001,2878,15246,72364,39943,59654,50009,67806,6183,47096,36189,2891,23618,9521,5955,9154,12688,23956,62453,77175,19536,59513,6608,33146,52859,80058,41135,52904,16836,2206,6342,49839,30269,24838,26731,4310,58917,75150,23080,37208,15513,38951,39566,70664,42065,36514,41933,42320,39241,26932,33961,52662,3361,14855,10350,7092,74060,23531,13868,82067,13742,16198,50382,119,57487,73377,73081,29010,4091,71915,91892,3630,27781,22270,49073,21025,5120,4981,20357,3476,15347,2121,46571,15434,2670,7047,26172,15192,4379,28241,7151,10464,9852,24612,1181,8526,3518,65244,24667,87317,7667,29806,5593,70494,83592,38158,6663,46365,35399,77926,40252,7152,4135,80077,85912,25607,39542,9870,1678,7327,9047,1591,6591,19760,10478,26366,5741,48671,30359,32663,3991,2305,3127,43697,77708,45944,38523,38734,85357,10787,52508,3218,36774,4339,37996,44198,30373,16099,15190,4094,68127,38667,2887,5596,38841,25419,11847,27117,69285,26373,2748,2897,80973,64162,742,74137,5222,48034,759,25835,78795,28588,221535,2186,45625,82168,52513,39275,11090,48146,41932,13364,3967,31950,3534,4201,24528,19461,81344,40584,34563,44734,79476,14583,34493,10358,21709]
ARC2_DRAFT_VOCAB_FILE = Path(PREFIX, "draft_vocab_ranked.json")


def _opt_env(cfg):
    opt = dict(cfg.get("OPT") or {})
    out = {}
    if opt.get("W8"):
        out.update(ARC2_W8_MODULES=ARC2_W8_DENSE, ARC2_W8_GROUP=str(opt.get("W8_GROUP", 128)))
    if opt.get("DRAFT_VOCAB"):
        ARC2_DRAFT_VOCAB_FILE.parent.mkdir(parents=True, exist_ok=True)
        ARC2_DRAFT_VOCAB_FILE.write_text(json.dumps(ARC2_DRAFT_VOCAB_IDS))
        out.update(QWEN4EXP_DRAFT_VOCAB=str(len(ARC2_DRAFT_VOCAB_IDS)), QWEN4EXP_DRAFT_VOCAB_FILE=str(ARC2_DRAFT_VOCAB_FILE))
    if opt.get("PLE_SEQ"):
        out.update(QWEN4EXP_PLE_MMAP_CHUNK="1000000")
    if opt.get("STEPLOG"):
        out.update(ARC2_STEP_PROFILE="1", ARC2_STEP_PROFILE_EVERY="200")
    if opt.get("EMBED_HOST"):
        out.update(ARC2_EMBED_HOST="1")
    if opt.get("W8_COMPACT"):
        if not opt.get("W8"):
            raise RuntimeError("CFG['OPT'] W8_COMPACT needs W8; run stopped")
        out.update(ARC2_W8_COMPACT="1")
    unknown = set(opt) - {"W8", "W8_GROUP", "DRAFT_VOCAB", "PLE_SEQ", "STEPLOG", "EMBED_HOST", "W8_COMPACT"}
    if unknown:
        raise RuntimeError(f"CFG['OPT'] has unknown switches {sorted(unknown)}; run stopped")
    return out


def _with_overrides(args, cfg):
    """Optional keys of cfg (unset in the notebook's CFG; the speed bench sets them): TP, PP, COMPILATION, PROFILER,
    SPEC_EXTRA (keys merged into --speculative-config), EXTRA_ARGS, CUSTOM_AR (drop --disable-custom-all-reduce),
    NO_EP (drop --enable-expert-parallel: one GPU)."""
    args = list(args)
    if cfg.get("TP"):
        args[args.index("--tensor-parallel-size") + 1] = str(cfg["TP"])
    if cfg.get("PP"):
        args += ["--pipeline-parallel-size", str(cfg["PP"])]
    if cfg.get("COMPILATION"):
        args[args.index("--compilation-config") + 1] = json.dumps(cfg["COMPILATION"])
    if cfg.get("PROFILER"):
        args += ["--profiler-config", json.dumps(cfg["PROFILER"])]
    if cfg.get("SPEC_EXTRA"):
        i = args.index("--speculative-config") + 1
        args[i] = json.dumps(dict(json.loads(args[i]), **cfg["SPEC_EXTRA"]))
    if cfg.get("EXTRA_ARGS"):
        args += [str(a) for a in cfg["EXTRA_ARGS"]]
    if cfg.get("CUSTOM_AR"):
        args.remove("--disable-custom-all-reduce")
    if cfg.get("NO_EP"):
        args.remove("--enable-expert-parallel")
    return args


def build_server_args(cfg):
    return _with_overrides([PYTHON, "-m", "vllm.entrypoints.openai.api_server", "--model", str(MODEL_DIR),
            "--served-model-name", SERVED_MODEL_NAME, "--host", "127.0.0.1", "--port", str(SERVED_MODEL_PORT),
            "--trust-remote-code", "--tensor-parallel-size", "4", "--enable-expert-parallel",
            "--disable-custom-all-reduce", "--quantization", "gptq_marlin",
            "--max-model-len", str(cfg["CTX"]), "--max-num-seqs", str(cfg["MAXREQ"]),
            "--gpu-memory-utilization", str(cfg["GMU"]), "--max-num-batched-tokens", str(cfg["CHUNK"]),
            "--enable-prefix-caching", "--enable-prompt-tokens-details", "--kv-cache-dtype", cfg["KVDTYPE"],
            "--prefix-cache-retention-interval", "None",   # dense GDN-state cache (cell 3m, 26-oct: SAME vs W8)
            "--language-model-only",
            "--mamba-ssm-cache-dtype", "bfloat16",
            "--reasoning-parser", "qwen3", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_xml",
            "--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": cfg["MTP"],
                                                "model": str(DRAFT_DIR)}),
            "--compilation-config", json.dumps({"mode": 0, "cudagraph_mode": "FULL_DECODE_ONLY",
                                                "cudagraph_capture_sizes": cfg["CAPTURE_SIZES"]})], cfg)


_PRECACHE_STARTED = [False]


def _log_tail(log_offset, nbytes=20000):
    with open(LOG, "rb") as f:
        f.seek(max(log_offset, Path(LOG).stat().st_size - nbytes))
        return f.read().decode(errors="replace")


def launch_server(cfg, timeout_s, label="main"):
    """Start the server detached; wait until /health answers, the process exits, or timeout_s passes."""
    args = build_server_args(cfg)
    if not _PRECACHE_STARTED[0]:
        precache_model_thread.start()
        _PRECACHE_STARTED[0] = True
    run_env = dict(env, **_opt_env(cfg))
    print(f"\n[{label}] " + shlex.join(map(str, args)) + "\n", flush=True)
    print(f"[{label}] ARC2 OPT {json.dumps(cfg.get('OPT'))} -> env "
          f"{json.dumps({k: v for k, v in run_env.items() if k.startswith(('ARC2_', 'QWEN4EXP_DRAFT', 'QWEN4EXP_PLE_MMAP_CHUNK'))})}",
          flush=True)
    ARC2_CPU_PHASE["phase"] = "server"  # CPU telemetry
    log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
    with open(LOG, "ab", buffering=0) as logf:
        logf.write(f"\n===== SERVER LAUNCH {label} =====\n".encode())
        proc = subprocess.Popen(list(map(str, args)), env=run_env, stdout=logf, stderr=subprocess.STDOUT,
                                start_new_session=True)
    print(f"[{label}] pid {proc.pid} -> {LOG}", flush=True)
    url, t0, last, status = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time(), -30, "timeout"
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            status = "exited"
            break
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    status = "ready"
                    break
        except Exception:
            pass
        if int(time.time() - t0) - last >= 30:
            last = int(time.time() - t0)
            print(f"  [{label}] {last}s ...", flush=True)
        time.sleep(3)
    seconds = int(time.time() - t0)
    print(f"[{label}] {status.upper()} after {seconds}s", flush=True)
    if status != "ready":
        print(_log_tail(log_offset), flush=True)
    return status, proc, log_offset, seconds


def stop_server(proc, wait_s=90):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, 15)
        proc.wait(wait_s)
    except Exception:
        try:
            os.killpg(proc.pid, 9)
        except Exception:
            pass


def server_report(log_offset):
    with open(LOG, "rb") as f:
        f.seek(log_offset)
        tail = f.read().decode(errors="replace")
    keys = ("GPU KV cache size", "Maximum concurrency", "Model loading took", "MARLIN", "Marlin", "WNA16", "ARC2-L4",
            "no KV cache group could be identified", "Mamba cache mode", "block size",
            "KV cache", "cuda graph", "CUDA graph", "speculative", "Available KV", "memory")
    return {"log": [l.strip()[:300] for l in tail.splitlines() if any(k in l for k in keys)][:40]}

# ---- launch (one configuration, no fallback: if it does not come up the run stops) ----
SERVER_PROC = None
_status, SERVER_PROC, _server_log_offset, _secs = launch_server(
    CFG, max(60, SERVER_STARTUP_TIMEOUT - (time.time() - NOTEBOOK_START_TIME)), "main")
if _status != "ready":
    stop_server(SERVER_PROC)
    raise RuntimeError(f"the server did not come up ({_status} after {_secs}s, see the log tail above); run stopped")
print("SERVER REPORT", json.dumps(server_report(_server_log_offset), default=str), flush=True)

# streams = the 68k windows the KV cache holds (at most CFG["MAXREQ"]), from vLLM's KV line of this launch
with open(LOG, "rb") as _f:
    _f.seek(_server_log_offset)
    _boot_log = _f.read().decode(errors="replace")
_kv = re.search(r"GPU KV cache size: ([\d,]+) tokens", _boot_log)
if not _kv:
    stop_server(SERVER_PROC)
    raise RuntimeError("the KV cache size is not in serve.log; run stopped")
KV_TOKENS = int(_kv.group(1).replace(",", ""))
# FULL MODEL + ARC-3 STYLE: streams from vLLM's measured KV, after the memory levers. vLLM logs it on rank 0
# only; the engine gives every rank the smallest rank's block count
_kv_gib = [float(x) for x in re.findall(r"Available KV cache memory: ([\d.]+) GiB", _boot_log)]
if not _kv_gib:
    stop_server(SERVER_PROC)
    raise RuntimeError("'Available KV cache memory' is not in serve.log (GMU mode expected); run stopped")
KV_GIB = min(_kv_gib)
ARC2_STREAMS = max(ARC2_STREAMS_MIN, min(int(CFG["MAXREQ"]),
                                         int(ARC2_STREAMS_KV_FRACTION * KV_GIB * 2**30 // ARC2_STREAM_KV_BYTES)))
_levers = {"embed_host": re.findall(r"ARC2 EMBED HOST: [^\n]*", _boot_log)[:2],
           "w8_compact": re.findall(r"ARC2 W8 COMPACT: [^\n]*", _boot_log)[:2]}
print(f"ARC2 STREAMS {ARC2_STREAMS}: KV {KV_GIB:.2f} GiB per GPU (rank 0), floor({ARC2_STREAMS_KV_FRACTION} x "
      f"KV / {ARC2_STREAM_KV_BYTES / 1e6:.0f} MB) in [{ARC2_STREAMS_MIN}, {CFG['MAXREQ']}]; memory levers: "
      f"{json.dumps(_levers)}", flush=True)
if "ARC3 GDN RecoverSSM enabled" not in _boot_log or "GDN decode kernel: triton" not in _boot_log:   # ARC2-L4 RecoverSSM
    stop_server(SERVER_PROC)
    raise RuntimeError("RecoverSSM or the Triton GDN decode kernel is not on (serve.log); run stopped")
_kv_windows = KV_TOKENS // int(CFG["CTX"])
_kv_bytes_applied = "as specified by kv_cache_memory_bytes config" in _boot_log
print(f"KV BYTES {'applied' if _kv_bytes_applied else 'NOT applied'} (--kv-cache-memory-bytes in CFG EXTRA_ARGS: "
      f"{'--kv-cache-memory-bytes' in [str(a) for a in CFG.get('EXTRA_ARGS', [])]})", flush=True)
print(f"KV CACHE {KV_TOKENS} tokens = {_kv_windows} full {CFG['CTX']}-token window(s); solver streams fixed at "
      f"{ARC2_STREAMS}", flush=True)
if ARC2_STREAMS > int(CFG["MAXREQ"]):
    stop_server(SERVER_PROC)
    raise RuntimeError(f"ARC2 STREAMS {ARC2_STREAMS} > CFG['MAXREQ'] {CFG['MAXREQ']}; run stopped")
if 1 <= _kv_windows < ARC2_STREAMS:
    print(f"WARNING: the KV cache holds {_kv_windows} full windows for {ARC2_STREAMS} streams: when every history is "
          f"long at once vLLM preempts and recomputes (slower, same answers). Expected for the FULL MODEL + ARC-3 "
          f"STYLE: the streams are sized to ~72 % of the KV at ~{ARC2_STREAM_KV_BYTES / 1e6:.0f} MB each, so some "
          f"windows are always short.",
          flush=True)
if _kv_windows < 1:
    stop_server(SERVER_PROC)
    raise RuntimeError(f"the KV cache holds {KV_TOKENS} tokens, less than one {CFG['CTX']}-token window; run stopped")

# ---- the PLE table in RAM (bench v9 on Kaggle 4x L4, 24-oct-2026: results_14 and results_15) ----
# Every decode step waits while the host reads that step's PLE rows from the 95 GB BF16 table on Kaggle's network
# storage; a row that is not in the page cache is a network read (1.4-1.7 ms). The read-ahead of cell 3 reads the whole
# 161 GB model, more than the page cache keeps, so ~7% of the table is left out. Here, once the server is up: the pages
# of every model file except the table are dropped (the weights are on the GPUs already) and the whole table is read
# once. Measured: 72.0 -> 81.0 and 70.1 -> 83.8 tokens/s per stream (+12.5% / +19.5%), PLE wait per step 5.7-7.7 ->
# 1.0 ms. The same bytes, only read from RAM instead of the network: the model's answers do not change. If this step
# fails, the run goes on with the table where it is (slower, same answers).
import struct as _ple_struct
from concurrent.futures import ThreadPoolExecutor as _PLE_POOL

PLE_RAM = dict(READ_THREADS=16, CHUNK_BYTES=16 << 20, READ_CAP_S=900, READAHEAD_WAIT_S=900, SAMPLE_ROWS=3000,
               ROW_BYTES=320, FAST_READ_US=100)          # the values of bench v9
_PLE_PAT = re.compile(r"\.ple\..*ngram_embedding\.shard_\d+\.weight$")


def _ple_spans(model_dir):
    """[(real path, byte offset, length)] of the PLE n-gram shard tensors under model_dir (adjacent ones merged)."""
    root = Path(model_dir)
    index = root / "model.safetensors.index.json"
    if index.is_file():
        files = sorted({fn for name, fn in json.loads(index.read_text())["weight_map"].items() if _PLE_PAT.search(name)})
    else:
        files = sorted(p.name for p in root.glob("*.safetensors"))
    out = []
    for fn in files:
        path = os.path.realpath(str(root / fn))
        with open(path, "rb") as f:
            n = _ple_struct.unpack("<Q", f.read(8))[0]
            header, base = json.loads(f.read(n)), 8 + n
        for b, e in sorted((base + m["data_offsets"][0], base + m["data_offsets"][1])
                           for k, m in header.items() if k != "__metadata__" and _PLE_PAT.search(k)):
            if out and out[-1][0] == path and out[-1][1] + out[-1][2] == b:
                out[-1] = (path, out[-1][1], out[-1][2] + e - b)
            else:
                out.append((path, b, e - b))
    return out


def _ple_model_files(tops):
    seen, out = set(), []
    for top in tops:
        for root, _, names in os.walk(top, followlinks=True):
            for fn in names:
                p = os.path.realpath(os.path.join(root, fn))
                if p not in seen and os.path.isfile(p):
                    seen.add(p)
                    out.append(p)
    return sorted(out)


def _ple_outside(files, spans):
    """(path, offset, length) of every byte of files outside spans (length 0 = to the end of the file)."""
    by_file = {}
    for p, o, n in spans:
        by_file.setdefault(p, []).append((o, n))
    out = []
    for p in files:
        pos = 0
        for o, n in sorted(by_file.get(p, [])):
            if o > pos:
                out.append((p, pos, o - pos))
            pos = max(pos, o + n)
        out.append((p, pos, 0))
    return out


def _ple_evict(spans):
    fds, done, errors = {}, 0, []
    try:
        for p, o, n in spans:
            try:
                if p not in fds:
                    fds[p] = os.open(p, os.O_RDONLY)
                os.posix_fadvise(fds[p], o, n, os.POSIX_FADV_DONTNEED)
                done += 1
            except OSError as exc:
                errors.append(f"{p}: {exc}"[:200])
    finally:
        for fd in fds.values():
            os.close(fd)
    return {"spans": done, "errors": errors[:5]}


def _ple_read(spans):
    chunk = int(PLE_RAM["CHUNK_BYTES"])
    jobs = [(p, o, min(chunk, off + n - o)) for p, off, n in spans for o in range(off, off + n, chunk)]
    local, fds, lock = threading.local(), {}, threading.Lock()
    t0 = time.time()

    def fd_of(p):
        with lock:
            if p not in fds:
                fds[p] = os.open(p, os.O_RDONLY)
            return fds[p]

    def job(j):
        if time.time() - t0 > PLE_RAM["READ_CAP_S"]:   # slow network disk: stop reading, the rest stays on disk
            return 0
        p, off, n = j
        if getattr(local, "buf", None) is None:
            local.buf = memoryview(bytearray(chunk))
        fd, got = fd_of(p), 0
        while got < n:
            r = os.preadv(fd, [local.buf[got:n]], off + got)
            if r <= 0:
                break
            got += r
        return got

    try:
        with _PLE_POOL(max_workers=int(PLE_RAM["READ_THREADS"])) as pool:
            got = sum(pool.map(job, jobs))
    finally:
        for fd in fds.values():
            os.close(fd)
    total = sum(j[2] for j in jobs)
    return {"gb": round(total / 1e9, 1), "complete": got == total, "seconds": round(time.time() - t0, 1)}


def _ple_sample(spans, seed):
    """Timed 320-byte reads of random table rows: the share under FAST_READ_US is the share of the table in RAM."""
    import random as _r
    rng, row = _r.Random(seed), int(PLE_RAM["ROW_BYTES"])
    sizes = [s[2] // row for s in spans]
    total = sum(sizes)
    fds, us = {}, []
    try:
        for _ in range(int(PLE_RAM["SAMPLE_ROWS"]) if total else 0):
            k = rng.randrange(total)
            for (p, o, _n), rows in zip(spans, sizes):
                if k < rows:
                    break
                k -= rows
            if p not in fds:
                fds[p] = os.open(p, os.O_RDONLY)
            t = time.perf_counter()
            os.pread(fds[p], row, o + k * row)
            us.append(1e6 * (time.perf_counter() - t))
    finally:
        for fd in fds.values():
            os.close(fd)
    return round(100 * sum(u < PLE_RAM["FAST_READ_US"] for u in us) / max(1, len(us)), 1)


_t_ple = time.time()
PLE_RAM_REPORT = {}
try:
    if precache_model_thread.is_alive():          # cell 3's read-ahead: let it finish first, or it evicts the table
        precache_model_thread.join(PLE_RAM["READAHEAD_WAIT_S"])
        PLE_RAM_REPORT["readahead_still_running"] = precache_model_thread.is_alive()
    _ple_table = _ple_spans(MODEL_DIR) + _ple_spans(DRAFT_DIR)
    if not _ple_table:
        raise RuntimeError("no PLE n-gram shard tensors found under the model folders")
    PLE_RAM_REPORT["table_gib"] = round(sum(s[2] for s in _ple_table) / 2**30, 2)
    PLE_RAM_REPORT["in_ram_before_pct"] = _ple_sample(_ple_table, 1)
    PLE_RAM_REPORT["dropped_rest"] = _ple_evict(_ple_outside(_ple_model_files([MODEL_DIR, DRAFT_MODEL_DIR]), _ple_table))
    PLE_RAM_REPORT["read"] = _ple_read(_ple_table)
    PLE_RAM_REPORT["in_ram_after_pct"] = _ple_sample(_ple_table, 2)
except Exception as _ple_exc:
    PLE_RAM_REPORT["error"] = f"{type(_ple_exc).__name__}: {_ple_exc}"[:300]
PLE_RAM_REPORT["seconds"] = round(time.time() - _t_ple, 1)
print("PLE TABLE IN RAM", json.dumps(PLE_RAM_REPORT), flush=True)

# %% ---------------------------------------------------------------- [cell 4: ARC-AGI-2 solver]
# Written for ARC-AGI-2 (no ARC-3 harness code). It works with the server the way the ARC-3 harness does:
#   - requests: thinking on, temperature 0.7, top_p 0.95, top_k 20, 12k output, retries on HTTP failures; tool
#     calls that Qwen writes as text are recovered;
#   - retained reasoning: every assistant turn goes back with its reasoning_content (preserve_thinking);
#   - the history only grows, so the prefix cache (radix + mamba extra_buffer) reuses it; when it passes the
#     window, one block of the oldest turns is drained at once (ARC-3: 58k of 140k; here 16k of 32k) and the
#     persistent memory (last rule stated, check results, answers recorded, retained functions) is written into
#     the system prompt, which only changes at a drain;
#   - python tool: 30 s per call, output cut in the middle at 3k tokens; a REPL per task: top-level functions,
#     classes and imports kept as code, variables pickled between calls (a call past 30 s changes nothing);
#   - one task per stream; 3-8 streams (ARC2_STREAMS, set in cell 3 from the KV vLLM measures).
# Per task: the model sees the example pairs and the test inputs as digit rows, writes transform(grid), runs
# check(transform) against the example pairs and answers with submit(transform). The task ends when a checked or
# submitted transform reproduces every example pair (its test outputs are the answer). A stream that frees up takes the next unsolved task; while time is
# left, unsolved tasks come back with their history and memory.
import json as _json
import os as _os
import re as _re
import signal as _signal
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
import threading as _threading
import time as _time
import traceback as _traceback
import uuid as _uuid
import shutil as _shutil
import ast as _ast
from pathlib import Path as _Path

import requests as _requests

ARC2 = dict(
    PROMPT_BUDGET=32 * 1024,      # history window per request; CFG["CTX"] = this + MAX_OUTPUT + 8k margin
                                  # (ARC-3 STYLE: 32k, was 48k)
    DRAIN=16 * 1024,              # drained at once when the window is passed: half the window, as ARC-3's
                                  # 16 x 64k (32k of 64k); was 20k of 48k
    MAX_TURNS=150,                # assistant turns kept in the history, as in ARC-3 ...
    TURN_DRAIN=30,                # ... and drained at once past that
    MAX_OUTPUT=12 * 1024,
    TEMPERATURE=0.7, TOP_P=0.95, TOP_K=20,
    TOOL_TIMEOUT=30,              # seconds per python call
    TOOL_OUTPUT_TOKENS=3072,      # tool output cut in the middle past this
    TOOL_MEMORY_GB=3,             # address-space limit of one python call: the host RAM also holds the PLE table
    TOOL_FILE_BYTES=8 * 2**20,    # what one call may print (and write to any file); past it a write fails in the call
    TOOL_READ_BYTES=256 * 1024,   # what the host reads of that output (head and tail) before the 3k-token cut
    TOOL_VARS_BYTES=32 * 2**20,   # REPL: pickled variables kept between the calls of one task
    REQUEST_TIMEOUT=1800,
    MAX_IDLE_REPLIES=3,           # replies in a row without a tool call before the task stops
    WORKERS=int(ARC2_STREAMS),    # 3-8 streams, set in cell 3 from the KV vLLM measures ('ARC2 STREAMS')
    MIN_SLICE_SECONDS=120,        # a task keeps its stream at least this long per slice; a stopped task is taken
                                  # again only with at least this much of its equal share left
    MIN_START_SECONDS=180,        # no task is taken with less than this before the end
    INTERACTIVE_TASKS=20,         # Save & Run: the first 20 evaluation tasks (sorted ids)
    INTERACTIVE_MINUTES=None,     # Save & Run: solving time; None = the time per task the 12-hour rerun gives
    RERUN_TASKS=240,              # tasks of the hidden test set (the visible test file is a 120-task placeholder)
    TOTAL_SECONDS=12 * 3600,      # Kaggle limit for the notebook
    END_MARGIN_SECONDS=20 * 60,   # solving stops this long before the limit
    # emulation of the 4x L4 time budget on other hardware (Colab A100): None = off (Kaggle). When set, the solver's
    # clock is virtual: it advances by the generated tokens / (EMULATE_TOK_S per stream x EMULATE_STREAMS streams), so
    # every task gets the tokens it would get on the L4s whatever the real speed. Use with WORKERS = EMULATE_STREAMS.
    EMULATE_TOK_S=None,           # effective tokens/s of one L4 stream, prefill and tool time included
    EMULATE_STREAMS=3,
    L4_STARTUP_SECONDS=15 * 60,   # what the L4 notebook spends before solving (install, load, graphs)
)
ARC2_EMU = {"tokens": 0, "t0": _time.time(), "lock": _threading.Lock()}


def _arc2_clock():
    """Wall clock on Kaggle; under emulation, the virtual L4 clock (tokens generated so far)."""
    if not ARC2["EMULATE_TOK_S"]:
        return _time.time()
    return ARC2_EMU["t0"] + ARC2_EMU["tokens"] / (float(ARC2["EMULATE_TOK_S"]) * int(ARC2["EMULATE_STREAMS"]))


def _arc2_level_of(used, left):
    """The equal share A of tasks that have used `used` seconds each, with `left` stream-seconds still to give: every
    task is filled up to A (sum of max(A - u, 0) == left); a task that has used more than A gets nothing more."""
    us = sorted(max(0.0, float(u)) for u in used)
    acc, level = 0.0, 0.0
    for k, u in enumerate(us, 1):
        acc += u
        level = (max(0.0, float(left)) + acc) / k
        if k == len(us) or level <= us[k]:
            break
    return level


def _arc2_yield(task_id, now):
    """Whether a running task gives its stream to a waiting one: never, until the scheduler (end of cell 4) is set."""
    return False

if ARC2["PROMPT_BUDGET"] + ARC2["MAX_OUTPUT"] + 8 * 1024 > int(CFG["CTX"]):
    raise RuntimeError("ARC2 window + output + 8k margin is larger than the server context CFG['CTX']")

# ---- the python sandbox: one process per call, the task and its helpers preloaded ----
_ARC2_SANDBOX = r'''
import ast, builtins, copy, json, numbers, os, pickle, resource, signal, sys, traceback, types

with open(sys.argv[1]) as _f:
    cfg = json.load(_f)
resource.setrlimit(resource.RLIMIT_AS, (int(cfg["memory_bytes"]), int(cfg["memory_bytes"])))
# printing (or writing any file) past the limit fails with an error in the call instead of filling the disk
resource.setrlimit(resource.RLIMIT_FSIZE, (int(cfg["file_bytes"]), int(cfg["file_bytes"])))
signal.signal(signal.SIGXFSZ, signal.SIG_IGN)
_records = open(cfg["records_path"], "a")
_TRAIN = cfg["train"]
_TEST = cfg["test"]


def _record(payload):
    # written at once: it survives a later timeout of this call
    _records.write(json.dumps(payload) + "\n")
    _records.flush()


class _Text(str):
    def __repr__(self):
        return str(self)


class _Region(dict):
    def __repr__(self):
        box = ",".join(str(v) for v in self.get("bbox", ()))
        return f"region color={self.get('color')} size={self.get('size')} bbox=({box})"


class _Late(Exception):
    pass


def _late(signum, frame):
    raise _Late()


def _as_grid(value, what):
    if hasattr(value, "tolist"):
        value = value.tolist()
    if not isinstance(value, (list, tuple)) or not value:
        raise ValueError(f"{what} must be a non-empty list of rows, got {type(value).__name__}")
    rows = []
    for r, row in enumerate(value):
        if hasattr(row, "tolist"):
            row = row.tolist()
        if not isinstance(row, (list, tuple)) or not row:
            raise ValueError(f"{what}: row {r} must be a non-empty list, got {type(row).__name__}")
        cells = []
        for c, v in enumerate(row):
            if isinstance(v, bool) or not isinstance(v, numbers.Integral) or not 0 <= v <= 9:
                raise ValueError(f"{what}: cell ({r},{c}) is {v!r}; cells must be ints 0-9")
            cells.append(int(v))
        rows.append(cells)
    widths = sorted({len(r) for r in rows})
    if len(widths) != 1:
        raise ValueError(f"{what}: rows have different lengths {widths}")
    if not (1 <= len(rows) <= 30 and 1 <= widths[0] <= 30):
        raise ValueError(f"{what}: size {len(rows)}x{widths[0]} is outside 1x1..30x30")
    return rows


def _size(g):
    return f"{len(g)}x{len(g[0])}"


def _diff_text(got, want, limit=8):
    if len(got) != len(want) or len(got[0]) != len(want[0]):
        return f"size {_size(got)}, expected {_size(want)}"
    cells = [(r, c) for r in range(len(want)) for c in range(len(want[0])) if got[r][c] != want[r][c]]
    shown = "; ".join(f"({r},{c}) got {got[r][c]} want {want[r][c]}" for r, c in cells[:limit])
    more = f"; ... {len(cells) - limit} more" if len(cells) > limit else ""
    return f"{len(cells)} of {len(want) * len(want[0])} cells differ: {shown}{more}"


def show(grid):
    g = _as_grid(grid, "show(grid)")
    return _Text("\n".join("".join(str(v) for v in row) for row in g))


def shape(grid):
    g = _as_grid(grid, "shape(grid)")
    return (len(g), len(g[0]))


def diff(a, b):
    ga, gb = _as_grid(a, "diff(a, b): a"), _as_grid(b, "diff(a, b): b")
    if ga == gb:
        return _Text("identical")
    return _Text(_diff_text(ga, gb, limit=20))


def objects(grid, background=0, diagonal=False):
    g = _as_grid(grid, "objects(grid)")
    h, w = len(g), len(g[0])
    seen = [[False] * w for _ in range(h)]
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diagonal:
        steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    out = []
    for r0 in range(h):
        for c0 in range(w):
            if seen[r0][c0] or g[r0][c0] == background:
                continue
            color, stack, cells = g[r0][c0], [(r0, c0)], []
            seen[r0][c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < h and 0 <= cc < w and not seen[rr][cc] and g[rr][cc] == color:
                        seen[rr][cc] = True
                        stack.append((rr, cc))
            cells.sort()
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            out.append(_Region(color=color, size=len(cells), cells=cells, bbox=(min(rs), min(cs), max(rs), max(cs))))
    return out


def _run_tests(transform, seconds):
    old = signal.signal(signal.SIGALRM, _late)
    try:
        signal.setitimer(signal.ITIMER_REAL, seconds)
        outs = []
        for i, g in enumerate(_TEST):
            try:
                outs.append(_as_grid(transform(copy.deepcopy(g)), "output"))
            except _Late:
                raise
            except Exception as exc:
                raise RuntimeError(f"test input {i}: {type(exc).__name__}: {exc}") from None
        return outs
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)


def check(transform, verbose=True):
    passed = 0
    for i, pair in enumerate(_TRAIN):
        try:
            got = transform(copy.deepcopy(pair["input"]))
        except Exception as exc:
            line = f"pair {i}: ERROR {type(exc).__name__}: {exc}"
        else:
            try:
                got = _as_grid(got, "output")
            except ValueError as exc:
                line = f"pair {i}: BAD OUTPUT {exc}"
            else:
                if got == pair["output"]:
                    passed += 1
                    line = f"pair {i}: ok"
                else:
                    line = f"pair {i}: WRONG " + _diff_text(got, pair["output"])
        if verbose:
            print(line)
    _record({"check": passed, "total": len(_TRAIN)})
    if verbose:
        print(f"check: {passed}/{len(_TRAIN)} example pairs reproduced")
        if passed == len(_TRAIN) and _TEST:
            # kept as the answer, as submit() would: a transform that reproduces every pair ends the task. When it
            # fails on a test input the model is told why (r16 16b78196, r17 20270e3b: nothing recorded, nothing said)
            try:
                outs, why = _run_tests(transform, 5.0), ""
            except _Late:
                outs, why = None, "ran past 5 s on the test inputs"
            except Exception as exc:
                outs, why = None, f"fails on {exc}"
            if outs:
                _record({"answer": outs, "passed": passed, "total": len(_TRAIN), "auto": True})
            else:
                print(f"NOT recorded as an answer: transform {why}. Fix it and check it again.")
    return passed


def submit(transform):
    passed = check(transform, verbose=False)
    outs = []
    for i, grid in enumerate(_TEST):
        try:
            got = transform(copy.deepcopy(grid))
        except Exception as exc:
            raise RuntimeError(f"submit: transform failed on test input {i}: {type(exc).__name__}: {exc}") from None
        outs.append(_as_grid(got, f"submit: output for test input {i}"))
    _record({"answer": outs, "passed": passed, "total": len(_TRAIN)})
    print(f"submitted: {passed}/{len(_TRAIN)} example pairs reproduced; "
          f"test output size(s): {', '.join(_size(o) for o in outs)}")


_KEPT = dict(cfg["kept"])


def source(name):
    if not isinstance(name, str):
        name = getattr(name, "__name__", str(name))
    if name not in _KEPT:
        raise KeyError(f"no definition named {name!r} kept from an earlier call; kept: {sorted(_KEPT)}")
    return _Text(_KEPT[name].rstrip("\n"))


# the namespace is a module registered as __main__, so pickle finds the classes and functions defined in it
_user_main = types.ModuleType("__main__")
sys.modules["__main__"] = _user_main
ns = _user_main.__dict__
ns.update({"__builtins__": builtins, "train": copy.deepcopy(_TRAIN), "test": copy.deepcopy(_TEST),
           "show": show, "shape": shape, "diff": diff, "objects": objects, "check": check, "submit": submit,
           "source": source, "_Text": _Text, "_Region": _Region})   # what show/objects return pickles
_PRELOADED = set(ns)


def _load_vars(path):
    # the variables of the earlier calls, pickled one by one: one that no longer loads is dropped alone
    if not path or not os.path.exists(path):
        return
    with open(path, "rb") as f:
        blobs = pickle.load(f)
    for name, data in blobs.items():
        try:
            ns[name] = pickle.loads(data)
        except Exception as exc:
            notes.append(f"Your variable {name} could not be restored ({type(exc).__name__}: {exc}); it is dropped.")


def _save_vars(path):
    # every top-level variable that pickles is kept, as in a REPL; functions and classes are kept as code. Past the
    # per-task budget the smallest values are kept first. What is not kept is named (it was assigned or changed in
    # this call: a value that is not kept is never restored, so it is named once).
    sized, dropped = [], []
    for name, value in list(ns.items()):
        if name.startswith("__") or name in _PRELOADED or name in kept or isinstance(value, types.ModuleType):
            continue
        if isinstance(value, (types.FunctionType, type)):
            if getattr(value, "__module__", None) == "__main__":   # not imported: restored by its import
                dropped.append(f"{name} (a function or class is kept only when defined with def/class at the top "
                               "level of a call)")
            continue
        try:
            data = pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as exc:
            dropped.append(f"{name} ({type(value).__name__} cannot be saved: {type(exc).__name__})")
            continue
        sized.append((len(data), name, data))
    blobs, total = {}, 0
    for size, name, data in sorted(sized):
        if total + size > int(cfg["vars_bytes"]):
            dropped.append(f"{name} ({size / 2**20:.1f} MB; the variables of a task are kept up to "
                           f"{int(cfg['vars_bytes']) // 2**20} MB)")
            continue
        blobs[name] = data
        total += size
    with open(path + ".tmp", "wb") as f:
        pickle.dump(blobs, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(path + ".tmp", path)
    return sorted(blobs), sorted(dropped)
notes, imports, kept = [], [], {}
for src in cfg["imports"]:
    try:
        exec(compile(src, "<retained>", "exec"), ns)
        imports.append(src)
    except Exception as exc:
        notes.append(f"Your retained import `{src}` failed ({type(exc).__name__}: {exc}); it is dropped.")
_pending = list(cfg["kept"])
while _pending:
    # in passes, so a definition that names another one is restored once that one exists
    _failed = []
    for name, src in _pending:
        try:
            exec(compile(src, "<retained>", "exec"), ns)
            kept[name] = src
        except Exception as exc:
            _failed.append((name, src, exc))
    if len(_failed) == len(_pending):
        for name, src, exc in _failed:
            notes.append(f"Your retained {name} could not be restored ({type(exc).__name__}: {exc}); it is dropped.")
        break
    _pending = [(name, src) for name, src, exc in _failed]
_load_vars(cfg.get("vars_in"))
result = {"notes": notes, "imports": imports, "kept": [], "new": [], "error": None, "vars": None, "vars_dropped": []}


def _save():
    result["kept"] = list(kept.items())
    with open(cfg["result_path"], "w") as f:
        json.dump(result, f)


code = cfg["code"]
try:
    tree = ast.parse(code, "<python>")
except SyntaxError as exc:
    result["error"] = "".join(traceback.format_exception_only(type(exc), exc))
    _save()
    sys.exit(0)
lines = code.splitlines(keepends=True)
defs, new_imports = {}, []
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        start = min([d.lineno for d in node.decorator_list] + [node.lineno])
        defs[node.name] = "".join(lines[start - 1:node.end_lineno])
    elif isinstance(node, (ast.Import, ast.ImportFrom)):
        new_imports.append(ast.get_source_segment(code, node))
_before = {name: ns.get(name) for name in defs}
try:
    exec(compile(tree, "<python>", "exec"), ns)
except SystemExit:
    pass
except BaseException as exc:
    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename in ("<python>", "<retained>")]
    result["error"] = "".join(traceback.format_list(frames) + traceback.format_exception_only(type(exc), exc))
for name, src in defs.items():
    if name in ns and ns[name] is not _before[name]:   # the def ran (not an older one left in place)
        kept[name] = src                                # same place as before when it was kept already
        result["new"].append(name)
for src in new_imports:
    if src and src not in imports:
        try:
            exec(compile(src, "<retained>", "exec"), {"__builtins__": builtins})   # only imports that work are kept
        except BaseException:
            continue
        imports.append(src)
if cfg.get("vars_out"):
    result["vars"], result["vars_dropped"] = _save_vars(cfg["vars_out"])
try:
    sys.stdout.flush()
except OSError:
    pass
_save()
'''
compile(_ARC2_SANDBOX, "<arc2 sandbox>", "exec")
ARC2_VARS_DIR = _Path(_tempfile.gettempdir()) / "arc2_vars"   # the REPL variables of each task, between calls
_shutil.rmtree(ARC2_VARS_DIR, ignore_errors=True)
ARC2_VARS_DIR.mkdir(parents=True, exist_ok=True)


def _arc2_run_python(task, kept, imports, code, vars_path=None):
    """One python call in a fresh process; returns what it printed, its error, the records and what is kept. With
    vars_path (REPL), the call starts from the variables in that file and, when it finishes, the file holds the
    variables after it; a call that times out or dies leaves the file as it was."""
    with _tempfile.TemporaryDirectory(prefix="arc2py-") as tmp:
        records_path, result_path = _Path(tmp, "records.jsonl"), _Path(tmp, "result.json")
        cfg_path, out_path = _Path(tmp, "call.json"), _Path(tmp, "stdout.txt")
        vars_out = _Path(tmp, "vars.pkl")
        cfg_path.write_text(_json.dumps({
            "train": task["train"], "test": [c["input"] for c in task["test"]], "kept": list(kept.items()),
            "imports": list(imports), "code": code, "records_path": str(records_path),
            "result_path": str(result_path), "memory_bytes": int(ARC2["TOOL_MEMORY_GB"] * 2**30),
            "file_bytes": int(ARC2["TOOL_FILE_BYTES"]),
            "vars_in": str(vars_path) if vars_path and _os.path.exists(vars_path) else None,
            "vars_out": str(vars_out) if vars_path else None, "vars_bytes": int(ARC2["TOOL_VARS_BYTES"])}))
        env = dict(_os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
        timed_out = False
        with open(out_path, "wb") as fout:
            # its own process group, so whatever the call starts is stopped with it
            proc = _subprocess.Popen([_sys.executable, "-u", "-c", _ARC2_SANDBOX, str(cfg_path)],
                                     stdin=_subprocess.DEVNULL, stdout=fout, stderr=_subprocess.STDOUT, cwd=tmp,
                                     env=env, start_new_session=True)
            try:
                proc.wait(timeout=ARC2["TOOL_TIMEOUT"])
            except _subprocess.TimeoutExpired:
                timed_out = True
            finally:
                try:
                    _os.killpg(proc.pid, _signal.SIGKILL)
                except (ProcessLookupError, PermissionError):
                    pass
                proc.wait()
        size = out_path.stat().st_size
        with open(out_path, "rb") as f:
            if size <= ARC2["TOOL_READ_BYTES"]:
                raw = f.read()
            else:
                half = ARC2["TOOL_READ_BYTES"] // 2
                head = f.read(half)
                f.seek(size - half)
                raw = head + f"\n... [{size - 2 * half} bytes not read] ...\n".encode() + f.read()
        out = raw.decode("utf-8", "replace")
        records = []
        if records_path.is_file():
            for line in records_path.read_text().splitlines():
                try:
                    records.append(_json.loads(line))
                except ValueError:
                    pass   # a line cut by the timeout
        result = None
        if result_path.is_file():
            try:
                result = _json.loads(result_path.read_text())
            except ValueError:
                result = None
        if result is not None and not timed_out and vars_path and result.get("vars") is not None \
                and vars_out.is_file():
            _shutil.copyfile(vars_out, vars_path)
    return {"stdout": out, "timed_out": timed_out, "capped": size >= ARC2["TOOL_FILE_BYTES"], "records": records,
            "result": result}


ARC2_SYSTEM_PROMPT = """You are solving one ARC-AGI-2 task. A grid is a list of rows of colors 0-9 (0 is usually the \
background). The example pairs show input -> output under one hidden rule. Find the rule and apply it to every test \
input.

You have one tool, `python`. Each call runs in a new Python process (30 s limit) that already has:
- train: the example pairs, a list of {"input": grid, "output": grid}; test: the list of test input grids. Grids are \
lists of lists of int.
- show(grid): returns the grid as a string of digit rows. shape(grid): (rows, cols). diff(a, b): returns a text \
saying how many cells differ and listing the first 20.
- objects(grid, background=0, diagonal=False): the connected single-color regions, each a dict with color, size, \
cells [(r, c), ...] and bbox (r0, c0, r1, c1), inclusive.
- check(transform): runs your transform(grid) on every example input and prints, per pair, ok or what differs; \
returns how many pairs match.
- submit(transform): runs transform on every test input and records the outputs as an answer.
- source(name): the code of a function or class kept from an earlier call.
Installed modules can be imported (numpy included). Functions, classes and imports you write at the top level of a \
call are kept for your later calls in this task.

How to work: study the pairs. Write transform(grid), run check(transform) and fix it until every pair \
matches, then call submit(transform). The task ends as soon as a checked or submitted transform reproduces every \
example pair; its outputs on the test inputs are recorded as the answer. \
Every test input gets 2 answers: the two best distinct submissions, ranked by how many example pairs they \
reproduce, then by recency."""

ARC2_NUDGE = "Continue with the python tool. When check(transform) reproduces every example pair, call submit(transform)."
ARC2_BAD_ARGS_NUDGE = ('Your python call could not be read: its arguments must be a JSON object like '
                       '{"code": "print(1)"}. Call the python tool again.')
ARC2_CUT_NUDGE = ("Your reply reached the output limit before a tool call. Your reasoning so far is kept above: continue "
                  "from where it stopped, keep it short, and call the python tool.")

ARC2_TOOLS = [{"type": "function", "function": {
    "name": "python",
    "description": "Run Python code in the task sandbox and return what it prints.",
    "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": "Python source"}},
                   "required": ["code"]}}}]


def _grid_rows(grid):
    return "\n".join("".join(str(v) for v in row) for row in grid)


def arc2_task_text(task_id, task):
    parts = [f"ARC-AGI-2 task {task_id}: {len(task['train'])} example pairs, {len(task['test'])} test input(s)."]
    for i, pair in enumerate(task["train"]):
        gi, go = pair["input"], pair["output"]
        parts.append(f"Example pair {i}\nInput ({len(gi)}x{len(gi[0])}):\n{_grid_rows(gi)}\n"
                     f"Output ({len(go)}x{len(go[0])}):\n{_grid_rows(go)}")
    for i, case in enumerate(task["test"]):
        gi = case["input"]
        parts.append(f"Test input {i} ({len(gi)}x{len(gi[0])}):\n{_grid_rows(gi)}")
    return "\n\n".join(parts)


def _est_tokens(text):
    # every digit is its own token (grid rows ~1 token per character); list punctuation is one token per mark and
    # ", " two (a printed list ~3 tokens per cell); other text ~3.5 characters per token. On the high side on purpose.
    digits = sum(map(str.isdigit, text))
    punct = sum(text.count(c) for c in ",[](){}")
    return digits + punct + text.count(", ") + int((len(text) - digits - punct) / 3.5) + 4


def _msg_tokens(m):
    n = _est_tokens(str(m.get("content") or "")) + _est_tokens(str(m.get("reasoning_content") or ""))
    for tc in m.get("tool_calls") or []:
        n += _est_tokens(str((tc.get("function") or {}).get("arguments") or ""))
    return n + 8


def _cut_middle(text, limit_tokens):
    if len(text) > 8 * limit_tokens:
        # far too long: cut by characters first, so the estimate below only counts what can stay
        keep = 4 * limit_tokens
        text = text[:keep] + f"\n... [{len(text) - 2 * keep} characters cut from the middle] ...\n" + text[-keep:]
    est = _est_tokens(text)
    if est <= limit_tokens:
        return text
    keep = int(len(text) * limit_tokens / est)
    head = int(keep * 0.6)
    tail = keep - head
    return text[:head] + f"\n... [{len(text) - keep} characters cut from the middle] ...\n" + text[len(text) - tail:]


def _signature(src):
    try:
        node = _ast.parse(src).body[0]
    except Exception:
        return src.split("\n", 1)[0][:80]
    if isinstance(node, _ast.ClassDef):
        return f"class {node.name}"
    if not isinstance(node, (_ast.FunctionDef, _ast.AsyncFunctionDef)):
        return src.split("\n", 1)[0][:80]     # a retained constant: its line
    return f"{node.name}({_ast.unparse(node.args)})"


# ---- tool calls ----
def _arc2_call(name, args):
    return {"id": f"call_{_uuid.uuid4().hex[:12]}", "type": "function",
            "function": {"name": str(name), "arguments": _json.dumps(args)}}


def _arc2_clean_calls(raw):
    """Calls whose arguments are a JSON object (the server rejects the others when they come back in history)."""
    out, dropped = [], 0
    for call in raw or []:
        fn = (call or {}).get("function") or {}
        args = fn.get("arguments")
        if isinstance(args, str):
            try:
                args = _json.loads(args)
            except ValueError:
                args = None
        if not isinstance(args, dict):
            dropped += 1
            continue
        out.append({"id": str(call.get("id") or f"call_{_uuid.uuid4().hex[:12]}"), "type": "function",
                    "function": {"name": str(fn.get("name") or ""), "arguments": _json.dumps(args)}})
    return out, dropped


_ARC2_TC_RE = _re.compile(r"<tool_call>(.*?)</tool_call>", _re.S)
_ARC2_FN_RE = _re.compile(r"<function=([\w.-]+)>(.*?)</function>", _re.S)
_ARC2_PARAM_RE = _re.compile(r"<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>", _re.S)


def _arc2_recover_calls(text):
    """Complete tool calls that Qwen wrote as text (qwen3_coder XML or JSON) -> (calls, text without them)."""
    calls = []
    for body in _ARC2_TC_RE.findall(text):
        m = _ARC2_FN_RE.search(body)
        if m:
            calls.append(_arc2_call(m.group(1), dict(_ARC2_PARAM_RE.findall(m.group(2)))))
            continue
        try:
            obj = _json.loads(body.strip())
        except ValueError:
            continue
        if isinstance(obj, dict) and obj.get("name"):
            args = obj.get("arguments", obj.get("parameters", {}))
            if isinstance(args, str):
                try:
                    args = _json.loads(args)
                except ValueError:
                    args = None
            if isinstance(args, dict):
                calls.append(_arc2_call(obj["name"], args))
    if not calls:
        # qwen3_coder markup without the <tool_call> wrapper
        for m in _ARC2_FN_RE.finditer(text):
            calls.append(_arc2_call(m.group(1), dict(_ARC2_PARAM_RE.findall(m.group(2)))))
        return calls, (_ARC2_FN_RE.sub("", text).strip() if calls else text)
    return calls, _ARC2_TC_RE.sub("", text).strip()


def _arc2_direct_call(name, args):
    """A call to check or submit made as a tool of its own (r22 271d71e2 lost a correct transform to "Unknown tool"):
    the python it stands for -- its `code`, or check(<arg>) / submit(<arg>) for its one argument -- or None."""
    if name not in ("check", "submit"):
        return None
    if isinstance(args.get("code"), str) and args["code"].strip():
        return args["code"]
    values = [v for v in args.values() if isinstance(v, str) and v.strip()]
    return f"{name}({values[0].strip()})" if len(values) == 1 else None


# ---- persistent memory: the rule the model last stated ----
_ARC2_RULE_RE = _re.compile(
    r"(?im)^[ \t>#*_-]*(?:current |updated |new |final |revised )?rule(?:[ \t]*\([^)\n]{0,30}\))?[ \t*_]*:[ \t*_]*(?:\n[ \t]*){0,2}[ \t*_]*"
    r"((?!```)\S.*(?:\n(?![ \t]*\n)(?![ \t]*```)(?![ \t]*#)(?![ \t>#*_-]*(?:current |updated |new |final |revised )?rule\b).*)*)")


def _arc2_last_rule(text, quoted_ok):
    rule = ""
    for m in _ARC2_RULE_RE.finditer(text or ""):
        cand = m.group(1).strip().strip("*_ ").strip()
        if not cand or (not quoted_ok and cand[0] in "\"'`\u201c\u2018"):
            continue
        rule = cand[:1000]
    return rule


class _PromptTooLong(Exception):
    pass


def _arc2_post(payload, timeout):
    url = f"http://127.0.0.1:{SERVED_MODEL_PORT}/v1/chat/completions"
    delay, last = 5.0, None
    for attempt in range(4):
        try:
            resp = _requests.post(url, json=payload, timeout=(10, timeout))
        except (_requests.ConnectionError, _requests.Timeout) as exc:
            last = exc
            _proc = globals().get("SERVER_PROC")
            if _proc is not None and _proc.poll() is not None:
                break   # the server process is gone: no point retrying
        else:
            if resp.status_code < 500:
                return resp
            last = RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
        if attempt < 3:
            if _arc2_clock() + (0 if ARC2["EMULATE_TOK_S"] else delay) >= ARC2_DEADLINE:
                break   # no retry past the end of solving
            _time.sleep(delay)
            delay = min(30.0, delay * 2)
    raise last


class Arc2Task:
    """One task: its conversation, retained functions, persistent memory and recorded answers (kept between slices)."""

    def __init__(self, task_id, task):
        self.task_id, self.task = task_id, task
        self.messages = [{"role": "system", "content": ARC2_SYSTEM_PROMPT},
                         {"role": "user", "content": arc2_task_text(task_id, task)}]
        self.kept, self.imports, self.candidates = {}, [], []
        self.vars_path, self.var_names = str(ARC2_VARS_DIR / f"{task_id}.pkl"), []   # REPL variables
        self.memory = dict(rule_visible="", rule_reasoning="", checks=0, best=0)
        self.stats = dict(requests=0, tool_calls=0, prompt_tokens=0, cached_tokens=0, completion_tokens=0, drains=0,
                          drained_tokens=0, memory_writes=0, cut_replies=0, idle_replies=0, errors=0, too_long=0,
                          timeouts=0, slices=0, seconds=0.0, stop="")
        self._known = None   # (messages sent, prompt tokens the server counted) of the last request

    # -- history size --
    def _estimate(self):
        if self._known is None:
            return sum(_msg_tokens(m) for m in self.messages)
        n, tokens = self._known
        return tokens + sum(_msg_tokens(m) for m in self.messages[n:])

    def _drain(self, tokens=0, turns=0):
        """Drop the oldest turns (an assistant reply with its tool results and any note after them) until `tokens`
        and `turns` are reached; the system prompt, the task and the latest turn stay. Then the memory is written."""
        removed_tokens = removed_turns = 0
        while removed_tokens < tokens or removed_turns < turns:
            starts = [i for i, m in enumerate(self.messages) if i >= 2 and m["role"] == "assistant"]
            if len(starts) <= 1:
                break
            a, b = starts[0], starts[1]
            removed_tokens += sum(_msg_tokens(m) for m in self.messages[a:b])
            removed_turns += 1
            del self.messages[a:b]
        if not removed_turns:
            return False
        self._known = None
        self.stats["drains"] += 1
        self.stats["drained_tokens"] += removed_tokens
        self.messages[0] = {"role": "system", "content": ARC2_SYSTEM_PROMPT + "\n\n" + self.memory_block()}
        self.stats["memory_writes"] += 1
        return True

    def memory_block(self):
        rule = self.memory["rule_visible"] or self.memory["rule_reasoning"] or "not stated"
        n = len(self.task["train"])
        subs = [f"{c['passed']}/{c['total']}" for c in self.candidates if not c.get("auto")]
        auto = sum(1 for c in self.candidates if c.get("auto"))
        kept = [_signature(s) for s in self.kept.values()]
        return "\n".join([
            "PERSISTENT MEMORY. Earlier turns of this task were removed from the conversation to fit the context "
            "window; this block keeps what they established.",
            f"- Last rule you stated: {rule}",
            f"- Checks against the example pairs so far: {self.memory['checks']}; best {self.memory['best']}/{n} "
            "pairs reproduced.",
            "- Answers recorded with submit(): " + (", ".join(subs) if subs else "none")
            + (f"; plus {auto} kept from checks that reproduced every pair" if auto else "") + ".",
            "- Still defined in the sandbox (source(name) prints the code): " + (", ".join(kept) if kept else "nothing")
            + (" (imports: " + "; ".join(self.imports) + ")" if self.imports else "") + ".",
            "- Variables kept from your calls: " + (", ".join(self.var_names[:40]) + (", ..." if len(self.var_names) > 40
                                                                                      else "")
                                                    if self.var_names else "none") + "."])

    # -- one request --
    def _chat(self):
        est = self._estimate()
        max_tokens = min(ARC2["MAX_OUTPUT"], int(CFG["CTX"]) - est - 512)
        if max_tokens < 2048:
            raise _PromptTooLong(f"prompt too long for the window ({est} estimated tokens)")
        payload = {"model": SERVED_MODEL_NAME, "messages": self.messages, "max_tokens": max_tokens,
                   "temperature": ARC2["TEMPERATURE"], "top_p": ARC2["TOP_P"], "top_k": ARC2["TOP_K"],
                   "tools": ARC2_TOOLS, "tool_choice": "auto",
                   "chat_template_kwargs": {"enable_thinking": True, "preserve_thinking": True}}
        timeout = (ARC2["REQUEST_TIMEOUT"] if ARC2["EMULATE_TOK_S"]
                   else max(60.0, min(ARC2["REQUEST_TIMEOUT"], ARC2_DEADLINE - _time.time() + 60)))
        n_sent = len(self.messages)
        resp = _arc2_post(payload, timeout)
        if resp.status_code == 400 and _re.search(r"context|too long|maximum.*length|exceed", resp.text, _re.I):
            raise _PromptTooLong(f"HTTP 400: {resp.text[:300]}")
        if resp.status_code >= 400:
            raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
        data = resp.json()
        choice = (data.get("choices") or [None])[0]
        if not choice:
            raise RuntimeError(f"no choices: {str(data)[:300]}")
        usage = data.get("usage") or {}
        self.stats["requests"] += 1
        self.stats["prompt_tokens"] += int(usage.get("prompt_tokens") or 0)
        self.stats["cached_tokens"] += int((usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0)
        self.stats["completion_tokens"] += int(usage.get("completion_tokens") or 0)
        with ARC2_EMU["lock"]:
            ARC2_EMU["tokens"] += int(usage.get("completion_tokens") or 0)
        return choice.get("message") or {}, str(choice.get("finish_reason") or ""), usage, n_sent

    # -- one python call --
    def _python(self, code):
        self.stats["tool_calls"] += 1
        run = _arc2_run_python(self.task, self.kept, self.imports, code, self.vars_path)
        res = run["result"]
        notes = []
        if res is None:
            # timed out or died: nothing new is retained, the previous functions stay
            if run["timed_out"]:
                self.stats["timeouts"] += 1
                notes.append(f"[stopped after {ARC2['TOOL_TIMEOUT']} s; nothing from this call is kept: functions and "
                             "variables are as they were before it]")
            else:
                notes.append("[the python process ended without finishing; nothing from this call is kept: functions "
                             "and variables are as they were before it]")
        else:
            notes.extend(res.get("notes") or [])
            before = dict(self.kept)
            self.kept = {name: src for name, src in res.get("kept") or []}
            self.imports = list(res.get("imports") or [])
            for name in res.get("new") or []:
                if before.get(name) != self.kept.get(name):
                    notes.append(f"Retained {_signature(self.kept[name])} for your later calls in this task.")
            if res.get("vars") is not None:
                self.var_names = list(res["vars"])
            if res.get("vars_dropped"):
                notes.append("Not kept for your later calls: " + "; ".join(res["vars_dropped"]) + ".")
        if run["capped"]:
            notes.append(f"[output stopped at {ARC2['TOOL_FILE_BYTES'] // 2**20} MB: printing past that fails in the call]")
        solved = False
        answered = False
        for rec in run["records"]:
            if "check" in rec:
                self.memory["checks"] += 1
                self.memory["best"] = max(self.memory["best"], int(rec["check"]))
            elif "answer" in rec:
                self.candidates.append({"grids": rec["answer"], "passed": rec["passed"], "total": rec["total"],
                                        "auto": bool(rec.get("auto")), "order": len(self.candidates)})
                answered = True
                if rec["total"] and rec["passed"] == rec["total"]:
                    # a check() that reproduces every pair ends the task like submit() (run 6: 269e22fb kept
                    # going for 15 minutes after its correct check)
                    solved = True
                else:
                    notes.append(f"Recorded as an answer ({rec['passed']}/{rec['total']} example pairs). "
                                 "The task continues.")
        if answered:
            _arc2_publish(self)   # into submission.json at once, not only when the slice ends
        parts = []
        if run["stdout"].strip():
            parts.append(run["stdout"].rstrip())
        if res and res.get("error"):
            parts.append("[error]\n" + str(res["error"]).rstrip())
        parts.extend(notes)
        text = _cut_middle("\n\n".join(parts) if parts else "(no output)", ARC2["TOOL_OUTPUT_TOKENS"])
        return text, solved

    def run(self, slice_deadline=None):
        started = _time.time()
        self.stats["slices"] += 1
        try:
            self.stats["stop"] = self._loop(slice_deadline)
        finally:
            self.stats["seconds"] = round(self.stats["seconds"] + _time.time() - started, 1)
            if self.stats["stop"] == "solved":
                try:
                    _os.remove(self.vars_path)
                except OSError:
                    pass
        return self.answer()

    def _loop(self, slice_deadline):
        idle = errors = too_long = 0
        while True:
            now = _arc2_clock()
            if now >= ARC2_DEADLINE or (slice_deadline is not None and now >= slice_deadline) \
                    or _arc2_yield(self.task_id, now):
                return "time"
            turns = sum(1 for m in self.messages if m["role"] == "assistant")
            if self._estimate() > ARC2["PROMPT_BUDGET"]:
                self._drain(tokens=ARC2["DRAIN"])
            elif turns > ARC2["MAX_TURNS"]:
                self._drain(turns=ARC2["TURN_DRAIN"])
            try:
                msg, finish, usage, n_sent = self._chat()
            except _PromptTooLong as exc:
                # the window was passed anyway: drain another block and ask again
                too_long += 1
                self.stats["too_long"] += 1
                self.stats["last_error"] = str(exc)[:400]
                if too_long > 3 or not self._drain(tokens=ARC2["DRAIN"]):
                    return "too_long"
                continue
            except Exception as exc:
                errors += 1
                self.stats["errors"] += 1
                self.stats["last_error"] = f"{type(exc).__name__}: {exc}"[:400]
                _proc = globals().get("SERVER_PROC")
                if _proc is not None and _proc.poll() is not None:
                    ARC2_SERVER_DEAD[0] = f"the server process exited with code {_proc.returncode} (see serve.log)"
                    return "server_dead"
                if errors >= 3:
                    return "errors"
                _time.sleep(5)
                continue
            errors = too_long = 0
            content = str(msg.get("content") or "")
            reasoning = str(msg.get("reasoning_content") or msg.get("reasoning") or "")
            calls, dropped = _arc2_clean_calls(msg.get("tool_calls"))
            if not calls and not dropped:
                calls, content = _arc2_recover_calls(content)
                if not calls:
                    calls, _unused = _arc2_recover_calls(reasoning)
            # persistent memory: the latest rule written in a visible reply wins; one in the reasoning counts only
            # while no visible one exists, and a quoted one there is not read
            rule = _arc2_last_rule(content, quoted_ok=True)
            if rule:
                self.memory["rule_visible"] = rule
            elif not self.memory["rule_visible"]:
                self.memory["rule_reasoning"] = _arc2_last_rule(reasoning, quoted_ok=False) or self.memory["rule_reasoning"]
            cut = finish == "length" and not calls
            if cut:
                # the whole cut reasoning stays in the history, so the next reply continues it (run 6: 20 cut replies,
                # ~40% of the generated tokens, were cut down to their last 3,000 characters)
                self.stats["cut_replies"] += 1
            entry = {"role": "assistant", "content": content, "reasoning_content": reasoning}
            if calls:
                entry["tool_calls"] = calls
            self.messages.append(entry)
            self._known = (n_sent, int(usage.get("prompt_tokens") or 0))
            if not calls:
                if cut:
                    self.messages.append({"role": "user", "content": ARC2_CUT_NUDGE})
                    continue
                idle += 1
                self.stats["idle_replies"] += 1
                self.messages.append({"role": "user", "content": ARC2_BAD_ARGS_NUDGE if dropped else ARC2_NUDGE})
                if idle >= ARC2["MAX_IDLE_REPLIES"]:
                    return "idle"
                continue
            idle = 0
            solved = False
            for call in calls:
                fn = call["function"]
                args = _json.loads(fn["arguments"])
                code = str(args.get("code") or "")
                direct = _arc2_direct_call(fn["name"], args)
                if solved:
                    text = "Not run: the task was already solved by an earlier call of this reply."
                elif fn["name"] != "python" and direct is not None:
                    text, now_solved = self._python(direct)
                    text = (f"[{fn['name']} is a function inside the python tool, not a tool; this call ran in python "
                            f"as: {direct[:300]}]\n\n" + text)
                    solved = solved or now_solved
                elif fn["name"] != "python":
                    text = f"Unknown tool {fn['name']!r}; the only tool is python."
                elif not code.strip():
                    text = "The python call carried no code."
                else:
                    text, now_solved = self._python(code)
                    solved = solved or now_solved
                self.messages.append({"role": "tool", "tool_call_id": call["id"], "content": text})
            if solved:
                return "solved"

    def answer(self):
        """Per test input: the best two distinct grids over all recorded answers (pairs reproduced, then recency)."""
        ranked = sorted(self.candidates, key=lambda c: (c["passed"] / max(1, c["total"]), c["order"]), reverse=True)
        out = []
        for i in range(len(self.task["test"])):
            picks = []
            for cand in ranked:
                grid = cand["grids"][i] if i < len(cand["grids"]) else None
                if grid is not None and grid not in picks:
                    picks.append(grid)
                if len(picks) == 2:
                    break
            if not picks:
                picks = [[[0]]]
            if len(picks) == 1:
                picks.append(picks[0])
            out.append({"attempt_1": picks[0], "attempt_2": picks[1]})
        return out


# ---- tasks, time, submission ----
_challenges_file = "arc-agi_test_challenges.json" if TRUE_SUBMISSION else "arc-agi_evaluation_challenges.json"
ARC2_TASKS = _json.loads((_Path(ARC2_DIR) / _challenges_file).read_text())
ARC2_IDS = sorted(ARC2_TASKS)
if not TRUE_SUBMISSION:
    ARC2_IDS = ARC2_IDS[:ARC2["INTERACTIVE_TASKS"]]
ARC2_SUBMISSION_PATH = _Path(WORKING_DIR) / "submission.json"
ARC2_LOG_DIR = _Path(WORKING_DIR) / "arc2_runs"
if not TRUE_SUBMISSION:
    ARC2_LOG_DIR.mkdir(parents=True, exist_ok=True)
# every task of the file has an entry from the start: a run that stops early still leaves a valid submission
ARC2_SUBMISSION = {tid: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in ARC2_TASKS[tid]["test"]]
                   for tid in sorted(ARC2_TASKS)}
_arc2_lock = _threading.Lock()


def _write_submission():
    tmp = ARC2_SUBMISSION_PATH.with_suffix(".tmp")
    tmp.write_text(_json.dumps(ARC2_SUBMISSION))
    tmp.replace(ARC2_SUBMISSION_PATH)


def _arc2_publish(st):
    with _arc2_lock:
        ARC2_SUBMISSION[st.task_id] = st.answer()
        _write_submission()


_write_submission()
_hard_end = NOTEBOOK_START_TIME + ARC2["TOTAL_SECONDS"] - ARC2["END_MARGIN_SECONDS"]
if ARC2["EMULATE_TOK_S"]:
    # the L4 rerun's solving time (12 h - startup - end margin) split over its 240 tasks, times the tasks played here
    ARC2_EMU["t0"] = _time.time()
    _interactive_s = ((ARC2["TOTAL_SECONDS"] - ARC2["END_MARGIN_SECONDS"] - ARC2["L4_STARTUP_SECONDS"])
                      * len(ARC2_IDS) / max(1, int(ARC2["RERUN_TASKS"])))
    ARC2_DEADLINE = ARC2_EMU["t0"] + _interactive_s
    print(f"ARC2 EMULATION of the 4x L4: {ARC2['EMULATE_STREAMS']} streams x {ARC2['EMULATE_TOK_S']} tok/s, "
          f"{_interactive_s / 60:.0f} virtual min for {len(ARC2_IDS)} tasks "
          f"(~{_interactive_s * ARC2['EMULATE_STREAMS'] * ARC2['EMULATE_TOK_S'] / len(ARC2_IDS):,.0f} tokens per task)",
          flush=True)
elif TRUE_SUBMISSION:
    ARC2_DEADLINE = _hard_end
elif ARC2["INTERACTIVE_MINUTES"] is not None:
    ARC2_DEADLINE = min(_hard_end, _time.time() + ARC2["INTERACTIVE_MINUTES"] * 60)
else:
    # the rerun's time left from here to its end, split over the hidden test set's tasks, times the tasks played
    # here: each task gets the share it would get in the rerun (Save & Run had 90 min = 13.5 min per task with 3
    # streams; the rerun with 240 test tasks gives ~8.6)
    _n_test = int(ARC2["RERUN_TASKS"])
    _interactive_s = max(0.0, _hard_end - _time.time()) * len(ARC2_IDS) / max(1, _n_test)
    ARC2_DEADLINE = min(_hard_end, _time.time() + _interactive_s)
    print(f"ARC2 Save & Run time: {_interactive_s / 60:.0f} min for {len(ARC2_IDS)} tasks = the rerun's share per task "
          f"({_hard_end - _time.time():.0f} s left of 12 h, {_n_test} test tasks)", flush=True)
ARC2_STATE = {tid: Arc2Task(tid, ARC2_TASKS[tid]) for tid in ARC2_IDS}
_arc2_running = {}   # task id -> clock time its current slice started
_arc2_used = {tid: 0.0 for tid in ARC2_IDS}   # clock seconds of each task's finished slices
_arc2_order = {tid: i for i, tid in enumerate(ARC2_IDS)}
_arc2_round = [1]    # the most slices any task has had (a task resumed with its history is a new round)
ARC2_SERVER_DEAD = [""]   # set when the server process has exited: no stream takes another task


def _arc2_unfinished(tid):
    return ARC2_STATE[tid].stats["stop"] in ("", "time")


def _arc2_usage(tid, now):
    start = _arc2_running.get(tid)
    return _arc2_used[tid] + (max(0.0, now - start) if start is not None else 0.0)


def _arc2_level(now):
    """Equal share of the solving time (26-oct, user's rule): each time a task ends or asks, the time left on all
    streams is shared again in equal parts among every unfinished task -- running, waiting or stopped earlier --
    counting what each has used. Time a fast task leaves goes to all the others, not only to the ones after it."""
    left = max(0.0, ARC2_DEADLINE - now) * ARC2["WORKERS"]
    return _arc2_level_of([_arc2_usage(t, now) for t in ARC2_IDS if _arc2_unfinished(t)], left)


def _arc2_candidates(now, level):
    """Tasks a free stream may take: never started ones first (in order), then stopped ones with at least
    MIN_SLICE_SECONDS of their equal share left (a resume re-reads the history), the most share left first."""
    out = []
    for t in ARC2_IDS:
        if t in _arc2_running or not _arc2_unfinished(t):
            continue
        fresh = ARC2_STATE[t].stats["slices"] == 0
        room = level - _arc2_usage(t, now)
        if fresh or room >= ARC2["MIN_SLICE_SECONDS"]:
            out.append((not fresh, -room, _arc2_order[t], t))
    return [c[-1] for c in sorted(out)]


def _arc2_take():
    """Next task for a free stream (see _arc2_level); it keeps the stream until it has used its equal share and
    another task waits for one (_arc2_yield)."""
    with _arc2_lock:
        now = _arc2_clock()
        if ARC2_DEADLINE - now < ARC2["MIN_START_SECONDS"] or ARC2_SERVER_DEAD[0]:
            return None, "end"
        level = _arc2_level(now)
        cands = _arc2_candidates(now, level)
        if not cands:
            return None, ("wait" if _arc2_running else "end")
        tid = cands[0]
        if ARC2_STATE[tid].stats["slices"]:
            _arc2_round[0] = max(_arc2_round[0], ARC2_STATE[tid].stats["slices"] + 1)
            print(f"ARC2 RESUME {tid} with its history: {level - _arc2_usage(tid, now):.0f} s of its equal share "
                  f"left ({_arc2_usage(tid, now):.0f} s used, share {level:.0f} s)", flush=True)
        _arc2_running[tid] = now
        return tid, None


def _arc2_yield(task_id, now):
    """True when a running task has used its equal share (after at least MIN_SLICE_SECONDS in this slice) and a
    waiting task could take its stream."""
    with _arc2_lock:
        start = _arc2_running.get(task_id)
        if (start is None or now - start < ARC2["MIN_SLICE_SECONDS"]
                or ARC2_DEADLINE - now < ARC2["MIN_START_SECONDS"]):
            return False
        level = _arc2_level(now)
        return _arc2_usage(task_id, now) >= level and bool(_arc2_candidates(now, level))


def _arc2_waiting():
    return sum(1 for t in ARC2_IDS if ARC2_STATE[t].stats["slices"] == 0)


def _arc2_worker(rank):
    while True:
        tid, deadline = _arc2_take()
        if tid is None:
            if deadline == "wait":
                _time.sleep(5)
                continue
            return
        st = ARC2_STATE[tid]
        t0 = _time.time()
        try:
            answer = st.run()
        except Exception as exc:
            st.stats["stop"] = f"crash: {type(exc).__name__}: {exc}"[:300]
            st.stats["traceback"] = _traceback.format_exc()[-2000:]
            answer = st.answer()
        with _arc2_lock:
            _start = _arc2_running.pop(tid, None)
            if _start is not None:
                _arc2_used[tid] += max(0.0, _arc2_clock() - _start)
            ARC2_SUBMISSION[tid] = answer
            _write_submission()
        if not TRUE_SUBMISSION:
            (ARC2_LOG_DIR / f"{tid}.json").write_text(_json.dumps(
                {"task": tid, "stats": st.stats, "memory": st.memory, "messages": st.messages,
                 "kept": st.kept, "imports": st.imports, "candidates": st.candidates, "answer": answer}, default=str))
        s = st.stats
        print(f"[arc2 {rank}] {tid}: {s['stop']} after {_time.time() - t0:.0f}s (slice {s['slices']}), "
              f"{s['requests']} requests, {s['tool_calls']} python calls, {s['drains']} drains, "
              f"best {st.memory['best']}/{len(st.task['train'])} | waiting {_arc2_waiting()}", flush=True)


ARC2_CPU_PHASE["phase"] = "solving"  # CPU telemetry
print(f"ARC2 START {len(ARC2_IDS)} tasks, {ARC2['WORKERS']} streams, "
      f"{(ARC2_DEADLINE - _arc2_clock()) / 60:.0f} min of solving", flush=True)
_arc2_threads = [_threading.Thread(target=_arc2_worker, args=(r,), daemon=True) for r in range(ARC2["WORKERS"])]
for _th in _arc2_threads:
    _th.start()
for _th in _arc2_threads:
    _th.join(None if ARC2["EMULATE_TOK_S"] else max(0.0, ARC2_DEADLINE - _time.time() + 600))
with _arc2_lock:
    # from every task's recorded answers, including a slice still running at the deadline
    for _tid in ARC2_IDS:
        ARC2_SUBMISSION[_tid] = ARC2_STATE[_tid].answer()
    _write_submission()
_solved = sum(1 for t in ARC2_IDS if ARC2_STATE[t].stats["stop"] == "solved")
print(f"ARC2 DONE: {_solved} of {len(ARC2_IDS)} tasks solved on their example pairs; submission at "
      f"{ARC2_SUBMISSION_PATH}", flush=True)
if ARC2_SERVER_DEAD[0]:
    print(f"ARC2 SERVER DIED: {ARC2_SERVER_DEAD[0]}; the run stopped taking tasks", flush=True)
ARC2_CPU_PHASE["phase"] = "done"  # CPU telemetry: the solving phase, summarized
try:
    print("ARC2 CPU", _json.dumps(arc2_cpu_summary("solving")), flush=True)
except Exception as _cpu_exc:
    print(f"ARC2 CPU summary failed: {type(_cpu_exc).__name__}: {_cpu_exc}"[:300], flush=True)

# ---- local score (Save & Run only: the evaluation solutions are in the competition data) ----
if not TRUE_SUBMISSION:
    _sol_path = _Path(ARC2_DIR) / "arc-agi_evaluation_solutions.json"
    if _sol_path.is_file():
        _sol = _json.loads(_sol_path.read_text())
        # official metric (arcprize.org): the average over all test outputs; the per-task average is kept as well
        _rows, _total, _outputs_ok, _outputs = [], 0.0, 0, 0
        for _tid in ARC2_IDS:
            _att = ARC2_SUBMISSION[_tid]
            _hits = [any(_att[i][k] == _want for k in ("attempt_1", "attempt_2")) for i, _want in enumerate(_sol[_tid])]
            _score = sum(_hits) / len(_hits)
            _total += _score
            _outputs_ok += sum(_hits)
            _outputs += len(_hits)
            _s = ARC2_STATE[_tid].stats
            _rows.append({"task": _tid, "score": _score, "stop": _s["stop"], "slices": _s["slices"],
                          "seconds": _s["seconds"], "requests": _s["requests"], "python_calls": _s["tool_calls"],
                          "prompt_tokens": _s["prompt_tokens"], "cached_tokens": _s["cached_tokens"],
                          "completion_tokens": _s["completion_tokens"], "drains": _s["drains"],
                          "best_pairs": ARC2_STATE[_tid].memory["best"], "pairs": len(ARC2_TASKS[_tid]["train"]),
                          "rule_kept": bool(ARC2_STATE[_tid].memory["rule_visible"] or
                                            ARC2_STATE[_tid].memory["rule_reasoning"])})
        _summary = {"tasks": len(ARC2_IDS), "outputs_correct": _outputs_ok, "outputs": _outputs,
                    "score_pct": round(100 * _outputs_ok / max(1, _outputs), 2), "score": round(_total, 3),
                    "task_average_pct": round(100 * _total / max(1, len(ARC2_IDS)), 2), "rounds": _arc2_round[0],
                    "rows": _rows}
        (_Path(WORKING_DIR) / "arc2_summary.json").write_text(_json.dumps(_summary, indent=1))
        for _row in _rows:
            print("ARC2", _json.dumps(_row), flush=True)
        print(f"ARC2 SCORE {_outputs_ok} / {_outputs} test outputs = {_summary['score_pct']}% (official metric); "
              f"per-task average {_summary['score']} / {len(ARC2_IDS)} = {_summary['task_average_pct']}%", flush=True)

TRUE_SUBMISSION=False
L4 ENV {
 "python": "3.12.13",
 "cpus": 48,
 "gpus": [
  {
   "index": "0",
   "name": "NVIDIA L4",
   "memory": "23034 MiB",
   "driver": "580.178.04",
   "cc": "8.9"
  },
  {
   "index": "1",
   "name": "NVIDIA L4",
   "memory": "23034 MiB",
   "driver": "580.178.04",
   "cc": "8.9"
  },
  {
   "index": "2",
   "name": "NVIDIA L4",
   "memory": "23034 MiB",
   "driver": "580.178.04",
   "cc": "8.9"
  },
  {
   "index": "3",
   "name": "NVIDIA L4",
   "memory": "23034 MiB",
   "driver": "580.178.04",
   "cc": "8.9"
  }
 ],
 "driver_cuda": "13.0",
 "ram_gib": {
  "MemTotal": 188.7,
  "MemAvailable": 185.2,
  "SwapTotal": 0.0
 },
 "disk": {
  "/": {
   "free_gib": 1048.8,
   "total_gib": 8156.9
  },
  "/kaggle/working": {
   "free_gib": 19.5,
   "total_gib": 19.5
  },
  "/tmp": {
   "free_gib": 1048.8,
   "total_gib": 8156.9
  },
  "/dev/shm": {
   "free_gib": 85.5,
   "total_gib": 85.5
  }
 },
 "gpu_links": [
  "\u001bGPU0\tGPU1\tGPU2\tGPU3\tCPU Affinity\tNUMA Aff

/tmp/ipykernel_65/556607086.py:610: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(PATCH_DIR)


  [main] 30s ...
Precaching: Started...
  [main] 60s ...
  [main] 90s ...
  [main] 120s ...
  [main] 150s ...
  [main] 180s ...
  [main] 210s ...
  [main] 240s ...
  [main] 270s ...
  [main] 300s ...
  [main] 330s ...
  [main] 360s ...
  [main] 390s ...
Precaching: loaded 62 files (185.40 GB) in 342.48 seconds.

  [main] 420s ...
[main] READY after 426s
SERVER REPORT {"log": ["(APIServer pid=464) INFO 11-01 02:00:30 [api_utils.py:286] non-default args: {'enable_auto_tool_choice': True, 'tool_call_parser': 'qwen3_xml', 'enable_prompt_tokens_details': True, 'host': '127.0.0.1', 'port': 8001, 'model': '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/tr", "(APIServer pid=464) INFO 11-01 02:00:48 [cache.py:348] Using fp8_e4m3 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor", "(APIServer pid=464) WARNING 11-01 02:00:58 [speculative.py:1323] Enabling num_sp